# CASMI 26: From Spectra to Structures

### From measured fragments to ranked molecular structures

The CASMI 2026 competition (Critical Assessment of Small Molecule Identification) asks for the structure of an unknown molecule from its tandem mass spectra (MS/MS). A molecule can yield several such spectra, each recording the fragment ions of one selected precursor ion, the ion isolated before fragmentation, and the answer is one ordered list of up to 25 structures per molecule. This notebook audits the competition data and runs a reproducible ranking method that connects the measured fragments to candidate structures. The worked example is caffeine: Figure 1 shows its structure, two of its measured spectra and the sizes of the largest reference libraries.

Sections 1–2 explain the task with the worked example; Sections 3–9 analyse the data; Sections 10–12 describe and run the model; Section 13 discusses the search, Section 14 lists the public scores of the main versions and Section 15 credits the sources. A [domain primer for non-chemists](https://www.kaggle.com/code/mohammadhmozafary/casmi-2026-domain-knowledge-for-non-chemists) provides background on mass spectrometry.


In [ ]:
import os, sys, subprocess, importlib.metadata, time
os.environ['PYTHONHASHSEED'] = '0'
NOTEBOOK_STARTED = time.monotonic()   # record total notebook time, including the retained analysis
from pathlib import Path
if not Path('/kaggle/input').exists() and Path('.runtime/rdkit2026033').exists():
    sys.path.insert(0, str(Path('.runtime/rdkit2026033').resolve()))
try:
    installed = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed = ''
if installed.replace('.03.', '.3.') != '2026.3.3':
    tag=f'cp{sys.version_info.major}{sys.version_info.minor}'
    wheels=sorted(Path('/kaggle/input').rglob(f'rdkit-2026.3.3-{tag}-*.whl'))
    assert wheels, f'Attach the RDKit 2026.3.3 offline wheel for {tag}.'
    subprocess.check_call([sys.executable,'-m','pip','install','--no-index','--no-deps','-q',str(wheels[0])], env={**os.environ, 'PYTHONHASHSEED': '0'})
from rdkit import rdBase
assert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion



In [ ]:
from pathlib import Path
import os
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

SEED = 20260915
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (11, 5.5), "axes.titleweight": "bold", "figure.dpi": 120})

# Sharp inline figures, with PNG retained for readers that do not support SVG.
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg', 'png')
plt.rcParams.update({
    'figure.dpi': 160, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'svg.fonttype': 'none', 'figure.facecolor': 'white',
})


In [ ]:
def locate_data_dir():
    candidates = [
        Path("/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-casmi26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"),
        Path("data/raw"),
        Path("../data/raw"),
    ]
    for path in candidates:
        if all((path / name).exists() for name in ("train.parquet", "test.parquet", "sample_submission.csv")):
            return path.resolve()
    raise FileNotFoundError("Could not find train.parquet, test.parquet, and sample_submission.csv")


DATA = locate_data_dir()
TRAIN_PATH = DATA / "train.parquet"
TEST_PATH = Path(os.environ.get("CASMI_TEST_PATH", DATA / "test.parquet"))
SAMPLE_PATH = Path(os.environ.get("CASMI_SAMPLE_PATH", DATA / "sample_submission.csv"))

In [ ]:
# The journal-style header is assembled from measured competition data: caffeine, the worked example of Sections 1-2.
CAFFEINE_KEY = "RYYVLZVUVIJVGH"
header_columns = ["normalized_smiles", "ms2_mzs", "ms2_normalized_intensities", "adduct", "precursor_mz",
                  "ingest_lib", "instrument_type", "collision_energy_ev"]
header_file = pq.ParquetFile(TRAIN_PATH)
caffeine_parts = []
for group in range(header_file.num_row_groups):
    keys = header_file.read_row_group(group, columns=["inchikey14", "adduct"]).to_pandas()
    hit = np.where((keys["inchikey14"] == CAFFEINE_KEY) & (keys["adduct"] == "[M+H]+"))[0]
    if len(hit):
        caffeine_parts.append(header_file.read_row_group(group, columns=header_columns).to_pandas().iloc[hit])
caffeine = pd.concat(caffeine_parts, ignore_index=True) if caffeine_parts else pd.DataFrame(columns=header_columns)


def single_energy(value):
    try:
        values = [float(v) for v in value]
    except TypeError:
        return np.nan
    return values[0] if len(values) == 1 else np.nan


caffeine["energy"] = caffeine["collision_energy_ev"].map(single_energy)
caffeine["peaks"] = caffeine["ms2_mzs"].map(len)
timstof = caffeine[(caffeine["ingest_lib"] == "enveda-np-examples") & caffeine["energy"].notna()].sort_values("energy")
if len(timstof) >= 2:
    views = [timstof.iloc[0], timstof.iloc[-1]]
else:
    views = [row for _, row in caffeine.sort_values("peaks", ascending=False).head(2).iterrows()]
is_caffeine = len(views) == 2 and len(timstof) >= 2
if not views:
    preview_pool = header_file.read_row_group(0, columns=header_columns).slice(0, 700).to_pandas()
    preview_pool["peaks"] = preview_pool["ms2_mzs"].map(len)
    views = [preview_pool.sort_values("peaks", ascending=False).iloc[0]]
mol = Chem.MolFromSmiles(views[0]["normalized_smiles"])
mol_image = Draw.MolToImage(mol, size=(900, 650), fitImage=True)
header_meta = pd.read_parquet(TRAIN_PATH, columns=["ingest_lib", "inchikey14"])
header_connectivities = header_meta["inchikey14"].nunique()
source_counts = header_meta["ingest_lib"].value_counts().head(6).sort_values()
del header_meta


def draw_view(ax, row, title, labels, step):
    """One measured spectrum; labels the given fragment m/z values and draws the mass step between two of them."""
    mz = np.asarray(row["ms2_mzs"], dtype=float)
    intensity = np.asarray(row["ms2_normalized_intensities"], dtype=float)
    precursor = float(row["precursor_mz"])
    ax.vlines(mz, 0, intensity, color="#b74759", linewidth=1.3)
    ax.axvline(precursor, color="#46566b", linestyle=":", linewidth=1)
    ax.text(precursor, 1.37, f"precursor\n{precursor:.3f}", ha="center", va="bottom", fontsize=9, color="#46566b")
    ax.set(xlabel="Fragment m/z", ylabel="Relative intensity", title=title, ylim=(0, 1.55), xlim=(30, precursor + 18))
    ax.set_yticks([0, 0.5, 1.0])
    for target in labels:
        j = int(np.argmin(np.abs(mz - target)))
        if abs(mz[j] - target) < 0.01:
            ax.text(mz[j], intensity[j] + 0.03, f"{mz[j]:.3f}", ha="center", va="bottom", fontsize=9)
    if step:
        start, end, text = step
        if not np.any(np.abs(mz - start) < 0.01):          # the arrow starts at a fragment that this spectrum no longer shows
            ax.axvline(start, color="#46566b", linestyle=":", linewidth=0.8)
            ax.text(start, 1.37, f"{start:.3f}\nnot seen", ha="center", va="bottom", fontsize=9, color="#46566b")
        ax.annotate("", xy=(end, 1.16), xytext=(start, 1.16), arrowprops=dict(arrowstyle="->", color="#16324f", lw=1.4))
        ax.text((start + end) / 2, 1.19, text, ha="center", va="bottom", fontsize=10, color="#16324f")


fig = plt.figure(figsize=(11.0, 10.5), facecolor="white")
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=.5, wspace=.3, left=.1, right=.96, bottom=.1, top=.86)
ax0 = fig.add_subplot(grid[0, 0])
ax0.imshow(mol_image); ax0.axis("off")
if is_caffeine:
    ax0.set_title("A. Caffeine, C$_8$H$_{10}$N$_4$O$_2$", loc="left")
    precursor = float(views[0]["precursor_mz"])
    draw_view(fig.add_subplot(grid[0, 1]), views[0], f"B. {views[0]['energy']:.0f} eV: one piece breaks off",
              [138.0666], (precursor, 138.0666, "−57.021 Da (C$_2$H$_3$NO)"))
    draw_view(fig.add_subplot(grid[1, 0]), views[1], f"C. {views[1]['energy']:.0f} eV: the ion breaks further",
              [123.0427], (138.0666, 123.0427, "−15.023 Da (CH$_3$)"))
    ax2 = fig.add_subplot(grid[1, 1])
    subtitle = f"Caffeine · [M+H]+ at m/z {precursor:.3f} · measured on timsTOF (enveda-np-examples)"
else:
    ax0.set_title("A. Known molecular graph", loc="left")
    for k, row in enumerate(views[:2]):
        draw_view(fig.add_subplot(grid[0, 1] if k == 0 else grid[1, 0]), row, "BC"[k] + ". Measured fragments", [], None)
    ax2 = fig.add_subplot(grid[1, 1] if len(views) > 1 else grid[1, :])
    subtitle = f"One measured example · precursor {float(views[0]['precursor_mz']):.4f} m/z · {views[0]['adduct']}"
ax2.barh(source_counts.index, source_counts.values / 1e3, color="#287a8c")
ax2.set(xlabel="Measured spectra (thousands)", title="D. Largest reference libraries")
ax2.set_xlim(0, source_counts.max() / 1e3 * 1.3)
ax2.grid(axis="y", visible=False)
for y, value in enumerate(source_counts.values):
    ax2.text(value / 1e3 + source_counts.max() / 1e3 * .014, y, f"{value / 1e3:,.0f}k", va="center", fontsize=10)
fig.suptitle("FROM SPECTRA TO STRUCTURES", fontsize=21, weight="bold", color="#16324f", y=.965)
fig.text(.5, .912, subtitle, ha="center", fontsize=12)
fig.text(.5, .025, f"{header_file.metadata.num_rows:,} spectra · {header_connectivities:,} stored connectivity groups",
         ha="center", fontsize=12, color="#46566b")
plt.show()

*Figure 1. Graphical abstract built from the competition corpus. (A) Caffeine, the worked example of Sections 1–2. (B, C) Two spectra of protonated caffeine (`[M+H]+`) from the `enveda-np-examples` training library, measured on a timsTOF instrument, the instrument family of the hidden test. At 35 eV the protonated ion (m/z 195.088) mostly loses 57.021 Da, a C₂H₃NO fragment, and m/z 138.067 is the strongest peak. At 60 eV both of these ions are gone and m/z 123.043 dominates; it lies 15.023 Da, the mass of a methyl group, below 138.067. (D) The six source libraries with the most spectra. Inference runs in the opposite direction, from spectra such as B and C to the structure in A.*


## Abstract

An accurately measured precursor mass restricts the candidates to a narrow mass window, but structures that share the mass, and often the formula, remain; the fragment pattern has to separate them. The notebook audits the competition corpus (Sections 3–9) and runs an inference pipeline based on the public E006 of [canqiang](https://www.kaggle.com/code/canqiang/casmi26-e006-window-40-ice-fallback-true) (Sections 10–12), which connects measured fragment patterns with candidate structures. The main engine ranks candidates from a pool of 710,701 training and COCONUT structures (COCONUT is an open natural-product collection) and from a generator that modifies spectral analogues; it uses the full1 fingerprint model, which predicts substructures from the spectra, and a ranker of four LightGBM boosters. A separate PubChem channel, built on FPNet A+B and a fold-safe fingerprint checkpoint, proposes well-documented structures from outside the pool. A second engine, with two learned rankers over the training, COCONUT, ChEBI/LIPID MAPS and PubChemLite catalogues, returns an independent list. After structures with the same scorer identity key have been collapsed, the main and second-engine lists are fused by reciprocal rank (metric-identity fusion). Within each molecular formula, the fused candidates are then reordered by spectral evidence: predicted-spectrum similarity from GLACIER where it has a score or from ICEBERG where GLACIER has none, and otherwise the share of measured intensity that the candidate's fragments explain; the remaining places are filled with distinct identities, and the leading structure of the second engine is finally placed at rank 2 unless it already holds rank 1.

The pipeline is built on [Obstacle V1.1](https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail) and [Bob V18](https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith) and uses the public engines and datasets credited in Section 15. Catalogued natural products carry PubMed documentation more often than a random PubChem sample (Figure 18), which motivates the documentation prior in the ranking. Section 11 retains the 250-natural-product development benchmark of earlier versions; its configuration differs from the present pipeline, and most pretrained components have no complete published training roster, so it describes development behaviour and does not validate the present pipeline. This pipeline scored 0.455 public MRR@25 (version 55); Section 14 records the public scores of the main versions.


## 1. From a molecule to a spectrum — and back to SMILES

A mass spectrometer measures the mass of a charged molecule, breaks it into fragments and measures the masses of the fragments. The spectrum records only these masses and the detected abundance of each fragment, never the molecular graph itself. The task is to propose up to 25 candidate structures as **SMILES** strings ([Simplified Molecular Input Line Entry System](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System), a one-line text notation for a molecular graph), most likely first.

### How a mass spectrometer weighs a molecule

A mass spectrometer gives the molecule an electric charge and records how the charged particle, an **ion**, moves in an electric field. The hidden test was measured on timsTOF instruments, in which a trapped ion-mobility stage (tims) sorts the ions by shape before a **time-of-flight (TOF)** tube measures their mass; the steps are those of a TOF instrument.

1. **Charging the molecule.** In electrospray ionisation (**ESI**), the usual source for liquid samples such as plant extracts, the dissolved molecule picks up a proton and becomes a positive ion (`[M+H]+`), or loses one and becomes a negative ion (`[M-H]-`, **negative mode**). The molecule stays intact until step 4.
2. **Accelerating the ions.** A voltage $V$ gives every ion with $z$ elementary charges $e$ the same energy, $zeV$, which turns into motion:
$$
zeV=\frac{1}{2}mv^2 \quad\Rightarrow\quad v=\sqrt{\frac{2zeV}{m}}
$$
3. **Timing the flight.** Over a flight tube of length $L$ an ion needs $t = L/v = L\sqrt{m/(2zeV)}$: heavier ions arrive later, and the arrival time depends only on the ratio $m/z$. The instrument converts every arrival time into a mass-to-charge ratio, **m/z**. Most ions here carry a single charge ($z = 1$), so the numerical m/z value equals the ion mass in daltons.
4. **Breaking and weighing again (MS/MS).** The instrument keeps only ions of one m/z, the **precursor**, lets them collide with gas molecules so that they break, and times the pieces in the same way. Only charged pieces reach the detector; a neutral piece that falls off is invisible and appears as a mass difference, a **neutral loss** (the arrows in Figure 1).

**From mass to formula.** Masses are given in daltons (Da). One dalton is one twelfth of the mass of a carbon-12 atom, close to the mass of a hydrogen atom. Atomic masses are not whole numbers (H 1.00783, C 12 exactly, N 14.00307, O 15.99491), so an accurately measured mass allows only a few elemental formulas. Caffeine's neutral mass, 194.0804 Da, fits C₈H₁₀N₄O₂; C₇H₁₄O₆ has the same whole-number mass, 194, but weighs 194.0790 Da. The difference, 7 parts per million, is within reach of an accurate instrument, although the 10-ppm search window used below still admits both. One formula still admits many connectivities, and the fragments are the evidence that separates them.

### One molecule from start to finish: caffeine

Caffeine (C₈H₁₀N₄O₂, Figure 1A) has 645 spectra in the training file, from nine source libraries; four of these spectra, all in `enveda-np-examples`, were measured on timsTOF, the instrument family of the hidden test. Its path through the experiment shows each step that the model has to invert; Figure 2 draws the same steps as a diagram.

| Step | What happens | Caffeine |
|---|---|---|
| Separation (**LC**, liquid chromatography) | a mixture is spread out in time, so its molecules reach the instrument one after another | the files contain no retention times, so this step leaves no trace in the data |
| Ionisation (**ESI**, electrospray) | the molecule picks up a charge (step 1 above); the **adduct** names the ion form | `[M+H]+`, one added proton, in 474 spectra; `[M+Na]+`, an added sodium ion, in 149; `[M-H]-` in 20; two other ion forms |
| Weighing the ion (**MS1**) | the instrument measures the mass-to-charge ratio, **m/z** | m/z 195.0877; subtracting the proton mass (1.0073, a hydrogen atom without its electron) gives the neutral **monoisotopic** mass (computed with each element's most abundant isotope), 194.0804 Da |
| Breaking the ion (**CID**, collision-induced dissociation) | the selected ion is accelerated into gas molecules and breaks apart | at a collision energy of 35 eV (electron volts) the ion mostly loses 57.021 Da, a C₂H₃NO piece (methyl isocyanate), leaving m/z 138.067 (Figure 1B); this fragment is the strongest one in about 60% of caffeine's `[M+H]+` spectra |
| Weighing the pieces (**MS2**, or MS/MS) | a spectrum: each peak is one fragment m/z with its intensity, scaled so that the strongest peak (the **base peak**) is 1 | at 60 eV the ion breaks further and m/z 123.043 dominates (Figure 1C): the collision energy changes the pattern (Figure 12 shows the same effect for a visible-test molecule) |

### Running the experiment backwards

Within 10 parts per million (ppm) of 194.0804 Da, about ±0.002 Da, the 710,701-structure candidate pool of this notebook holds 48 structures. Thirty-one are sugar-like compounds (C₇H₁₄O₆, for example methyl glycosides) whose mass differs from caffeine's by only 7 ppm, and two have a third formula. Fifteen share caffeine's formula exactly, among them isocaffeine, which differs only in the position of one methyl group. A loss of 57.021 Da fits caffeine, but one loss rarely settles the question: isomers can lose the same piece, and rearrangements make a bond-by-bond reading unreliable. The useful evidence is the whole pattern of fragments across all spectra of a molecule.

Caffeine has reference spectra in the library, and a direct spectral search finds it (*A Complete Statistical Tour*, in [its own section 14](https://www.kaggle.com/code/josefreitasalvesneto/casmi-2026-a-complete-statistical-tour), measures how far direct search alone goes; [a public retrieval notebook](https://www.kaggle.com/code/haideptry/enveda-casmi-2026-fast-spectral-cosine-baseline) records the same limit in its version history before adding analogue, fragmentation and fingerprint channels). A molecule without a reference spectrum needs indirect evidence: a predicted fingerprint (a vector of yes/no answers about substructures), a predicted spectrum or measured relatives. A structure absent from the searched databases needs a generator to enter the candidate list. Section 2 tabulates these three situations.

[CSI:FingerID](https://www.pnas.org/doi/10.1073/pnas.1509788112) and [MIST](https://www.nature.com/articles/s42256-023-00708-3) predict fingerprints that help rank database structures; [MSNovelist](https://www.nature.com/articles/s41592-022-01486-3) generates structures de novo, as does the [de novo tutorial notebook](https://www.kaggle.com/code/inversion/casmi-denovo-tutorial-notebook) published for this competition; [MassSpecGym](https://github.com/pluskal-lab/MassSpecGym) provides evaluation tasks for both settings. Here, database retrieval is supplemented by a constrained generator that modifies spectral analogues, as described in Section 10.

**Related public material.** The Wikipedia articles on [time-of-flight mass spectrometry](https://en.wikipedia.org/wiki/Time-of-flight_mass_spectrometry), [tandem mass spectrometry](https://en.wikipedia.org/wiki/Tandem_mass_spectrometry) and [electrospray ionisation](https://en.wikipedia.org/wiki/Electrospray_ionization) have instrument diagrams, and the article on [the dalton](https://en.wikipedia.org/wiki/Dalton_%28unit%29) defines the mass unit. [Domain Knowledge for Non-Chemists](https://www.kaggle.com/code/mohammadhmozafary/casmi-2026-domain-knowledge-for-non-chemists) follows caffeine through the measurement columns of the training file and defines SMILES, stereochemistry and InChIKeys. [A Complete Statistical Tour](https://www.kaggle.com/code/josefreitasalvesneto/casmi-2026-a-complete-statistical-tour) examines real spectra, adducts, collision energies and neutral losses.


In [ ]:

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.set(xlim=(0, 1), ylim=(0, 1)); ax.axis("off")
steps = [(0.07,"LC\nseparation"),(0.23,"ESI\nionisation"),(0.39,"MS1\nprecursor"),
         (0.55,"CID\nfragmentation"),(0.71,"MS2\nfragments"),(0.92,"ranked\nSMILES")]
colors = sns.color_palette("crest", len(steps))
for i_step, ((x, label), color) in enumerate(zip(steps, colors)):
    ax.text(x, .57, label, ha="center", va="center", fontsize=12, color="white" if i_step >= 2 else "#16324f", zorder=3,
            bbox=dict(boxstyle="round,pad=.65", fc=color, ec="white", lw=1.5))
    if i_step < len(steps)-1:
        next_x = steps[i_step+1][0]
        ax.annotate("", xy=(next_x-.065,.57), xytext=(x+.065,.57), zorder=2,
                    arrowprops=dict(arrowstyle="-|>", mutation_scale=16, lw=1.8, color="#46566b"))
ax.axvline(.815, ymin=.20, ymax=.83, color="#b04a5a", ls="--", lw=1.5)
ax.text(.41,.13,"physical acquisition",ha="center",fontsize=12,color="#284b63",weight="bold")
ax.text(.91,.13,"inverse problem",ha="center",fontsize=12,color="#9b2c3b",weight="bold")
ax.set_title("Figure 2. From experimental acquisition to structure inference", pad=12, fontsize=16)
plt.show()


*Figure 2. From experimental acquisition to structure inference: chromatographic separation, electrospray ionisation, MS1 measurement of the precursor ion, collision-induced fragmentation and the MS2 spectrum of the fragments. The computational task begins with the MS1 and MS2 measurements and ends with ranked molecular structures.*


## 2. What counts as a correct prediction

The submission contains one ranked list per **`molecule_id`**, built from all spectra of that molecule. The [official evaluation](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/overview) uses **mean reciprocal rank among the first 25 guesses (MRR@25)**:

$$
\operatorname{MRR@25}=\frac{1}{U}\sum_{u=1}^{U}
\begin{cases}
1/r_u, & \text{if the first correct candidate has rank }r_u\leq25,\\
0, & \text{if it is absent from the first 25.}
\end{cases}
$$

Each of the $U$ molecules contributes the reciprocal of the rank of its first correct guess, or zero if the answer is not among the first 25, and the score is the average ([mean reciprocal rank](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)). Every molecule has equal weight, however many spectra it has. An example with three molecules:

| Molecule | Rank of the correct structure | Contribution |
|---|---:|---:|
| A | 1 | 1 |
| B | 4 | 0.25 |
| C | not among the 25 guesses | 0 |
| **MRR@25** | | **(1 + 0.25 + 0) / 3 = 0.42** |

The first ranks dominate: rank 1 is worth 1, rank 2 is worth 0.5 and rank 25 only 0.04, so moving a correct structure from rank 2 to rank 1 adds 0.5, the same gain as placing a previously absent answer at rank 2.

**What counts as the same answer.** The scorer compares *connectivity*, which atoms are bonded to which. It uses **RDKit 2026.03.3**, a chemistry toolkit, to pick one standard form among **tautomers** (structures that differ mainly in where a hydrogen and a double bond sit), then compares the first 14 characters of the [InChIKey](https://en.wikipedia.org/wiki/International_Chemical_Identifier), a hashed structure identifier whose first block encodes connectivity. Keys computed with the scorer's RDKit version:

| Pair | Same answer? | Why |
|---|---|---|
| L-alanine and D-alanine | yes (`QNAYBMKLOCPYGJ`) | mirror images: stereochemistry is ignored |
| the two carvones, one smelling of spearmint and one of caraway | yes (`YAHNGTQFNICUIJ`) | mirror images again |
| 2-pyridone and 2-hydroxypyridine | yes (`UBQKCCHYAOITMY`) | tautomers collapse to one form |
| caffeine and isocaffeine | no (`RYYVLZVUVIJVGH` vs `LPHGQDQBBGAPDZ`) | same formula, one methyl group elsewhere |

For modelling, the correct connectivity has to enter the candidate list, and its rank then sets the molecule's contribution. Stereoisomers and tautomers need no separate guesses, and repeating an equivalent structure wastes a rank. A close analogue earns nothing: a list that offers isocaffeine but never caffeine scores zero for that molecule. The submission file has `molecule_id,smiles` columns with up to 25 SMILES per molecule, best first and separated by semicolons. The [primer for non-chemists](https://www.kaggle.com/code/mohammadhmozafary/casmi-2026-domain-knowledge-for-non-chemists) gives InChIKey examples, and [a public notebook on key changes](https://www.kaggle.com/code/dariushafshar/casmi26-10-4-of-keys-change-0-48-cost-you) measures how often the tautomer step changes the key that a structure library ships (10.4% of COCONUT) and how often that change moves a score.


### The visible sample and the hidden task

The [official data description](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data) distinguishes three novelty classes, according to the evidence available for the answer:

| Available evidence for the answer | Example | Main route to a candidate | Main failure mode |
|---|---|---|---|
| Public reference spectrum | caffeine, with 645 training spectra | direct library search | acquisition differences or incorrect ranking |
| PubChem/COCONUT structure, no public spectrum | most catalogued natural products: only 8.0% of the entries of the LOTUS and NPAtlas natural-product catalogues have a training spectrum (Figure 16) | structure database plus indirect spectral evidence | missing candidate or weak discrimination among isomers |
| Novel structure, absent from PubChem | a newly discovered metabolite | de novo structure proposal | the correct graph is never generated |

Class membership and proportions are hidden; [one public starter notebook](https://www.kaggle.com/code/denpugovkin/three-hidden-contests-one-ranked-list) treats the classes as three separate contests inside one ranked list. The released `test.parquet` is only a **format example**: its spectra come from the training file, so it verifies that a notebook runs, not how well it identifies new molecules. Kaggle reruns the notebook against a hidden test file for the leaderboard score.

An offline estimate requires that every spectrum of a validation molecule be removed from the reference library and from the training features before that molecule is ranked. Section 11 states this requirement, measures the overlap that ordinary splits leave (Figure 19) and reports an earlier development benchmark that does not meet it; Section 12 checks execution. [Data Pitfalls That Break Your CV](https://www.kaggle.com/code/kitopl/casmi-2026-data-pitfalls-that-break-your-cv) shows how a split that leaves a validation molecule's spectra in the library inflates the score.


### Files used in this notebook

The [competition data](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data) provide labelled measured spectra in `train.parquet`, anonymous molecule groups in `test.parquet` and the row order of `sample_submission.csv`. Kaggle replaces the visible format example with hidden spectra when scoring. The inference inputs are public models, candidate catalogues and documentation tables; the panel benchmark and its copy of the identity-disjoint folds also support the retained analysis figures.

| Public datasets | Role |
|---|---|
| [v2 pool](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v2-pool), [v3 models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v3-models), [v4b models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v4b-models), [full1](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-fpnet-full1) | Main candidate arrays, feature modules, ranker and fingerprint checkpoints; v3 supplies the isolated PubChem A+B channel. |
| [PubChem tier](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-pubchem-tier), [popularity prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) | Structure-only mass windows, PubChem documentation, depositor flags, natural-product membership and aligned LOTUS reference counts. |
| [ICEBERG](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-iceberg), [GLACIER](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier) | Forward-spectrum checkpoints, inference code and isolated dependencies. |
| [Ranker features](https://www.kaggle.com/datasets/prvsiyan/casmi26-ranker-features), [fingerprint models](https://www.kaggle.com/datasets/prvsiyan/casmi26-fp-models-v2), [simulated ranker rows](https://www.kaggle.com/datasets/megayak/casmi26-simulated-ranker-rows) | Encoders and fitting rows for the second engine. |
| [Standardised ChEBI/LIPID MAPS](https://www.kaggle.com/datasets/sergiomalv/casmi-chebi-lmsd-structures), [COCONUT](https://www.kaggle.com/datasets/prvsiyan/coconut-casmi26-candidates), [PubChemLite fingerprints](https://www.kaggle.com/datasets/franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints), [hengck23 DreaMS resource](https://www.kaggle.com/datasets/hengck23/hengck23-dreams-enveda-casmi26) | Candidate catalogues of the second engine (ChEBI/LIPID MAPS, COCONUT and PubChemLite) and, from the DreaMS resource, mass-matched structures that fill vacant places at the end of a list. |
| [CASMI26 Fold-Safe FPNet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) | A fingerprint checkpoint trained without identity fold 0 and the 250 panel molecules, added to FPNet A+B in the PubChem channel. |
| [Official RDKit wheel](https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel), [simulator-compatible wheels](https://www.kaggle.com/datasets/eveliaveldrine/rdkit-wheels) | RDKit 2026.03.3 for the notebook/metric and 2025.03.6 for the isolated simulator processes. |
| [Natural-product knowledge table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table), [identity-disjoint validation folds](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-identity-disjoint-validation-folds), [natural-product panel benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | Natural-product classes and literature counts (read by inference through the aligned popularity arrays), validation folds and the development tables of Section 11. |

**Mounted resource packages.** The notebook mounts the competition data and five datasets of its author: [CASMI26 Public Models and Candidate Sets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets), [PubChem Popularity Prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior), [Natural-Product Panel Benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark), [Natural Product Knowledge Table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) and [CASMI26 Fold-Safe FPNet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet). The models package holds attributed copies of the public models and candidate sets in folders named after their source datasets, with the original authors and licences in `ATTRIBUTION.md`; the panel package also holds an unchanged copy of the identity-disjoint folds. The other packages supply documentation, natural-product annotations and the fold-safe checkpoint. Section 15 links the original sources.

| Package | What it supplies | Interpretation |
|---|---|---|
| `casmi26-v2-pool` | 710,701 structure rows, fragments and fingerprints | Train-derived and COCONUT proposals; measured reference spectra come from the current competition training file. |
| `casmi26-v3-models` | Earlier code and FPNet A+B weights | An isolated PubChem proposal process. |
| `casmi26-v4b-models` + `casmi26-fpnet-full1` | Main-engine code, its feature groups, the ranker of four LightGBM boosters and the full1 fingerprint encoder | Pretrained weights of the main channel. |
| `casmi26-pubchem-tier` | Mass-sorted structures and screening fingerprints | A structure-only candidate universe, not measured reference spectra. |
| `casmi26-iceberg` + `casmi26-glacier` | Forward models, source and offline wheels | Same-formula ordering for supported acquisitions. |

The upstream datasets retain their own terms and source notices: the notebook's Apache-2.0 declaration does not relicense model weights, competition-derived data or the candidate catalogues, and each dataset card and bundled licence notice governs reuse. The dataset card of the popularity prior documents its PubChem download release and matching rules. Section 15 credits these inputs and lists further reading.


## 3. File-level audit and schema

In both files a row is a **spectrum**: one fragmentation measurement of one precursor ion. The prediction target is a **molecule**, and one molecule usually has several rows. The attached training file holds about 2.54 million spectra of 275,810 stored connectivity groups, the `inchikey14` values (Table 1 gives the exact counts of the mounted file): a median of 6 spectra per structure, 14 or fewer spectra for nine structures in ten, and up to 1,468 for the most measured ones (Table 7 and Figure 7 in Section 5).

Three fields have distinct roles: `ingest_lib` identifies the source collection, `adduct` describes the measured ion, and `inchikey14` supplies a practical training-connectivity group. The prediction unit is therefore the molecule, and its repeated spectra are not independent molecules; `ingest_lib` records the origin of a spectrum, so a GNPS tag (Global Natural Products Social Molecular Networking) says which collection contributed the record and nothing about the chemical class of the molecule.

Tables 1 and 2 give the file sizes and explain each stored field. The example column of Table 2 follows one real record, the caffeine spectrum at 35 eV from Figure 1B, and the table below it lists that record's eight strongest peaks with the mass each fragment has lost.


In [ ]:
train_pf = pq.ParquetFile(TRAIN_PATH)
test_pf = pq.ParquetFile(TEST_PATH)

FIELD_MEANING = {
    "ingest_lib": "source spectral library", "normalized_smiles": "standardised molecular graph",
    "inchikey": "full hashed structure identifier", "inchikey14": "stored connectivity key for training grouping; scoring recomputes after tautomer canonicalisation",
    "molecular_formula": "elemental composition", "ionization_mode": "positive or negative ion polarity",
    "instrument_type": "mass-spectrometer family", "adduct": "normalised measured ion form",
    "adduct_orig": "ion form as supplied by the source", "precursor_mz": "selected precursor mass-to-charge ratio",
    "precursor_error_ppm": "precursor mass error in parts per million", "ms2_mzs": "fragment m/z array",
    "ms2_normalized_intensities": "base-peak-normalised fragment intensities", "num_peaks": "number of fragment peaks",
    "base_peak_intensity": "absolute intensity of the strongest peak", "collision_energy_ev": "collision energies mapped to eV",
    "collision_energy_orig": "source collision-energy text", "collision_energy_orig_units": "source energy unit",
    "molecule_id": "anonymous test-molecule group", "spectrum_id": "anonymous acquisition identifier",
}
# Table 2's example column follows one real record: the caffeine [M+H]+ spectrum at 35 eV of Figure 1B (Section 1).
_ex = views[0] if (globals().get("is_caffeine") and len(views)) else None
if _ex is not None:
    _mz = np.asarray(_ex["ms2_mzs"], dtype=float); _it = np.asarray(_ex["ms2_normalized_intensities"], dtype=float)
    _ce = np.atleast_1d(np.asarray(_ex["collision_energy_ev"], dtype=float))
    FIELD_EXAMPLE = {"ingest_lib": str(_ex["ingest_lib"]), "normalized_smiles": str(_ex["normalized_smiles"]), "inchikey14": CAFFEINE_KEY,
        "molecular_formula": "C8H10N4O2", "ionization_mode": "positive", "instrument_type": str(_ex["instrument_type"]),
        "adduct": str(_ex["adduct"]), "precursor_mz": f"{float(_ex['precursor_mz']):.4f}",
        "ms2_mzs": "[" + ", ".join(f"{x:.3f}" for x in _mz[:4]) + f", ...]  ({len(_mz)} values, increasing)",
        "ms2_normalized_intensities": "[" + ", ".join(f"{x:.3f}" for x in _it[:4]) + ", ...]  (strongest peak = 1)",
        "num_peaks": str(len(_mz)), "collision_energy_ev": "[" + ", ".join(f"{x:g}" for x in _ce) + "]",
        "molecule_id": "anonymous group id (test only)", "spectrum_id": "anonymous spectrum id (test only)"}
else:
    FIELD_EXAMPLE = {"ingest_lib":"gnps", "normalized_smiles":"CC(=O)O", "inchikey14":"QTBSBXVTEAMEQO",
        "molecular_formula":"C2H4O2", "ionization_mode":"positive", "instrument_type":"timsTOF",
        "adduct":"[M+H]+", "precursor_mz":"375.0709", "precursor_error_ppm":"1.5",
        "num_peaks":"230", "collision_energy_ev":"[20, 40, 60]", "molecule_id":"m_005e53",
        "spectrum_id":"s_4d19d522"}

def schema_table(pf):
    schema = pf.schema_arrow
    return pd.DataFrame({"field": schema.names, "storage type": [str(x.type) for x in schema],
                         "experimental meaning": [FIELD_MEANING.get(x, "source metadata") for x in schema.names],
                         "example": [FIELD_EXAMPLE.get(x, "—") for x in schema.names]})

summary = pd.DataFrame([
    {"file": "train.parquet", "rows": train_pf.metadata.num_rows,
     "columns": train_pf.metadata.num_columns, "row_groups": train_pf.metadata.num_row_groups},
    {"file": "test.parquet", "rows": test_pf.metadata.num_rows,
     "columns": test_pf.metadata.num_columns, "row_groups": test_pf.metadata.num_row_groups},
])
display(Markdown("**Table 1. Physical layout of the competition files.**"))
display(summary.style.format({"rows": "{:,}"}))

display(Markdown("**Table 2. Field dictionary: storage, scientific meaning, and representative values.**"))
display(pd.concat({"train": schema_table(train_pf), "test": schema_table(test_pf)}, names=["file"]))
if _ex is not None:                       # the eight strongest peaks of the example record and the mass each fragment has lost
    _prec = float(_ex["precursor_mz"]); _top = np.argsort(-_it)[:8]
    _loss = _prec - _mz[_top]
    _reading = {57.0215: "C2H3NO (methyl isocyanate), Figure 1B", 85.0164: "C2H3NO + CO", 72.0450: "C2H3NO + CH3"}
    peaks = pd.DataFrame({"fragment m/z": _mz[_top], "relative intensity": _it[_top], "loss from the precursor (Da)": _loss,
                          "reading": [next((v for k, v in _reading.items() if abs(l - k) < 0.005), "precursor itself" if abs(l) < 0.01 else "")
                                      for l in _loss]})
    display(Markdown(f"**Table 2 (continued). The eight strongest peaks of the example record** "
                     f"(caffeine `[M+H]+`, precursor m/z {_prec:.4f}, {len(_mz)} peaks in total)."))
    display(peaks.style.format({"fragment m/z": "{:.4f}", "relative intensity": "{:.3f}", "loss from the precursor (Da)": "{:.4f}"}).hide(axis="index"))


*Tables 1–2. File layout and field dictionary. Table 1 gives the dataset scale. Table 2 translates storage fields into experimental meaning; its example column is one consistent record (caffeine, `[M+H]+`, timsTOF, 35 eV). The peak arrays `ms2_mzs` and `ms2_normalized_intensities` are two aligned lists: the i-th intensity belongs to the i-th m/z. In the continuation, the neutral loss is the precursor m/z minus the fragment m/z: a loss of 57.021 Da is the C₂H₃NO fragment of Section 1, a loss of 85.016 Da is that fragment plus CO, and a loss of 72.045 Da is that fragment plus a methyl group.*


The two files differ in schema. Training carries molecular labels and source metadata; test carries anonymous `molecule_id` and `spectrum_id` values. Shared measurement fields use the same names, but the complete column sets differ; in particular, the training file has no test-style molecule ID, so its connectivity key is used for grouping.

The paired arrays `ms2_mzs` and `ms2_normalized_intensities` hold each spectrum; they have equal length, and filtering or sorting keeps each m/z with its intensity. Only measurement fields are available as inputs at test time: the peak arrays, `precursor_mz`, `adduct`, `ionization_mode`, `instrument_type` and the collision-energy fields. Structure labels (`normalized_smiles`, `inchikey`, `molecular_formula`), the source library and `precursor_error_ppm`, which is computed from the labelled structure, exist in the training file only. Table 3 checks the label counts and the integrity of the test peak arrays.

**MS2 only.** Every row is an MS2 spectrum: one round of fragmentation, from a selected precursor ion to its fragments. The files hold no MS3 or deeper (MSⁿ) spectra. The many spectra of one molecule therefore come from measuring it in different ways: other ion forms, the other polarity, other collision energies, other instruments and other libraries (Section 5 counts each). About a fifth of the training rows merge several collision energies into one spectrum; their `collision_energy_ev` holds a list.


In [ ]:
TRAIN_SCALAR_COLS = [
    "ingest_lib", "normalized_smiles", "inchikey", "inchikey14", "molecular_formula",
    "ionization_mode", "instrument_type", "adduct", "adduct_orig", "precursor_mz",
    "precursor_error_ppm", "num_peaks", "base_peak_intensity",
    "collision_energy_orig", "collision_energy_orig_units",
]

# The two peak arrays dominate disk volume. Scalar metadata is enough for most EDA and keeps memory bounded.
train = pd.read_parquet(TRAIN_PATH, columns=TRAIN_SCALAR_COLS)
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

assert test["spectrum_id"].is_unique
assert sample["molecule_id"].is_unique
assert set(test["molecule_id"]) == set(sample["molecule_id"])

test_peak_counts = test["ms2_mzs"].str.len()
alignment_ok = test_peak_counts.eq(test["ms2_normalized_intensities"].str.len())
normalisation_ok = test["ms2_normalized_intensities"].map(lambda x: len(x) > 0 and np.isclose(np.max(x), 1.0))
sorted_ok = test["ms2_mzs"].map(lambda x: np.all(np.diff(x) >= 0))

audit = pd.Series({
    "train spectra (rows)": len(train),
    "train unique SMILES labels": train["normalized_smiles"].nunique(),
    "train unique stored InChIKey14 groups": train["inchikey14"].nunique(),
    "train unique molecular formulae": train["molecular_formula"].nunique(),
    "visible-test spectra": len(test),
    "visible-test molecules": test["molecule_id"].nunique(),
    "sample-submission rows": len(sample),
    "test peak arrays aligned": int(alignment_ok.sum()),
    "test spectra base-peak normalised": int(normalisation_ok.sum()),
    "test m/z arrays sorted": int(sorted_ok.sum()),
}, name="value")
display(Markdown("**Table 3. Dataset integrity and label cardinality.**"))
display(audit.to_frame().style.format({"value": "{:,}"}))

*Table 3. File integrity and label cardinality. Counts distinguish spectra from training connectivities. The peak-array checks run on whichever test file is mounted, including the hidden file at scoring time.*


The scorer compares tautomer-standardised connectivity keys (Section 2). The stored `inchikey14` column skips the tautomer step, so for some structures it differs from the key the scorer computes: the training file stores carvone as `ULDHMXUKGWMISQ`, while the scorer computes `YAHNGTQFNICUIJ`. Identity-sensitive joins, deduplication and validation must account for this tautomer step. The stored field is used for the preliminary file counts above; final candidate deduplication follows the scorer identity. [The notebook cited in Section 2](https://www.kaggle.com/code/dariushafshar/casmi26-10-4-of-keys-change-0-48-cost-you) and [section 4 of the data-pitfalls notebook](https://www.kaggle.com/code/kitopl/casmi-2026-data-pitfalls-that-break-your-cv) measure how often the two keys disagree.


## 4. Visible test: molecules, spectra, adducts, and quality

The released visible test is a **format example**: its molecule groups, acquisition settings and peak arrays show what the model must accept, but it carries no structure answers. The attached file holds 400 molecules and 1,213 spectra, all measured on timsTOF: one to nine spectra per molecule, a median of three. Most molecules (86%) have spectra at more than one collision energy (CE), 27% have more than one ion form, 24% have spectra in both polarities, and no two spectra of a molecule share every setting.

**From a group of spectra to one answer.** The target is one structure per `molecule_id`, and every spectrum of the group points to the same answer, so the model never predicts per spectrum. Library search compares each spectrum with the references and keeps the strongest match as the molecule's library similarity. The fingerprint network reads up to 16 spectra of the molecule at once, together with their ion form, polarity and energy. The forward models compare their predicted spectrum with each measured view (Section 10).

Figure 3 shows one group of spectra. Figure 4 counts spectra per molecule, shows how many peaks they carry and places the molecules on the mass axis; Table 4 summarises the same counts. A longer peak list carries more fragment masses, and it can also carry noise or co-isolated ions (other ions entering the precursor isolation window alongside the target).


In [ ]:
full_example_group = max((group for _, group in test.groupby("molecule_id", sort=False)), key=len)
example_group = full_example_group.head(3)
n_views = len(example_group)
fig, axes = plt.subplots(1, n_views + 1, figsize=(5 + 3.4 * n_views, 3.5),
                         gridspec_kw={"width_ratios":[.75] + [1] * n_views})
axes = np.atleast_1d(axes)
axes[0].axis("off")
axes[0].text(.5, .60, "one underlying\nmolecule", ha="center", va="center", fontsize=14, weight="bold",
             bbox=dict(boxstyle="circle,pad=.62", fc="#dcefe7", ec="#2a9d8f", lw=2))
axes[0].annotate(f"{n_views} of {len(full_example_group)} measured views shown", xy=(.50,.23), xytext=(.50,.08), ha="center", color="#46566b",
                 arrowprops=dict(arrowstyle="-|>", color="#2a9d8f", lw=1.6))
for ax, (_, row) in zip(axes[1:], example_group.iterrows()):
    mz = np.asarray(row["ms2_mzs"], dtype=float); intensity = np.asarray(row["ms2_normalized_intensities"], dtype=float)
    keep = np.argsort(intensity)[-45:]
    ax.vlines(mz[keep], 0, intensity[keep], color="#d1495b", lw=1.1)
    energy_values = np.atleast_1d(np.asarray(row["collision_energy_ev"], dtype=float))
    energy = ", ".join(f"{x:g}" for x in energy_values)
    ax.set(title=f"{row['adduct']} · CE {energy} eV", xlabel="fragment m/z", ylim=(0,1.05))
    ax.spines[["top","right"]].set_visible(False)
axes[1].set_ylabel("relative intensity")
fig.suptitle("Figure 3. One molecule can produce several complementary MS/MS spectra", fontsize=16, weight="bold")
plt.tight_layout()
plt.show()

*Figure 3. Several acquisitions of one visible-test molecule: three of the views of the molecule with the most spectra. The molecular identity is shared; the measured precursor ion and fragment pattern change with adduct and acquisition conditions. The output is one ranked list for the whole group. Each view draws its 45 strongest peaks.*


In [ ]:
import re
spectra_per_molecule = test.groupby("molecule_id").size()
_ion_forms = test.groupby("molecule_id")[["adduct", "ionization_mode"]].apply(lambda g: g.drop_duplicates().shape[0])
_settings = test.groupby("molecule_id")["collision_energy_orig"].apply(lambda s: s.astype(str).nunique())
test_summary = pd.DataFrame({
    "statistic": ["min", "median", "mean", "max"],
    "spectra per molecule": [spectra_per_molecule.min(), spectra_per_molecule.median(), spectra_per_molecule.mean(), spectra_per_molecule.max()],
    "ion forms per molecule": [_ion_forms.min(), _ion_forms.median(), _ion_forms.mean(), _ion_forms.max()],
    "collision settings per molecule": [_settings.min(), _settings.median(), _settings.mean(), _settings.max()],
    "peaks per spectrum": [test_peak_counts.min(), test_peak_counts.median(), test_peak_counts.mean(), test_peak_counts.max()],
})
display(Markdown("**Table 4. Spectra, ion forms, collision settings and peaks in the visible test.**"))
display(test_summary.style.format({c: "{:.2f}" for c in test_summary.columns if c != "statistic"}).hide(axis="index"))

# Neutral monoisotopic mass: visible molecules from precursor m/z and adduct (median over their spectra),
# training structures from their molecular formulas.
_ADDUCT_SHIFT = {"[M+H]+": (1, 1.007276), "[M+Na]+": (1, 22.989218), "[M+K]+": (1, 38.963158), "[M+NH4]+": (1, 18.033823),
                 "[M-H]-": (1, -1.007276), "[M+Cl]-": (1, 34.969402), "[M+CH2O2-H]-": (1, 44.998201), "[M+FA-H]-": (1, 44.998201),
                 "[M+H-H2O]+": (1, -17.003289), "[M-H2O+H]+": (1, -17.003289), "[2M+H]+": (2, 1.007276), "[2M-H]-": (2, -1.007276)}
def neutral_mass(mz, adduct):
    k, shift = _ADDUCT_SHIFT.get(str(adduct), (None, None))
    return (float(mz) - shift) / k if k else np.nan
_MONO = {"C": 12.0, "H": 1.00782503207, "N": 14.0030740048, "O": 15.99491461956, "P": 30.97376163, "S": 31.97207100,
         "F": 18.99840322, "Cl": 34.96885268, "Br": 78.9183371, "I": 126.904473, "Si": 27.9769265325, "B": 11.0093054, "Se": 79.9165213}
def formula_mass(f):
    if not isinstance(f, str):
        return np.nan
    parts = re.findall(r"([A-Z][a-z]?)(\d*)", f)
    if not parts or any(el not in _MONO for el, _ in parts):
        return np.nan
    return float(sum(_MONO[el] * (int(n) if n else 1) for el, n in parts))
test_mass = (test.assign(M=[neutral_mass(m, a) for m, a in zip(test["precursor_mz"], test["adduct"])])
             .groupby("molecule_id")["M"].median())
_formula_mass = {f: formula_mass(f) for f in train["molecular_formula"].dropna().unique()}
train_structure_mass = train.drop_duplicates("inchikey14")["molecular_formula"].map(_formula_mass)

display(Markdown("**Figure 4. Spectra per molecule, peaks per spectrum and neutral mass of the visible test.**"))
fig, axes = plt.subplots(3, 1, figsize=(10.5, 13))
sns.histplot(spectra_per_molecule, discrete=True, ax=axes[0], color="#2a9d8f")
axes[0].set(title="Spectra per molecule (complementary views of one answer)", xlabel="spectra", ylabel="molecules")
sns.histplot(test_peak_counts, bins=40, ax=axes[1], color="#457b9d")
axes[1].set(title="Fragment peaks per visible-test spectrum", xlabel="number of peaks")
_bins = np.arange(0, 1400, 20)
axes[2].axvspan(157, 1159, color="#f4a261", alpha=.15, label="hidden-test range 157-1,159 Da")
sns.histplot(train_structure_mass.clip(upper=1390), bins=_bins, stat="density", ax=axes[2], color="#adb5bd", label="training structures")
sns.histplot(test_mass.clip(upper=1390), bins=_bins, stat="density", ax=axes[2], color="#e76f51", alpha=.75, label="visible-test molecules")
axes[2].axvline(194.0804, color="#16324f", ls=":", lw=1.2); axes[2].text(200, axes[2].get_ylim()[1] * .9, "caffeine", color="#16324f", fontsize=10)
axes[2].set(title="Neutral monoisotopic mass", xlabel="mass (Da)", ylabel="density", xlim=(0, 1400))
axes[2].legend(frameon=False)
plt.tight_layout()
plt.show()
display(Markdown(f"{train_structure_mass.between(157, 1159).mean():.1%} of training structures lie inside "
                 f"157–1,159 Da. Median neutral mass is {train_structure_mass.median():.0f} Da in training "
                 f"and {test_mass.median():.0f} Da in the visible test."))


*Table 4 and Figure 4. Spectra per molecule, peak counts and neutral mass in the mounted test file. Table 4 also counts the distinct ion forms and collision settings per molecule. The bottom panel converts each molecule's precursor m/z into its neutral monoisotopic mass with the adduct equations of Section 7 and draws it against the masses of the training structures (computed from their formulas); the shaded band is the published hidden-test range of 157–1,159 Da. Figure 5 and Table 5 show which adducts and polarities these acquisitions contain.*


In [ ]:
display(Markdown("**Figure 5. Adduct and polarity composition of the visible test.**"))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
test["adduct"].value_counts().sort_values().plot.barh(ax=axes[0], color="#577590")
axes[0].set(title="Visible-test adducts", xlabel="spectra", ylabel="adduct")
test["ionization_mode"].value_counts().plot.pie(ax=axes[1], autopct="%1.1f%%",
                                                colors=["#43aa8b", "#f9c74f"])
axes[1].set(title="Ionisation modes", ylabel="")
plt.tight_layout()
plt.show()

display(Markdown("**Table 5. Cross-tabulation of normalised adduct and ionisation polarity.**"))
display(pd.crosstab(test["adduct"], test["ionization_mode"], margins=True))

*Figure 5 and Table 5. Ion forms in the visible test. Adduct and polarity determine the equation that recovers the neutral molecular mass, so they constrain candidate generation before spectral ranking.*


Figure 5 and Table 5 show the ion forms: `[M+H]+` in 959 of the 1,213 spectra, `[M-H]-` in 193, and five rarer adducts (formate, sodium, ammonium, potassium, chloride). Table 3 confirmed that every peak array is well formed; Section 6 examines whether the peaks are informative.

The organisers describe hidden molecules spanning **157–1,159 Da** in neutral monoisotopic mass and **1–16 spectra per molecule**, acquired on timsTOF ([official data description](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data)). The bottom panel of Figure 4 shows that the training structures already fill this range: half of them lie between about 300 and 370 Da, nine in ten between 250 and 580 Da, and 99% inside the hidden range. Mass also correlates with chemical class. Below about 200 Da sit small metabolites such as amino acids, simple phenolics and caffeine (194 Da). Most natural products lie between 250 and 500 Da: flavonoid aglycones around 270–320 Da, many alkaloids, and terpenoids built from C₅ units, from about 220 Da (C₁₅) to about 450 Da (C₃₀) once oxygens are counted. Above 600 Da sit glycosides, where each attached hexose adds 162 Da, saponins and complex lipids. Figure 16 shows the same scale for each natural-product class. Inference therefore has to handle every adduct and the full published mass range, including cases absent from the visible sample.


## 5. Training corpus: scale, provenance, and overlap

`ingest_lib` names source collections, not molecular classes. The training corpus includes experimental libraries such as [GNPS](https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/), [MassBank](https://github.com/MassBank/MassBank-data) and [MoNA](https://massbank.us/), Enveda acquisitions, and upstream labels such as RIKEN, MS-DIAL and `pluskal_ms2`. These collections overlap in chemistry but differ in instruments, processing and replication; Table 6 lists their labels and counts.

Table 6 reports two counts per library, **spectra** and **distinct stored InChIKey14 groups**; their ratio measures replication, because a large row count can reflect broader chemistry, more repeated measurements, or both. Figure 6 shows the same counts as bars. `enveda-180`, measured on timsTOF like the hidden test, contributes both the most spectra (1.15 million) and the most distinct structures (about 183,000 of the 275,810). The ratio of the two, spectra per structure, is about 6 for `enveda-180` and 5 for GNPS, but about 11 for `pluskal_ms2` and over 20 for RIKEN, which measured fewer structures more often.


In [ ]:
library_stats = (train.groupby("ingest_lib")
                 .agg(spectra=("inchikey14", "size"), structures=("inchikey14", "nunique"))
                 .sort_values("spectra", ascending=False))
library_stats["spectra_per_structure"] = library_stats["spectra"] / library_stats["structures"]
LIBRARY_DESCRIPTION = {"enveda-180":"Enveda synthetic, instrument-matched collection",
 "pluskal_ms2":"Pluskal laboratory MS/MS compilation", "riken":"RIKEN metabolomics / natural products",
 "gnps":"Global Natural Products Social Molecular Networking", "massbank":"curated public high-resolution spectra",
 "mona":"MassBank of North America", "spectraverse":"community spectrum repository",
 "msdial":"MS-DIAL public library", "drug_plus":"drug-like reference set",
 "enveda-np-examples":"organiser natural-product examples", "masaryk":"Masaryk University spectra"}
library_stats.insert(0, "collection", [LIBRARY_DESCRIPTION.get(x, "source collection") for x in library_stats.index])
display(Markdown("**Table 6. Spectra, stored InChIKey14 groups, and replication by source library.**"))
display(library_stats.style.format({
    "spectra": "{:,}", "structures": "{:,}", "spectra_per_structure": "{:.1f}"
}).background_gradient(subset=["spectra", "structures"], cmap="Blues"))

display(Markdown("**Figure 6. Relative scale of the source libraries.**"))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
library_stats.sort_values("spectra")["spectra"].plot.barh(ax=axes[0], color="#277da1")
axes[0].set(title="Spectra by source library", xlabel="rows", ylabel="")
library_stats.sort_values("structures")["structures"].plot.barh(ax=axes[1], color="#90be6d")
axes[1].set(title="Unique stored InChIKey14 groups by source", xlabel="structures", ylabel="")
for ax in axes:
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
plt.tight_layout()
plt.show()

*Table 6 and Figure 6. Training-library scale. `spectra` counts acquisitions; `structures` counts unique stored InChIKey14 groups; their ratio measures experimental replication. Table 7 and Figure 7 then look at repeated measurements and recurrence across libraries.*

In [ ]:
structure_repeats = train.groupby("inchikey14").size()
library_pairs = train[["inchikey14", "ingest_lib"]].drop_duplicates()
libraries_per_structure = library_pairs.groupby("inchikey14").size()

repeat_table = pd.DataFrame({
    "spectra_per_structure": structure_repeats.describe(percentiles=[.25, .5, .75, .9, .95, .99]),
})
display(Markdown("**Table 7. Spectra per stored InChIKey14 group (before scorer tautomer canonicalisation).**"))
display(repeat_table)

display(Markdown("**Figure 7. Replication depth and cross-library recurrence.**"))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.histplot(np.log10(structure_repeats), bins=50, ax=axes[0], color="#4d908e")
axes[0].set(title="Long-tailed spectral replication", xlabel="log10(spectra per stored InChIKey14)")
libraries_per_structure.value_counts().sort_index().plot.bar(ax=axes[1], color="#f9844a")
axes[1].set(title="Source libraries per stored structure key", xlabel="number of libraries", ylabel="stored keys")
plt.tight_layout()
plt.show()

multi_library = int((libraries_per_structure > 1).sum())
display(Markdown(f"{multi_library:,} stored connectivity groups "
                 f"({multi_library / len(libraries_per_structure):.2%}) occur in more than one source library."))

*Table 7 and Figure 7. Repeated measurements and cross-library overlap. A connectivity recorded in several libraries (10% of the stored training structures) has spectra from more than one source; the library names alone do not show that those records are independent or correctly labelled.*


**Complementary views and replicates.** Table 7 and Figure 7 show the long tail of spectra per structure (a median of 6, up to 1,468; Section 3). Those spectra differ in the conditions of the measurement. Of the stored training structures, 85% have spectra at more than one collision setting, 55% at more than one ion form, 35% in both polarities, 10% in more than one library and 7% on more than one instrument family. Each setting breaks the molecule differently, so several settings supply more structural evidence than one. About 27% of the rows repeat a setting that the same structure already has (same ion form, polarity, collision setting and instrument family), and 57,642 rows are byte-identical copies of another spectrum of the same structure, almost all of them the same record held by two libraries. Section 6 recommends deduplicating the byte-identical rows before library search and weighting training by structure, so that heavily measured molecules do not dominate.

The long tail also explains why a random row split is misleading: it can leave near-identical spectra of a validation molecule in the search library. Such a split is a valid test of **reference retrieval** but an optimistic test of **generalisation to a molecule with no reference spectrum**.

`enveda-180` contributes 1.15 million of the 2.54 million training spectra, the largest share and under half of the rows, and sharing the timsTOF instrument family does not place two libraries in the same region of chemical space (Figure 15). Holding out the smaller `enveda-np-examples` collection alone leaves matching structures and spectra in other sources, and the frozen model artifacts may already have seen them; Table 16 in Section 11 shows this for the full1 fingerprint model, whose stronger result on that panel agrees with its exposure to those molecules during fitting. A structure-disjoint test removes all spectra of each held-out connectivity from every library, not only from the named source; [section 6 of the data-pitfalls notebook](https://www.kaggle.com/code/kitopl/casmi-2026-data-pitfalls-that-break-your-cv) gives a worked example of this leakage.


## 6. Missingness, label-quality signals, and outliers

Missing metadata and suspicious measurements are different problems: a source can omit the raw intensity and still supply a usable normalised spectrum, and a finite precursor mass can still conflict with the labelled structure and adduct. Tables 8–10 and Figures 8–9 quantify both. The handling table at the end of this section gives the size of each issue and the recommended treatment; each rule is a candidate to be checked on held-out identities.


In [ ]:
missing = (train.isna().mean().mul(100).sort_values(ascending=False).rename("missing_%"))
display(Markdown("**Table 8. Missing metadata by field.**"))
display(missing.to_frame().style.format("{:.2f}"))

display(Markdown("**Figure 8. Fields with incomplete training metadata.**"))
fig, ax = plt.subplots(figsize=(10, 5))
missing[missing > 0].sort_values().plot.barh(ax=ax, color="#bc4749")
ax.set(title="Training metadata missingness", xlabel="missing rows (%)", ylabel="")
plt.tight_layout()
plt.show()

*Table 8 and Figure 8. Missing metadata by field. The pattern identifies which inputs need explicit missing-value handling; it does not by itself show that the associated spectra are of poor quality.*


In [ ]:
numeric_summary = train[["precursor_mz", "precursor_error_ppm", "num_peaks", "base_peak_intensity"]].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).T
display(Markdown("**Table 9. Numerical metadata summary with robust quantiles.**"))
display(numeric_summary.style.format("{:.3g}"))

quality_counts = pd.Series({
    "|precursor error| > 5 ppm": train["precursor_error_ppm"].abs().gt(5).sum(),
    "|precursor error| > 20 ppm": train["precursor_error_ppm"].abs().gt(20).sum(),
    "precursor m/z > 2,000": train["precursor_mz"].gt(2000).sum(),
    "fewer than 5 peaks": train["num_peaks"].lt(5).sum(),
    "more than 2,000 peaks": train["num_peaks"].gt(2000).sum(),
}, name="rows")
display(Markdown("**Table 10. Explicit quality-control flags.**"))
display(quality_counts.to_frame().assign(percent=lambda x: 100*x["rows"]/len(train)).style.format({"rows": "{:,}", "percent": "{:.3f}%"}))

*Table 9 gives percentile summaries (1st to 99th) of precursor m/z, precursor error, peak count and base-peak intensity; Table 10 counts explicit quality flags (absolute precursor error above 5 and above 20 ppm, precursor m/z above 2,000, fewer than 5 peaks, more than 2,000 peaks). Together they separate the typical acquisition regime from records that require clipping, filtering or separate validation.*


In [ ]:
display(Markdown("**Figure 9. Peak-count, precursor-error, and precursor-mass distributions.**"))
fig, axes = plt.subplots(3, 1, figsize=(10.5, 13))
sns.histplot(np.log10(train["num_peaks"]), bins=60, ax=axes[0], color="#577590")
axes[0].set(title="Peak-count tail", xlabel="log10(number of peaks)")

ppm_clip = train["precursor_error_ppm"].dropna().clip(-25, 25)
sns.histplot(ppm_clip, bins=80, ax=axes[1], color="#43aa8b")
axes[1].set(title="Precursor error (clipped for display)", xlabel="error (ppm)")

mass_cap = train["precursor_mz"].quantile(.995)
sns.histplot(train.loc[train["precursor_mz"] <= mass_cap, "precursor_mz"], bins=80, ax=axes[2], color="#f9844a")
axes[2].set(title="Precursor m/z (≤99.5th percentile)", xlabel="precursor m/z")
plt.tight_layout()
plt.show()

*Figure 9. Display transforms reveal the central distributions: peak counts are log10-transformed, precursor errors outside ±25 ppm are clipped to the boundary bins, and precursor m/z values above the 99.5th percentile are omitted from the bottom panel. These plotting choices do not filter the model inputs.*

An extreme value in Figure 9 marks a record for inspection: a precursor error of 80 ppm can reflect coarse mass precision, a wrong adduct label or an incorrect structure label, and a two-peak spectrum may be a real measurement with almost no fragment information. Two fields need separate treatment. `precursor_error_ppm` is computed from the labelled structure, so it exists only for training rows: it can clean the reference library but can never be a feature for a hidden query. Missing `base_peak_intensity` can reveal which library exported the spectrum (Figure 8), so a model can learn a source shortcut; any quality filter or source prior therefore has to be evaluated on held-out identities.

### Cleaning and feature choices

The counts refer to the current training file (Tables 8–10 and Figures 8–9 show the distributions); each recommended handling is a rule to be checked on held-out identities.

| Issue | Size in the training file | Recommended handling |
|---|---|---|
| Missing `base_peak_intensity` | 46% of rows | Normalised intensities as the signal; the missing flag is not a feature, because it identifies the source library rather than the molecule. |
| No collision energy | 13% of rows | Spectra kept, with an explicit "energy unknown" state instead of an imputed number. |
| Several energies merged into one spectrum | 19.5% of rows | A separate acquisition type; averaging the energies into one number would hide the merge. |
| Absolute precursor error above 20 ppm | about 120,000 rows | Usually a wrong adduct, a wrong label or coarse precision: removal from the reference library, or a wider mass window for those sources only (a related count, the 121,805 spectra that [a fixed ppm window loses](https://www.kaggle.com/code/dariushafshar/casmi26-your-ppm-window-drops-121-805-spectra), is discussed in Section 7). |
| Fewer than 3 peaks | about 99,000 rows | Almost no structural information: low weight in training, weak library evidence only. |
| Neutral mass outside 157–1,159 Da | 1% of structures, 2.4% of rows | Use the published mass range as a candidate-search bound, with a margin for measurement error; the spectra can still teach fragmentation. |
| Byte-identical spectra of one structure | 57,642 rows | Deduplication before library search, so that a copied record is not counted twice. |
| Repeated settings of one structure | 27% of rows | Training weighted by structure rather than by row, so that heavily measured molecules do not dominate. |
| Stored versus scorer key | some structures | Grouping, joins and deduplication by the recomputed scorer key (Section 3). |

**Measurement-derived features.** From the measurement: the neutral mass from the adduct equation (Section 7), polarity, ion form, collision energy with an "unknown" flag, instrument family, peak count, [spectral entropy](https://doi.org/10.1038/s41592-021-01331-z) and the neutral losses between the precursor and the strong peaks. From the comparison with each candidate: the best library similarity, the predicted-fingerprint agreement, the share of peak intensity explained by candidate fragments and the forward-model similarity (Section 10 uses all of these). Fields known only because the answer is known, such as `precursor_error_ppm` and the source library, are excluded.


## 7. Instruments, collision energy, ion mode, and adduct shift

Spectral similarity depends on how the spectrum was acquired: polarity and adduct fix the ion chemistry, collision energy changes the fragment pattern, and instrument and preprocessing decide which signals survive. Figure 10 shows how widely these conditions vary across the corpus; Table 11 lists the most common adducts with their polarity, and its continuation compares the instrument families. The pipeline of Section 10 passes adduct, polarity and energy to the fingerprint model and restricts GLACIER to `[M+H]+` spectra.


In [ ]:
def instrument_family(label):
    s = str(label).lower() if isinstance(label, str) else ""
    if not s or s in ("nan", "none"):
        return "missing"
    if "tims" in s:
        return "timsTOF"
    if "orbitrap" in s or "qft" in s or "itft" in s or "exactive" in s or "hcd" in s or ("ft" in s and "tof" not in s):
        return "Orbitrap / FT"
    if "tof" in s:
        return "QTOF / TOF"
    if "qqq" in s or "qq" in s or "triple" in s:
        return "triple quadrupole"
    if "it" in s or "trap" in s:
        return "ion trap"
    return "other"

train["instrument_family"] = train["instrument_type"].map(instrument_family)
family_table = (train.groupby("instrument_family")
                .agg(spectra=("instrument_family", "size"), structures=("inchikey14", "nunique"),
                     median_abs_ppm=("precursor_error_ppm", lambda x: float(np.nanmedian(np.abs(x)))),
                     median_peaks=("num_peaks", "median"))
                .sort_values("spectra"))
display(Markdown("**Figure 10. Experimental domains in the training corpus.**"))
fig, axes = plt.subplots(3, 1, figsize=(10.5, 13))
train["ionization_mode"].value_counts().plot.bar(ax=axes[0], color=["#277da1", "#f9c74f"])
axes[0].set(title="Ionisation mode", xlabel="", ylabel="spectra")
train["collision_energy_orig_units"].value_counts().plot.bar(ax=axes[1], color="#90be6d")
axes[1].set(title="Original collision-energy units", xlabel="", ylabel="spectra")
axes[2].barh(family_table.index, family_table["spectra"] / 1e3, color="#f3722c")
for y, (n, pk, ppm) in enumerate(zip(family_table["spectra"], family_table["median_peaks"], family_table["median_abs_ppm"])):
    axes[2].text(n / 1e3 + family_table["spectra"].max() / 1e3 * .01, y, f"{pk:.0f} peaks · {ppm:.1f} ppm", va="center", fontsize=10)
axes[2].set(title="Instrument families: spectra, median peaks per spectrum, median precursor error", xlabel="spectra (thousands)",
            xlim=(0, family_table["spectra"].max() / 1e3 * 1.3))
plt.tight_layout()
plt.show()

adduct_table = pd.crosstab(train["adduct"], train["ionization_mode"]).assign(total=lambda x: x.sum(axis=1))
display(Markdown("**Table 11. Most frequent adducts by ionisation polarity.**"))
display(adduct_table.sort_values("total", ascending=False).head(25).style.format("{:,}"))
display(Markdown("**Table 11 (continued). Instrument families.** Free-text instrument labels grouped by analyser type."))
display(family_table.sort_values("spectra", ascending=False).style.format(
    {"spectra": "{:,}", "structures": "{:,}", "median_abs_ppm": "{:.1f}", "median_peaks": "{:.0f}"}))
display(Markdown(f"The training file contains {train['instrument_type'].nunique(dropna=True)} distinct instrument labels."))


*Figure 10 and Table 11. Acquisition heterogeneity. Polarity, energy units, instrument families and adducts form distinct experimental domains that a search score needs to distinguish. The bottom panel groups the free-text instrument labels into families; its labels give the median number of peaks per spectrum and the median absolute precursor error. The continuation of Table 11 lists the same families in numbers.*


The middle panel of Figure 10 shows collision energy arriving in three units: eV (an absolute energy), NCE (normalised collision energy, a percentage that Orbitrap software scales with the precursor m/z) and V (volts); 13% of the rows carry no energy at all. The file converts them into one column, `collision_energy_ev`, which serves as the common input. The original unit is kept as well, because the conversion from NCE is approximate, and the energy list of a merged spectrum is kept rather than averaged. In negative mode the raw energy field may carry a minus sign that marks the polarity; `collision_energy_ev` holds the absolute value.

Mass filtering must use the full adduct equation. For a singly protonated molecule, the neutral mass is the precursor m/z minus the proton mass. For a singly charged `[M+Na]+`, the sodium-ion mass is subtracted instead; for a dimer such as `[2M+H]+`, the proton mass is subtracted and the remainder divided by two. An incorrect equation can discard the correct structure before the ranker sees it. The [Fiehn lab adduct table](https://fiehnlab.ucdavis.edu/staff/kind/metabolomics/ms-adduct-calculator) lists the equation for each common adduct. [A public notebook on mass windows](https://www.kaggle.com/code/dariushafshar/casmi26-your-ppm-window-drops-121-805-spectra) shows that a purely relative window also loses 121,805 training spectra whose precursor m/z is reported at coarse absolute precision or disagrees with its adduct label, and proposes a hybrid 10 ppm / 0.01 Da tolerance; the pipeline of Section 10 instead searches within 10 ppm of the estimated neutral mass and widens the window to 30 ppm when it is empty.

**Instrument effects.**
1. **Mass accuracy.** Time-of-flight and [Orbitrap](https://en.wikipedia.org/wiki/Orbitrap) instruments measure the precursor within about 1–2 ppm (median absolute error, Figure 10), whereas the [triple quadrupoles](https://en.wikipedia.org/wiki/Triple_quadrupole_mass_spectrometer) in the training data report it at nominal precision, a median of about 80 ppm. Their reference spectra require wider mass windows or fragment-only matching.
2. **Peak lists.** The timsTOF spectra here carry a median of about 140 peaks (Figure 10), including many weak ones, whereas curated library spectra from Orbitrap and QTOF instruments carry about 14–22. Spectra are comparable only after the same cleaning (an intensity floor or the top-k peaks) and with a similarity measure that is not dominated by peak count.
3. **Fragmentation physics.** [Collision-induced dissociation](https://en.wikipedia.org/wiki/Collision-induced_dissociation) in a collision cell (quadrupole time-of-flight (QTOF), timsTOF and Orbitrap higher-energy collisional dissociation (HCD)) gives similar patterns at similar energies. Resonance excitation in an [ion trap](https://en.wikipedia.org/wiki/Quadrupole_ion_trap) is gentler and cannot retain fragments below roughly a third of the precursor m/z, so its spectra look different. timsTOF adds a [trapped ion-mobility](https://en.wikipedia.org/wiki/Ion-mobility_spectrometry%E2%80%93mass_spectrometry) stage that separates ions by shape before the time-of-flight analyser; it changes which ions are isolated together, not the fragmentation itself.

The hidden test is all timsTOF, so its closest references are timsTOF and other collision-cell spectra; ion-trap and triple-quadrupole references remain useful but warrant lower weight in a similarity score.

Figure 10 groups spelling variants of instrument labels into families. Search preprocessing also needs to distinguish acquisition regimes and apply the right mass equation for every supported adduct, including those absent from the visible file.


## 8. What an MS/MS spectrum looks like in these files

Figures 11 and 12 plot fragment m/z against intensity relative to the base peak: each vertical line is one detected fragment ion. Figure 11 shows three visible-test spectra with different peak counts; Figure 12 shows several acquisitions of one molecule. *A Complete Statistical Tour*, in [its own section 8](https://www.kaggle.com/code/josefreitasalvesneto/casmi-2026-a-complete-statistical-tour), shows one spectrum before and after removing low-intensity noise peaks.


In [ ]:
def read_rows_by_position(path, positions, columns):
    # Read a few global Parquet row positions without materialising all peak arrays.
    pf = pq.ParquetFile(path)
    positions = np.asarray(sorted(set(map(int, positions))), dtype=np.int64)
    chunks = []
    start = 0
    for rg in range(pf.metadata.num_row_groups):
        n = pf.metadata.row_group(rg).num_rows
        mask = (positions >= start) & (positions < start + n)
        if mask.any():
            local = pa.array(positions[mask] - start)
            chunks.append(pf.read_row_group(rg, columns=columns).take(local))
        start += n
    if not chunks:
        return pd.DataFrame(columns=columns)
    return pa.concat_tables(chunks).to_pandas()


def plot_spectrum(ax, mzs, intensities, title, color="#264653", top_n=None):
    mzs = np.asarray(mzs, dtype=float)
    intensities = np.asarray(intensities, dtype=float)
    if top_n is not None and len(mzs) > top_n:
        keep = np.argpartition(intensities, -top_n)[-top_n:]
        mzs, intensities = mzs[keep], intensities[keep]
        order = np.argsort(mzs)
        mzs, intensities = mzs[order], intensities[order]
    ax.vlines(mzs, 0, intensities, lw=0.8, color=color)
    ax.set(title=title, xlabel="fragment m/z", ylabel="relative intensity")
    ax.set_ylim(0, 1.05)


display(Markdown("**Figure 11. MS/MS spectra with low, median and high peak counts.**"))

# Pick three visible-test spectra spanning low/median/high peak counts.
q_targets = test_peak_counts.quantile([.1, .5, .9]).to_numpy()
chosen = []
for target in q_targets:
    idx = (test_peak_counts - target).abs().idxmin()
    if idx not in chosen:
        chosen.append(idx)

fig, axes = plt.subplots(len(chosen), 1, figsize=(10.5, 3.2 * len(chosen)))
axes = np.atleast_1d(axes)
for ax, idx in zip(axes, chosen):
    row = test.loc[idx]
    plot_spectrum(ax, row["ms2_mzs"], row["ms2_normalized_intensities"],
                  f"{row['spectrum_id']} · {row['adduct']} · precursor {row['precursor_mz']:.4f} · "
                  f"{len(row['ms2_mzs'])} peaks", top_n=500)
plt.tight_layout()
plt.show()

*Figure 11. Three visible-test spectra chosen at the 10th, 50th and 90th percentiles of the peak-count distribution. Fragment density and m/z coverage differ widely between them; the peak count reflects the acquisition (Section 7) and is no measure of molecular complexity or identification confidence. At most the 500 strongest peaks of a spectrum are drawn.*


In [ ]:
display(Markdown("**Figure 12. Multiple collision-energy and adduct views of one molecule.**"))

# Multiple spectra of one molecule: evidence changes with adduct and collision energy.
example_id = spectra_per_molecule.sort_values(ascending=False).index[0]
example = test[test["molecule_id"].eq(example_id)].head(6)
fig, axes = plt.subplots(len(example), 1, figsize=(10.5, 2.7 * len(example)), sharex=True)
axes = np.atleast_1d(axes)
for ax, (_, row) in zip(axes, example.iterrows()):
    energy_unit = (row['collision_energy_orig_units']
                   if pd.notna(row['collision_energy_orig_units']) else 'unit not recorded')
    plot_spectrum(ax, row["ms2_mzs"], row["ms2_normalized_intensities"],
                  f"{example_id} · {row['adduct']} · CE {row['collision_energy_orig']} {energy_unit}", top_n=350)
plt.tight_layout()
plt.show()

*Figure 12. Repeated MS/MS views of one molecule, the visible-test molecule with the most spectra (up to six views shown). The panel titles report each acquisition's adduct and collision setting; the fragment patterns differ, and Section 10 combines the views into one list per `molecule_id`. The titles show the raw energy field, which carries a minus sign in negative mode; the sign marks the polarity, not a lower energy, and `collision_energy_ev` holds the absolute value. At most the 350 strongest peaks of a view are drawn.*


In Figure 12 the fragment pattern of one molecule changes with adduct and collision energy; Section 10 combines the evidence of all views into one candidate list per `molecule_id`.


## 9. Molecular labels: structures, descriptors, and chemical space

The labels show which molecular graphs the training spectra represent, and the question for modelling is whether these reference molecules resemble the compounds to be identified: a model can transfer poorly even when train and test use the same instrument, if the chemistry differs. Figure 13 shows one natural product per class, Table 12 and Figure 14 summarise the size and polarity of the training structures, Figure 15 maps the sources over chemical space, and Figure 16 measures how much of each natural-product class has been measured at all. The classes come from [NPClassifier](https://doi.org/10.1021/acs.jnatprod.1c00399), which assigns a natural product to one of seven biosynthetic pathways, such as terpenoids or alkaloids.


In [ ]:
# Figure 13: one measured natural product per NPClassifier pathway (natural-product knowledge table) plus a drug-like contrast.
import glob, json

def _first(patterns):
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True), key=len)
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(patterns[0])

np_table = pd.read_parquet(
    _first(['/kaggle/input/**/casmi26-natural-product-knowledge-table/**/natural_products.parquet',
            os.environ.get('CASMI_NP_TABLE', 'natural_products.parquet')]),
    columns=['ik14', 'smiles', 'formula', 'mono_mass', 'n_heavy', 'np_pathway', 'np_superclass', 'top_genera',
             'in_casmi_train', 'in_casmi_pool'])
_lib_of = train.drop_duplicates("inchikey14").set_index("inchikey14")["ingest_lib"]
_gallery_np_keys = [
    "DLNWZIVYKQXLTN",  # alkaloid
    "ODAZGDXFSFEAAA",  # peptide
    "INAPMGSXUVUWAF",  # polyol
    "BXOCHUWSGYYSFW",  # fatty amide (spilanthol-type alkamide)
    "WHNAJPFARNBVDZ",  # xanthone
    "YLZYAUCOYZKLMA",  # isoflavonoid
    "VGNXXIHXIBOUBQ",  # sesquiterpenoid
]
_gallery_drug_key = "RKLNONIVDFXQRX"
gallery = np_table.set_index("ik14").reindex(_gallery_np_keys)
# A reference removed from the current catalogue retains its measured structure,
# with its unavailable catalogue annotation stated explicitly.
for _key in gallery.index[gallery.smiles.isna()]:
    _r = train.loc[train.inchikey14.eq(_key)].iloc[0]
    gallery.loc[_key, ["smiles", "formula", "mono_mass", "np_pathway", "np_superclass", "top_genera"]] = [
        _r.normalized_smiles, _r.molecular_formula, _formula_mass[_r.molecular_formula],
        "Not in current NP catalogue", "unclassified", "n/a"]
gallery = gallery.reset_index()
_np_keys = set(np_table.ik14)
_drug = train[(train.ingest_lib == "drug_plus") & train.inchikey14.eq(_gallery_drug_key)
              & ~train.inchikey14.isin(_np_keys)].drop_duplicates("inchikey14")
assert len(_drug) == 1, "Pinned drug-like reference is missing or now catalogued."
panels = [dict(smiles=r.smiles, title=f"{r.np_pathway} · {r.np_superclass}",
               sub=f"{r.formula} · {r.mono_mass:.1f} Da · genus {str(r.top_genera).split(';')[0] if isinstance(r.top_genera, str) else 'n/a'} · library {_lib_of.get(r.ik14, '')}")
          for r in gallery.itertuples()]
if len(_drug):
    r = _drug.sample(n=1, random_state=SEED).iloc[0]
    panels.append(dict(smiles=r.normalized_smiles, title="Not a catalogued natural product (drug-like)",
                       sub=f"{r.molecular_formula} · {_formula_mass.get(r.molecular_formula, float('nan')):.1f} Da · library {r.ingest_lib}"))
display(Markdown("**Figure 13. One measured natural product per biosynthetic pathway, and a drug-like structure for contrast.**"))
_ncol = 2; _nrow = int(np.ceil(len(panels) / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(12, 4.4 * _nrow))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")
for ax, p in zip(np.atleast_1d(axes).ravel(), panels):
    m = Chem.MolFromSmiles(p["smiles"])
    if m is not None:
        ax.imshow(Draw.MolToImage(m, size=(560, 360), fitImage=True))
    ax.set_title(p["title"] + "\n" + p["sub"], fontsize=10.5, loc="left")
plt.tight_layout(h_pad=2.0)
plt.show()


*Figure 13. Seven measured reference structures, plus one drug-like structure for contrast. The fatty-amide reference is absent from the current natural-product catalogue, so its panel marks the classification as unavailable. Each natural-product panel names the pathway and superclass, formula and neutral mass, a reported organism genus from LOTUS, and one training library containing its spectrum. The drug-like panel gives its formula, mass and library. The pathways differ visibly: terpenoids are carbon skeletons built from C₅ units, usually ring-rich and decorated with a few oxygens; alkaloids carry nitrogen; shikimates and phenylpropanoids carry aromatic rings with oxygen substituents; carbohydrates carry many hydroxyl groups. Each panel is a single example, not a class average.*


In [ ]:
# Descriptor analysis on a fixed, bounded sample of unique stored InChIKey14 groups.
unique_structures = train.drop_duplicates("inchikey14").copy()
n_descriptor = min(20_000, len(unique_structures))
descriptor_sample = unique_structures.sample(n=n_descriptor, random_state=SEED)

records = []
invalid = 0
for _, row in descriptor_sample.iterrows():
    mol = Chem.MolFromSmiles(row["normalized_smiles"])
    if mol is None:
        invalid += 1
        continue
    records.append({
        "ingest_lib": row["ingest_lib"],
        "inchikey14": row["inchikey14"],
        "mol": mol,
        "MolWt": Descriptors.MolWt(mol),
        "cLogP": Descriptors.MolLogP(mol),
        "heavy_atoms": rdMolDescriptors.CalcNumHeavyAtoms(mol),
        "rings": rdMolDescriptors.CalcNumRings(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
    })
desc = pd.DataFrame(records)
display(Markdown(f"**Table 12. Molecular-descriptor summary for {len(desc):,} sampled connectivities.**"))
display(desc.drop(columns="mol").describe().T.style.format("{:.2f}"))

display(Markdown("**Figure 14. Molecular-property distributions.**"))
fig, axes = plt.subplots(3, 2, figsize=(10.5, 12))
for ax, col, color in zip(axes.flat,
                          ["MolWt", "cLogP", "heavy_atoms", "rings", "rotatable_bonds", "TPSA"],
                          sns.color_palette("viridis", 6)):
    cap = desc[col].quantile(.995)
    sns.histplot(desc.loc[desc[col] <= cap, col], bins=45, ax=ax, color=color)
    ax.set_title(f"{col} (≤99.5th percentile)")
plt.tight_layout()
plt.show()

*Table 12 and Figure 14. Molecular-descriptor summary and distributions for a random sample of 20,000 stored training structures. Each structure is described by average molecular weight (g/mol; the mass windows elsewhere use the monoisotopic mass in Da), predicted lipophilicity (cLogP, the preference for oil-like over aqueous surroundings), heavy-atom (non-hydrogen) count, ring count, rotatable-bond count and topological polar surface area (TPSA, Å², a structure-derived estimate of the polar surface); each histogram is cut at the 99.5th percentile. A typical training structure weighs about 330 g/mol, has 20–25 heavy atoms and three rings; the long right tails come from glycosides, peptides and lipids. The values describe sampled training structures, not the complete candidate pool.*


In [ ]:
display(Markdown("**Figure 15. A map of chemical space: source libraries and natural-product pathways.**"))

# Morgan fingerprints (circular substructure fingerprints) projected only for visual EDA.
n_fp = min(6_000, len(desc))
fp_sample = desc.sample(n=n_fp, random_state=SEED).reset_index(drop=True)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)
X = np.zeros((n_fp, 512), dtype=np.uint8)
for i, mol in enumerate(fp_sample["mol"]):
    DataStructs.ConvertToNumpyArray(generator.GetFingerprint(mol), X[i])
svd = TruncatedSVD(n_components=2, random_state=SEED)
projection = svd.fit_transform(X)
fp_sample[["SVD1", "SVD2"]] = projection

top_libs = library_stats.head(7).index
fp_sample["source"] = fp_sample["ingest_lib"].where(fp_sample["ingest_lib"].isin(top_libs), "other")
# A sample of catalogued natural products (knowledge table) placed on the same axes.
_np_s = np_table.dropna(subset=["np_pathway"]).sample(n=min(3_000, int(np_table.np_pathway.notna().sum())), random_state=SEED)
_np_rows, _np_path = [], []
for smi, pw in zip(_np_s.smiles, _np_s.np_pathway):
    with rdBase.BlockLogs():  # catalogue entries may contain an isolated hydrogen
        m = Chem.MolFromSmiles(smi)
    if m is not None:
        v = np.zeros(512, dtype=np.uint8); DataStructs.ConvertToNumpyArray(generator.GetFingerprint(m), v)
        _np_rows.append(v); _np_path.append(pw)
np_proj = pd.DataFrame(svd.transform(np.asarray(_np_rows)), columns=["SVD1", "SVD2"]).assign(pathway=_np_path)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
sns.scatterplot(data=fp_sample, x="SVD1", y="SVD2", hue="source", s=11, alpha=.5, linewidth=0, ax=axes[0])
axes[0].set_title("Training structures, coloured by source library")
axes[1].scatter(fp_sample["SVD1"], fp_sample["SVD2"], s=8, color="#dedede", linewidths=0, label="training structures")
sns.scatterplot(data=np_proj.sort_values("pathway"), x="SVD1", y="SVD2", hue="pathway", palette="tab10", s=11, alpha=.75, linewidth=0, ax=axes[1])
axes[1].set_title("Catalogued natural products on the same map, by pathway")
for ax in axes:
    ax.legend(markerscale=2, fontsize=9, loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

axis_meaning = pd.DataFrame({ax_: {prop: float(np.corrcoef(fp_sample[ax_], fp_sample[prop])[0, 1])
                                    for prop in ["heavy_atoms", "rings", "MolWt", "TPSA", "cLogP", "rotatable_bonds"]}
                              for ax_ in ["SVD1", "SVD2"]})
axis_meaning.loc["variance explained"] = svd.explained_variance_ratio_
display(Markdown("**What the axes follow.** Correlation of each axis with molecular descriptors in this sample (last row: share of fingerprint variance the axis explains)."))
display(axis_meaning.style.format("{:.2f}"))


*Figure 15. A map of chemical space. Each point is one training structure (a sample of 6,000), described by a 512-bit [Morgan fingerprint](https://www.rdkit.org/docs/GettingStartedInPython.html#morgan-fingerprints-circular-fingerprints) (hashed indicators of local atom environments). [Truncated SVD](https://scikit-learn.org/stable/modules/decomposition.html#truncated-singular-value-decomposition-and-latent-semantic-analysis) projects the fingerprint matrix onto two singular-vector directions without centring it. SVD1 and SVD2 are such directions, not chemical quantities: their units are arbitrary, and together they retain under 5% of the fingerprint variance (last row of the table under the figure). The table also shows what the axes follow: SVD1 rises with the number of rings; SVD2 rises with flexible chains (rotatable bonds, lipophilicity, size) and falls with rings, so lipid- and fatty-acid-like molecules sit at the top and compact ring systems lower down. Left, training structures coloured by source library (the seven largest libraries; the rest as "other"). A stored connectivity group is assigned the source of its first training record. Right, the same training structures in grey, with 3,000 catalogued natural products (LOTUS and NPAtlas) placed on the same axes and coloured by NPClassifier pathway.*


On the left, the libraries overlap widely, and only part of GNPS and of the library tagged `spectraverse` reaches the chain-rich top. On the right, the natural-product classes form their own neighbourhoods: fatty acids among the flexible chains at the upper left, terpenoids and phenylpropanoids in the middle, alkaloids towards the ring-rich lower right and peptides at the far right. The densest training region, at the lower right, is mostly the `enveda-180` collection of commercially available synthetic compounds, and few catalogued natural products fall into it: references and training rows are plentiful where the hidden natural products are expected to be scarce. Compressing 512 dimensions into two distorts distances and can create apparent clusters, and one connectivity can carry more than one source label. Figure 16 measures reference-spectrum coverage directly; the summary beneath it also reports the share present in the candidate pool.


### Natural products: what is known and what has been measured

The hidden test comes from biological extracts, so most of its answers are expected to be natural products, compounds made by living organisms. The [natural-product knowledge table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) combines the LOTUS frozen export of 2026-04-13 and NPAtlas into 153,964 entries, one per InChIKey connectivity block. Where LOTUS supplies it, an entry carries its NPClassifier pathway, a broad biosynthetic grouping such as terpenoids or alkaloids. Figure 16 compares the size of each pathway group with the number of its entries that have at least one training spectrum, and shows the mass range of each class.


In [ ]:
# Figure 16: which natural-product classes are known, which of them have a training spectrum, and how heavy they are.
np_mass = pd.read_parquet(
    _first(['/kaggle/input/**/casmi26-natural-product-knowledge-table/**/natural_products.parquet',
            os.environ.get('CASMI_NP_TABLE', 'natural_products.parquet')]),
    columns=['np_pathway', 'in_casmi_train', 'in_casmi_pool', 'mono_mass'])
by_class = (np_mass.dropna(subset=['np_pathway']).groupby('np_pathway')
            .agg(known=('in_casmi_train', 'size'), with_spectrum=('in_casmi_train', 'sum')).sort_values('known'))
by_class['share'] = by_class.with_spectrum / by_class.known
display(Markdown('**Figure 16. Known natural products by biosynthetic class: measured share and mass range.**'))
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(16, 5.6), gridspec_kw={"width_ratios": [1.15, 1]})
ax.barh(by_class.index, by_class.known / 1e3, color='#cfe3e6', label='known structures (LOTUS and NPAtlas)')
ax.barh(by_class.index, by_class.with_spectrum / 1e3, color='#287a8c', label='with a training spectrum')
for y, (known, share) in enumerate(zip(by_class.known, by_class.share)):
    ax.text(known / 1e3 + by_class.known.max() / 1e3 * .012, y, f'{share:.0%} measured', va='center', fontsize=11)
ax.set(xlabel='distinct connectivities (thousands)', xlim=(0, by_class.known.max() / 1e3 * 1.25), title='How much of each class has been measured')
ax.grid(axis='y', visible=False); ax.legend(frameon=False, loc='lower right')
_m = [np_mass.loc[np_mass.np_pathway.eq(p), 'mono_mass'].dropna().values for p in by_class.index]
ax2.axvspan(157, 1159, color="#f4a261", alpha=.15)
ax2.boxplot(_m, vert=False, whis=(5, 95), showfliers=False, widths=.6, patch_artist=True,
            boxprops=dict(facecolor='#cfe3e6', color='#287a8c'), medianprops=dict(color='#16324f'))
ax2.set_yticks(range(1, len(by_class) + 1)); ax2.set_yticklabels([])
ax2.set(xlabel='neutral monoisotopic mass (Da)', xlim=(0, max(1400, max(np.quantile(v, .95) for v in _m) * 1.05)),
        title='Mass range of each class (shaded: hidden-test range)')
plt.tight_layout(); plt.show()
display(Markdown(f"Of {len(np_mass):,} catalogued natural products, {np_mass.in_casmi_train.mean():.1%} have a training "
                 f"spectrum and {np_mass.in_casmi_pool.mean():.1%} occur in the candidate pool. "
                 f"{int(np_mass.np_pathway.isna().sum()):,} entries have no pathway annotation."))
del np_mass, np_table


*Figure 16. Reference-spectrum coverage and mass range within the LOTUS–NPAtlas catalogue, by NPClassifier pathway. Left: light bars count catalogue entries, dark bars count entries with a training spectrum. Right: the neutral monoisotopic mass of each class (box: middle half; whiskers: 5th–95th percentile) with the hidden-test range shaded and the full whiskers visible. Terpenoids are the largest class and the least measured (under 5%); fatty acids are the lightest class, and the amino-acid-and-peptide class the heaviest, because it includes large peptides (the single amino acids of Section 4 lie below about 200 Da). Peptides also span the widest mass range, because every added residue adds 57–186 Da; carbohydrates have the next-widest range, with 162 Da per attached sugar. The 7,101 entries without a pathway annotation are left out. Across the whole table, 8.0% of the entries have a training spectrum and 98.1% occur in the candidate pool: for most entries the pool supplies a candidate but no reference spectrum, so the ranking has to rely on fingerprint, analogue and forward-model evidence (Section 10). These percentages describe the catalogue, not the novelty mixture of the hidden test.*


## 10. The model: combining candidate sources and spectral evidence

The inference pipeline follows [canqiang's E006](https://www.kaggle.com/code/canqiang/casmi26-e006-window-40-ice-fallback-true); the "window 40" in its name is the shortlist of 40 identities that enters the final reordering. Three searches propose candidates, and predicted spectra then refine the order of structures that share a molecular formula (Figure 17). A structure has to enter one of the lists before any scoring model can rank it.

The **main engine** searches the pool of 710,701 training and COCONUT structures within 10 ppm of the neutral mass estimated from the precursor m/z and adduct (the median over the molecule's spectra), widening to 30 ppm when the narrow window is empty, and adds structures generated by modifying measured spectral analogues. FPNet full1 predicts substructure fingerprints from up to 16 spectra and their acquisition metadata; a ranker of four LightGBM boosters combines the fingerprint agreement with measured-spectrum similarity, fragment matches and analogue relations. The best 60 distinct candidates form the main list, whose pool candidates are then reordered by `z(ranker score) + 0.15 · pop`; generated candidates keep their positions.

Two further searches broaden the proposals. The **PubChem channel** screens a mass-sorted catalogue of 105,875,489 PubChem structures and returns up to 25 proposals outside the pool. Its fingerprint logits combine FPNet A+B with half the fold-safe checkpoint's logits, retaining the A+B coefficient in the raw-margin gates; this checkpoint was trained without identity fold 0 and the 250 panel molecules. Candidates are ordered by `S = z(fingerprint score) + 0.15 · pop_PC`, with the enriched documentation prior defined below. The **second engine** (labelled BIO+AFIX in the source) draws candidates from the training structures, COCONUT, the 98,433 standardised ChEBI and LIPID MAPS structures of [sergiomalv](https://www.kaggle.com/datasets/sergiomalv/casmi-chebi-lmsd-structures), and 385,760 PubChemLite candidates, mounted from [CASMI26 Public Models and Candidate Sets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets). It scores them by library matching, analogue propagation, fingerprint models and fragment matching, blends two gradient-boosted rankers and returns up to 40 structures. Both rankers use trees of depth 6 and average eight and four fitted models (four and two random seeds, each with two class weightings). The PubChemLite addition follows [guhongbin](https://www.kaggle.com/code/guhongbin/casmi26-v447-pubchemlite), using the block of [Ahmed Berat Özer](https://www.kaggle.com/code/ahmedberatozer/casmi26-v14-pubchemlite); the PubChem prior weight follows [guhongbin's v465](https://www.kaggle.com/code/guhongbin/casmi26-v465-v15-pop015). The three channels differ in candidate sources and scoring models, so their lists can hold different answers.

Here `pop = ln(1 + substance records) + ln(1 + PubMed articles)` is the **documentation prior**, also called the popularity prior, computed from PubChem counts. The pool reordering uses this unmodified value. In the PubChem channel, `pop_PC = pop + 0.5 · I_spectral + 1.0 · I_NP + 0.5 · ln(1 + LOTUS references)`: `I_spectral` marks a depositor from MassBank Europe, MoNA or the NIST Mass Spectrometry Data Center, and `I_NP` marks COCONUT, LOTUS or NPAtlas membership. The enriched value is also used for the 200-row documentation shortlist and the promotion condition `pop_PC ≥ 5`. A depositor flag does not establish that a usable reference spectrum exists. These counts favour well-documented compounds and do not show that a compound occurs in the sample; Figure 18 compares the unmodified PubChem documentation counts across candidate populations.

ICEBERG and GLACIER predict a spectrum for each candidate of the eligible same-formula groups and compare it with the measured spectra; GLACIER receives only protonated positive ions, `[M+H]+`. Before the merge, their agreement reorders the PubChem list within each molecular formula where ICEBERG returned scores; the main list keeps its ranker order at this stage. PubChem proposals enter the main list only when the molecule's strongest measured-library similarity is below 0.9. They are interleaved at positions 2, 4, 6, 8 and 10 when the fingerprint margin Δ (the best PubChem fingerprint score minus the best pool fingerprint score, before the documentation bonus) exceeds 600, and at positions 4, 8, 12, 16 and 20 otherwise. The leading proposal moves to rank 1 when Δ exceeds 150, or when its documentation-adjusted score exceeds 6 and its enriched documentation prior `pop_PC` is at least 5. The Δ > 150 threshold follows [seyitkaangunes' public comparison](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-pubchem-promo-rel-150-lb-0-446).

The merged main list and the second-engine list are combined by **reciprocal-rank fusion** after structures with the same metric identity (the scorer's tautomer-standardised connectivity key together with the heavy-atom composition) have been collapsed to one vote each:

$$
s(c)=\frac{1}{3+r_{\mathrm{main}}(c)}+\frac{0.6}{3+r_{\mathrm{second}}(c)}.
$$

A candidate absent from one list receives zero from that list. The best 40 fused identities are reordered once more within each molecular formula, by GLACIER where it has usable scores and by ICEBERG where only ICEBERG has scores; where neither has scores, a fragment score takes their place. This MetFrag-lite score enumerates the neutral fragments left after one or two bond cleavages of the candidate and measures the share of the measured intensity they explain, allowing hydrogen shifts and the charge carriers of the precursor adduct; a peak explained only by a two-cleavage fragment counts 0.6. Within a formula group the candidates are then sorted by `z(fused score) + 0.5 · z(fragment score)`. The fragment implementation follows [Ahmed Berat Özer's v7c](https://www.kaggle.com/code/ahmedberatozer/casmi26-v7c-e006-fraggap). Invalid or equivalent guesses are removed, and vacant positions are filled from the remaining ranked candidates (the overflow of the fused, main, second-engine and PubChem lists) and, if places remain, from mass-compatible structures of the DreaMS resource. The output contains up to 25 distinct structures per molecule. A last step, taken from [antriksharya's public notebook](https://www.kaggle.com/code/antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert), moves the leading structure of the second-engine list to rank 2 when it does not already hold rank 1; its other copy is removed, so the list stays at 25 distinct identities. The step guarantees the strongest independent proposal a high place without displacing the fused leader. Section 12 retains the complete E006 implementation; Section 15 credits its component notebooks, models and datasets.


In [ ]:
# Figure 17: schematic only; no inference inputs, state or outputs are accessed.
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 9))
ax.set(xlim=(0, 15), ylim=(0, 12))
ax.axis('off')
boxes = [
    (0.3, 10.65, 14.4, 0.8, 'Measured spectra, precursor mass and adduct', '#e9eef2'),
    (0.3, 8.0, 4.5, 1.55, 'Main pool + generated analogues\nFPNet full1 + learned rankers\n60 candidates; documentation prior', '#e4f0f2'),
    (5.25, 8.0, 4.5, 1.55, 'PubChem search\nFPNet A+B + fold-safe FPNet\n25 proposals; documentation prior', '#edf2e6'),
    (10.2, 8.0, 4.5, 1.55, 'Second engine: BIO + AFIX\nTwo rankers; PubChemLite added\n40 candidates', '#f4ece4'),
    (0.3, 5.5, 9.45, 1.25, 'Within-formula ICEBERG / GLACIER scoring\nMain shortlist + gated PubChem insertion', '#e4f0f2'),
    (0.3, 3.15, 14.4, 1.25, 'Fusion of unique molecular identities\nMain weight 1.0; second-engine weight 0.6; best 40 identities', '#e9eef2'),
    (0.3, 0.75, 14.4, 1.25, 'Final within-formula GLACIER / ICEBERG / fragment scoring\nDistinct identities filled; second-engine leader → rank 2 unless already first', '#e4f0f2'),
]
for x, y, w, h, label, color in boxes:
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=.08',
                               facecolor=color, edgecolor='#527380', linewidth=1.3))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=10.5, color='#193441', linespacing=1.5)
arrows = [
    ((2.55, 10.55), (2.55, 9.68)), ((7.5, 10.55), (7.5, 9.68)),
    ((12.45, 10.55), (12.45, 9.68)),
    ((2.55, 7.9), (2.55, 6.87)), ((7.5, 7.9), (7.5, 6.87)),
    ((5.0, 5.39), (5.0, 4.53)), ((12.45, 7.9), (12.45, 4.53)),
    ((7.5, 3.04), (7.5, 2.13)),
]
for start, end in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops={'arrowstyle': '-|>', 'lw': 1.5, 'color': '#527380'})
ax.set_title('Figure 17. From candidate searches to the final ranked list', fontsize=15, pad=12)
plt.tight_layout()
plt.show()


*Figure 17. Candidate lists and their combination in E006. Three searches propose structures; the learned rankers and the documentation prior set the order within each list. PubChem proposals enter the main list through the library gate, reciprocal-rank fusion combines that list with the second engine, and the final spectral or fragment pass reorders candidates that share a molecular formula. The leading second-engine structure then moves to rank 2 unless it already leads. Deduplication follows the competition's definition of molecular identity.*


### Documentation across candidate populations

The documentation prior draws on an unevenly annotated catalogue. Table 13 and Figure 18 compare three candidate populations: structures with training spectra, COCONUT structures without a training spectrum, and a random sample of the PubChem tier. Linked PubMed articles are more frequent in the training-spectrum and COCONUT populations than in the PubChem sample, which is the enrichment the prior exploits; a structure with no substance record and no article has a zero base prior. The additional natural-product and LOTUS terms can still supply evidence, while rarely studied compounds can remain disadvantaged.


In [ ]:
# Figure 18: how well documented are the structures in each candidate population?
import glob

def _first(patterns):
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True), key=len)
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(patterns[0])

POP_FIG_DIR = _first(['/kaggle/input/**/casmi26-pubchem-popularity-prior/**/pool_lsid.npy',
                      os.environ.get('CASMI_POP_DIR', '.') + '/pool_lsid.npy']).parent
pool_meta_path = _first(['/kaggle/input/**/casmi26-v2-pool/**/pool_meta.parquet', '/kaggle/input/**/v2_pool__pool_meta.parquet', '/kaggle/input/**/v2_pool/pool_meta.parquet',
                         os.environ.get('CASMI_POOL_META', 'pool_meta.parquet')])
pool_src = pd.read_parquet(pool_meta_path, columns=['src']).src.values
pool_pop = (np.load(POP_FIG_DIR / 'pool_lsid.npy').astype(np.float32)
            + np.load(POP_FIG_DIR / 'pool_lpmid.npy').astype(np.float32))
pool_pmid = np.load(POP_FIG_DIR / 'pool_lpmid.npy').astype(np.float32)
tier_lsid = np.load(POP_FIG_DIR / 'pc_lsid.npy', mmap_mode='r')
tier_lpmid = np.load(POP_FIG_DIR / 'pc_lpmid.npy', mmap_mode='r')
_figure18_rng = np.random.default_rng(SEED)
tier_rows = np.sort(_figure18_rng.choice(len(tier_lsid), size=500_000, replace=False))
tier_pop = tier_lsid[tier_rows].astype(np.float32) + tier_lpmid[tier_rows].astype(np.float32)
tier_pmid = tier_lpmid[tier_rows].astype(np.float32)

groups = {
    'Training-spectrum structures': (pool_pop[pool_src != 2], pool_pmid[pool_src != 2]),
    'COCONUT natural products': (pool_pop[pool_src == 2], pool_pmid[pool_src == 2]),
    'PubChem tier (random 500k)': (tier_pop, tier_pmid),
}
pop_table = pd.DataFrame({
    name: {'structures': f'{len(p):,}', 'median pop': f'{np.median(p):.2f}',
           'nonzero documentation counts': f'{(p > 0).mean():.1%}', 'with ≥1 PubMed article': f'{(m > 0).mean():.1%}'}
    for name, (p, m) in groups.items()}).T
display(Markdown('**Table 13. PubChem documentation by candidate population.**'))
display(pop_table)

# One comparison replaces the redundant distribution and coverage panels.
fig, ax = plt.subplots(figsize=(10.5, 4.1))
labels = ['Training-spectrum structures', 'COCONUT without training spectra', 'PubChem random sample']
shares = [(m > 0).mean() * 100 for _, m in groups.values()]
colors = ['#287a8c', '#6a994e', '#b74759']
ax.barh(labels, shares, height=0.55, color=colors)
for y, value in enumerate(shares):
    ax.text(value + 0.5, y, f'{value:.1f}%', va='center', fontsize=11)
ax.set(xlabel='Structures with at least one linked PubMed article (%)',
       xlim=(0, max(5, max(shares) * 1.22)),
       title='Figure 18. Literature documentation differs across candidate sources')
ax.invert_yaxis()
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()
del tier_lsid, tier_lpmid


*Table 13 summarises the stored PubChem documentation counts of each population; Figure 18 shows the share with at least one linked PubMed article: 13.2% of training-spectrum structures, 22.3% of COCONUT structures and 3.5% of the sampled PubChem rows. The COCONUT group excludes structures that also have training spectra; the PubChem group is a random sample of 500,000 tier rows. These are properties of the candidate catalogues, not identification accuracies or measurements of the hidden test.*


## 11. Validation: candidate coverage and ranking

MRR@25 is zero for a molecule whose correct identity is absent from the first 25 guesses, so the analysis below reports candidate coverage, whether the correct identity is available to the search at all, separately from its rank when it is present; the separation distinguishes missing candidates from ranking errors.

An independent estimate also requires that molecular identities be separated across all learned components. Most public pretrained resources do not provide complete identity-level training rosters, although the fold-safe checkpoint documents its exclusion split, so the natural-product benchmark below describes development behaviour and training exposure and does not establish the accuracy of E006 on new molecules. Section 12 checks execution; Section 14 lists the public scores of the main versions.


### Separation of molecular identities

The public [identity-disjoint folds](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-identity-disjoint-validation-folds) group the spectra that share either the stored structure key or the scorer-normalised key, so that every spectrum of one molecule falls into one fold. A second split also keeps each Bemis–Murcko scaffold, the ring systems with the chains that connect them, in one fold; structures without rings have no scaffold and stay grouped by identity only. Figure 19 measures the overlap with the training partition under four splitting schemes.


In [ ]:
# Figure 19: how much of a validation set is already known to the training side under four ways of splitting?
folds_path = _first(['/kaggle/input/**/casmi26-identity-disjoint-validation-folds/**/structures.parquet',
                     os.environ.get('CASMI_FOLDS', 'structures.parquet')])
fold_tab = (pd.read_parquet(folds_path, columns=['inchikey14', 'identity_group', 'scaffold_group', 'fold_identity', 'fold_scaffold'])
            .drop_duplicates('inchikey14').set_index('inchikey14'))
_row = fold_tab.reindex(train.inchikey14.values)
_known = _row.identity_group.notna().values
ident = _row.identity_group.values[_known].astype(np.int64)
scaf = _row.scaffold_group.values[_known].astype(np.int64)
stored = train.inchikey14.values[_known]
split_rng = np.random.default_rng(SEED)
stored_keys = pd.unique(stored)
held_keys = set(split_rng.choice(stored_keys, len(stored_keys) // 5, replace=False))
splits = {
    'random spectra': split_rng.random(len(ident)) < 0.2,
    'stored inchikey14': pd.Series(stored).isin(held_keys).values,
    'scored identity (fold 0)': _row.fold_identity.values[_known] == 0,
    'scaffold group (fold 0)': _row.fold_scaffold.values[_known] == 0,
}
leak_rows = []
for name, val in splits.items():
    val_ident = np.unique(ident[val])
    seen_identity = np.isin(val_ident, np.unique(ident[~val])).mean()
    val_scaf = pd.Series(scaf[val], index=ident[val]).groupby(level=0).first()
    seen_scaffold = np.isin(val_scaf.values, np.unique(scaf[~val])).mean()
    leak_rows.append({'split': name, 'held-out spectra (%)': 100 * val.mean(), 'held-out identity groups': len(val_ident),
                      'same identity in training (%)': 100 * seen_identity, 'same scaffold group in training (%)': 100 * seen_scaffold})
leak = pd.DataFrame(leak_rows).set_index('split')
# The two overlap measures share the same row labels and percentage scale.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
split_labels = ['Random spectra', 'Stored structure key', 'Scored identity', 'Scaffold group']
for ax, column, color, title in (
    (axes[0], 'same identity in training (%)', '#b74759', 'A. Same molecular identity'),
    (axes[1], 'same scaffold group in training (%)', '#287a8c', 'B. Same scaffold group'),
):
    values = leak[column].to_numpy()
    ax.barh(np.arange(len(values)), values, height=0.55, color=color)
    for y, value in enumerate(values):
        ax.text(value + 1.2, y, f'{value:.1f}%', va='center', fontsize=10)
    ax.set(xlim=(0, 115), xticks=[0, 25, 50, 75, 100],
           xlabel='Validation identities represented in training (%)', title=title)
    ax.grid(axis='y', visible=False)
    ax.set_axisbelow(True)
axes[0].set_yticks(np.arange(len(split_labels)), labels=split_labels)
axes[0].invert_yaxis()
fig.suptitle('Figure 19. Overlap depends on the unit used to split the data', fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()
del _row, ident, scaf, stored, splits


*Figure 19. Overlap between validation and training partitions when approximately one fifth of the spectra are held out. With a random split of spectra, 97.7% of the held-out identity groups also have spectra on the training side, so a score on such a split measures library lookup. Splitting by the stored key reduces the overlap to 0.9%, a residue of tautomer variants stored under different keys; the scored-identity folds and the scaffold folds reduce it to zero. Identity folds still share a scaffold group with training for 58.2% of the held-out identities; the scaffold folds remove that overlap as well, although close analogues can still have different scaffolds. These checks concern the records in the split; independent evaluation also requires every learned component to be fitted without the held-out identities.*


### An earlier natural-product benchmark

The public [natural-product panel benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) stores cached candidates and scores for 250 molecules of `enveda-np-examples` in three scenarios: the panel's own spectra removed, all reference spectra of the answer removed, or the answer's pool entry removed. The cached scores allow shortlisting and merging rules to be replayed without rerunning the models. They describe an earlier configuration without the forward models and the second engine.

In the PubChem channel, deepening the fingerprint shortlist from 1,000 to 5,000 rows raised the share of answers inside the shortlist from 90% to 98% while MRR@25 stayed at 0.51; adding the documentation prior raised top-1 accuracy from 40% to 93% and MRR@25 to 0.95 (Table 14). A rule that moves a strongly scored PubChem proposal to rank 1 raised MRR@25 in the pool-removal scenario from 0.519 to 0.718 and changed neither of the other two scenarios (Table 15). That scenario favours the rule: almost all panel answers are documented COCONUT compounds that would normally remain in the pool.

Version 20 combined this promotion rule with stronger forward-model weights, and its public score fell from 0.386 to 0.374 (Figure 20). The simultaneous changes prevent attribution to either component. E006 uses a related gate inside a different complete pipeline, so this replay does not validate that combination.


In [ ]:
# Tables 14–15 and Figure 20: published historical benchmark, not E006 evaluation.
bench_dir = _first(['/kaggle/input/**/casmi26-natural-product-panel-benchmark/**/results_merge_rules.json',
                    os.environ.get('CASMI_BENCH_DIR', '.') + '/results_merge_rules.json']).parent
pc_res = json.loads((bench_dir / 'results_pubchem_channel.json').read_text())
merge_res = json.loads((bench_dir / 'results_merge_rules.json').read_text())
pc_names = {
    'v18: N1=1000 rows': 'FPNet: 1,000 candidates',
    'v4i: N1=5000 rows': 'FPNet: 5,000 candidates',
    'v19: N1=5000 + pop 0.25 + union 200': 'FPNet + documentation prior',
}
by_name = {v['name']: v for v in pc_res['variants']}
table14 = pd.DataFrame([{
    'PubChem scoring': label,
    'Answer in shortlist': f"{by_name[key]['in_shortlist']:.0%}",
    'Top-1': f"{by_name[key]['top1']:.0%}",
    'MRR@25': f"{by_name[key]['mrr25']:.3f}",
} for key, label in pc_names.items()]).set_index('PubChem scoring')
display(Markdown('**Table 14. PubChem shortlisting and ranking in the natural-product benchmark.**'))
display(table14)

scen = {'lib': 'Panel spectra removed', 'nospec': 'No reference spectra', 'dbdrop': 'Pool entry removed'}
rule_names = {
    'v19 gate': 'Fixed insertion positions',
    'promote if S>6 and pop>=5; early slots if S>5': 'Proposal promotion',
}
rules = {r['name']: r for r in merge_res['rules']}
table15 = pd.DataFrame({
    label: {scen[s]: f"{rules[key][s]['mrr25']:.3f}" for s in scen}
    for key, label in rule_names.items()
}).T
display(Markdown('**Table 15. MRR@25 under two merge rules, without forward models or the second engine.**'))
display(table15)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
comparisons = [
    (axes[0], [rules[key]['dbdrop']['mrr25'] for key in rule_names],
     ['Fixed positions', 'Promotion'], 'A. Earlier panel: pool entry removed', '#287a8c'),
    (axes[1], [0.386, 0.374], ['v19', 'v20'],
     'B. Public leaderboard: complete versions', '#b74759'),
]
for ax, values, labels, title, color in comparisons:
    ax.bar(labels, values, color=color, width=0.55)
    for x, value in enumerate(values):
        ax.text(x, value + 0.025, f'{value:.3f}', ha='center', fontsize=11)
    ax.set(ylim=(0, 1), yticks=np.arange(0, 1.01, 0.2), ylabel='MRR@25', title=title)
    ax.grid(axis='x', visible=False)
    ax.set_axisbelow(True)
fig.suptitle('Figure 20. A favourable replay did not establish a public-score improvement', fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()


*Tables 14–15 and Figure 20. Table 14 compares three settings of the PubChem channel: a fingerprint shortlist of 1,000 or 5,000 candidates and the shortlist with the documentation prior. Table 15 compares two merge rules, fixed insertion positions and proposal promotion, in the three scenarios; the two tables come from separate cached replays with slightly different candidate-exclusion rules. Figure 20 sets the apparent benefit of promotion in the pool-removal scenario (A) beside the public score change from version 19 to version 20 (B); the two panels use different evaluation populations and configurations. Threshold selection on the same natural products, possible training exposure and the artificial removal of candidates limit the benchmark's interpretation.*


### Training exposure in fingerprint ranking

Table 16 isolates fingerprint ranking inside a mass window that contains the answer. Each query combines spectra from one structure and polarity. The supplied training code excludes the panel library when fitting FPNet A and B, whereas full1 includes it; complete checkpoint rosters are unavailable. Results on training queries and panel queries consequently reflect both acquisition differences and training exposure. The present pipeline uses full1 in the main engine and adds the fold-safe checkpoint, trained without identity fold 0 and the panel molecules, to A+B in the PubChem channel. Table 16 retains the earlier fingerprint-only comparison; it does not measure this ensemble.


In [ ]:
# Table 16: pool-window retrieval by the fingerprint model alone, on training structures and on the natural-product panel.
fp_res = json.loads((bench_dir / 'results_fingerprint_seen_unseen.json').read_text())
table16 = pd.DataFrame([{
    'Fingerprint checkpoint': m['name'].split(' (')[0], 'panel in training': 'yes' if m['trained_on_panel'] else 'no',
    'training queries: top-1': f"{m['training_sample']['top1']:.0%}", 'training queries: MRR@25': f"{m['training_sample']['mrr25']:.3f}",
    'panel queries: top-1': f"{m['panel']['top1']:.0%}", 'panel queries: top-25': f"{m['panel']['top25']:.1%}", 'panel queries: MRR@25': f"{m['panel']['mrr25']:.3f}",
} for m in fp_res['models']]).set_index('Fingerprint checkpoint')
display(Markdown(f"**Table 16. Ranking by the fingerprint model alone: {fp_res['models'][0]['training_sample']['n']:,} training queries "
                 f"and {fp_res['models'][0]['panel']['n']} natural-product panel queries.**"))
display(table16)
del fp_res, table16


*Table 16. Fingerprint-only ranking on training and natural-product panel queries, as recorded in the public benchmark. FPNet A ranks the answer first for 89% of the training queries and 42% of the panel queries; full1, whose fitting included the panel, reaches 89% on the panel, which agrees with an effect of training exposure. The two query sets also differ in chemistry and window size.*


In [ ]:
# EDA has finished: release scalar tables and RDKit objects before constructing the search index.
import gc
for _name in ("train", "unique_structures", "descriptor_sample", "desc", "fp_sample", "records",
              "preview_pool", "representatives", "mols", "X", "projection", "library_pairs"):
    globals().pop(_name, None)
plt.close("all")
_ = gc.collect()


## 12. Execution of the complete model

The following cells execute E006 with the mounted public datasets and pretrained models and write the final list to `/kaggle/working/submission.csv`. The output checks cover molecule order, valid structures and distinct metric identities. The worked example after the run shows one resulting candidate list.

The scoring pipeline scored 0.455 public MRR@25 (version 55) and uses the refinements described in Section 10.


In [ ]:
'''CASMI26: Metric-Identity Fusion
One vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. The final pass reorders same-formula candidates by GLACIER, ICEBERG fallback or fragment evidence, fills distinct identities and places the second-engine leader at rank 2 unless it already leads. The mounted packages and current scoring configuration are described in Sections 10 and 15. A visible smoke run checks execution, not identification accuracy.'''


In [ ]:
'''Credits & Attribution
@obstacledeveloper — direct source: CASMI26 V1.1 Metric Audit and Safe Tail, version 1, scriptVersionId 356393001. Source: https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail?scriptVersionId=356393001
Scores of upstream notebooks describe those upstream runs; Section 14 records the main notebook's measured versions.
@bobthebot369 — inherited Sovereign Zenith ensemble and confidence-promotion ideas: https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith/notebook
@ahmedberatozer — engine, ranker, pool, PubChem, full FPNet and simulator packages: https://www.kaggle.com/ahmedberatozer
@prvsiyan — analog propagation, spectral matching and fingerprint models: https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline
@dmitriigluzdov — PubChem popularity prior: https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior
@hengck23 — inherited hengck23-dreams-enveda-casmi26 structural candidate resource: https://www.kaggle.com/hengck23
Official metric implementation, version 14: https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank
ICEBERG/GLACIER research tooling: https://github.com/coleygroup/ms-pred
Competition: https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra
RDKit official metric wheel: https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel
ChEBI/LIPID MAPS and COCONUT resources remain credited in inherited code and input metadata.
The public notebook declares Apache-2.0; upstream datasets/components retain their own licences. Substantial code, including embedded modules, is reused verbatim. The adaptation combines the public engines with the priors, fingerprint ensemble and final ordering described in Section 10; authorship of the engines remains with their authors. Upstream comments are retained as provenance, not independently established component-performance claims.'''


In [ ]:
'''Pipeline provenance
The public E006 source and its inherited engine, fusion and simulator code are reused with the refinements described in Section 10. Metric identity follows the scorer's tautomer standardisation and heavy-atom composition helpers. Final reordering is confined to candidates of the same molecular formula. Public leaderboard feedback does not establish performance on new molecular identities.'''


In [ ]:
'''# Inherited CASMI26 V1.1 — metric audit and safe tail

The following description belongs to the upstream V1.1 audit; the current method is described in Section 10.
This is a conservative candidate-coverage experiment, not a claimed leaderboard improvement.

Changes: pin official RDKit 2026.03.3; preserve the V1 inference/ranking pipeline;
deduplicate only after ranking using the official metric identity; append ranked overflow
and mass-compatible DreaMS candidates into vacant slots; assert every original matchable
identity has equal or better rank. The old DreaMS branch was bypassed by `NameError: Path`.

No hidden-test labels, manual labels, or network access are used. The visible test overlaps
training data, so its score is an operational check and is **not** a validation estimate.

Source attribution: copied V1 ensemble from BobtheBot369 / Ahmed Berat Özer / prvsiyan
and the other authors credited in its code. Scoring reference: Kaggle Competition Metrics,
`https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank`, version 14, Apache-2.0.
'''


In [ ]:
'''## 1. Setup

Locates every dataset, installs the competition's RDKit wheel, sets the popularity-prior parameters for the PubChem channel (`POP_LAM = 0.15`, `POP_UNION = 200`) and checks the v4b model files against their manifest.
'''


In [ ]:
# Dataset folders (the data notebook defines these in its data sections)
def dataset_dir(slug, marker):
    hits = sorted(glob.glob(f'/kaggle/input/**/{slug}/{marker}', recursive=True), key=len)
    return Path(hits[0]).parent if hits else None
POP = dataset_dir('casmi26-pubchem-popularity-prior', 'pc_lsid.npy')
FOLD = dataset_dir('casmi26-fold-safe-fpnet', 'MODEL_INPUTS.json')
assert POP is not None and FOLD is not None, 'popularity prior or fold-safe FPNet dataset not attached'
'''CASMI26 v4n = v4i + ICE_LAM = GL_LAM = 1.0 (v4l, LB 0.380) + engine FP bank fpnet_full1 (v4m, LB 0.378)'''
'''CASMI26 v4i = v4h + PubChem-only channel pass-1 depth N1 1000 -> 5000 (recall; gate/slots unchanged)'''
'''CASMI26 v4h = v4g + GLACIER third re-scoring term: z(ranker) + 0.5 z(ICEBERG) + 0.5 z(GLACIER) (casmi26-glacier)'''
'''casmi26 v28 = v17 + pool popularity re-rank of the base list (mu 0.15, mode A); v17 = Ahmed v4g + popularity prior in the PubChem list (see make_v17.py)'''
'''CASMI26 v4f = v4b (FPNet A engine bank, LB 0.354) + ICEBERG post-ranker isomer re-scoring = the v3.6 model (v1 engine + FPNet A + moderate generation + gated PubChem channel, LB 0.358)'''
'''+ fe_v4 feature families computed on the SAME v1 engine run (derivation evidence / class-3 priors, fragmentation 2.0,'''
'''analog-structure relations, DreamsFP views) + a ranker retrained on the v1-engine simulation sim1x with them.'''
'''Built by work/analysis/fe_v4/build_v4v1.py -- edit there.'''
import os, sys, glob, re, subprocess, time, json, hashlib, pickle
from pathlib import Path
T0 = time.time()
LIB_TAU, REL_TH = 0.9, 600.0
TOPN, ICE_LAM, ICE_BUDGET, ICE_PC = 60, 1.0, 5400, True
SLOTS_AGG, SLOTS_GENTLE = [2, 4, 6, 8, 10], [4, 8, 12, 16, 20]
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]
def sha256(path, n=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()
FOLD0_PATH = str(FOLD / 'fpnet_fold0.pt')
assert sha256(FOLD0_PATH) == 'ddf6eaaa8e541a16e3f0357ed30f0346e529054f75d5619aedb6024dcb1bd7f0', 'Untested fold-safe checkpoint'
os.environ['CASMI_FPNET_FOLD0'] = FOLD0_PATH
print('PubChem fold-safe model:', FOLD0_PATH, flush=True)
POP_LAM, POP_UNION = 0.15, 200            # v17: popularity prior for the PubChem-only list
os.environ['CASMI_POP_DIR'] = str(POP); os.environ['CASMI_POP_LAM'] = str(POP_LAM); os.environ['CASMI_POP_UNION'] = str(POP_UNION)
print('v17 popularity prior', POP_LAM, POP_UNION, os.environ['CASMI_POP_DIR'])
SRC_BONUS, SRC_BITS = 0.5, 1   # documentation prior + bonus for depositor-source groups (pc_src_mask.npy, dataset v5)
os.environ['CASMI_SRC_BONUS'] = str(SRC_BONUS); os.environ['CASMI_SRC_BITS'] = str(SRC_BITS)
assert os.path.exists(os.path.join(os.environ['CASMI_POP_DIR'], 'pc_src_mask.npy')), 'pc_src_mask.npy missing'
print('depositor-source bonus', SRC_BONUS, 'bits', SRC_BITS, flush=True)
NP_BONUS = 1.0   # natural-product bonus: COCONUT, LOTUS or NPAtlas membership (pc_np.npy, popularity prior)
os.environ['CASMI_NP_BONUS'] = str(NP_BONUS)
assert os.path.exists(os.path.join(os.environ['CASMI_POP_DIR'], 'pc_np.npy')), 'pc_np.npy missing'
print('natural-product bonus', NP_BONUS, flush=True)
LOTUS_W = 0.5   # literature prior: ln(1 + LOTUS references) from the natural-product knowledge table
os.environ['CASMI_LOTUS_W'] = str(LOTUS_W)
assert (POP / 'aligned_public_priors.py').is_file(), 'aligned_public_priors.py missing'
assert sha256(POP / 'aligned_public_priors.py') == '89873c7312c16be2180d44bb9eb3c4e9ce19c821e74ab92bb41666ea0c0f9248', 'Untested aligned prior helper'
assert (POP / 'pc_lotus_refs.npy').is_file(), 'pc_lotus_refs.npy missing'
print('LOTUS literature prior', LOTUS_W, flush=True)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w and Path(w).name.startswith('rdkit-2026.3.3-')]
assert whl, 'Official RDKit 2026.3.3 wheel is missing'
print('rdkit wheel:', whl)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
r.check_returncode()
import rdkit; print('rdkit', rdkit.__version__)
assert rdkit.__version__ == '2026.03.3'
V4CODE = os.path.dirname(os.path.dirname(find('casmi26-v4b-models/**/casmi/engine.py')))
V4MOD = os.path.dirname(find('casmi26-v4b-models/**/MANIFEST.json'))
V3CODE = os.path.dirname(os.path.dirname(find('casmi26-v3-models/**/casmi/engine.py')))
V3MOD = os.path.dirname(find('casmi26-v3-models/**/ranker_0.pkl'))
POOL_DIR = os.path.dirname(find('pool_meta.parquet'))
PC_DIR = os.path.dirname(find('pc_smiles.npy'))
COMP = os.path.dirname(find('test.parquet'))
MAN = json.load(open(os.path.join(V4MOD, 'MANIFEST.json')))
bad = []
for rel, h in sorted(MAN['files'].items()):
    p = os.path.join(V4CODE, rel[len('code/'):]) if rel.startswith('code/') else os.path.join(V4MOD, rel)
    if not os.path.exists(p) or sha256(p) != h:
        bad.append(rel)
assert not bad, f'dataset files differ from MANIFEST.json: {bad}'
print('V4CODE', V4CODE, '\nV4MOD', V4MOD, '\nV3CODE', V3CODE, '\nPOOL', POOL_DIR, '\nPC', PC_DIR, '\nCOMP', COMP)
print(f"{len(MAN['files'])} files match MANIFEST.json; families {MAN['families']}", f'{time.time()-T0:.0f}s')

'''Compatibility repair: select the simulator RDKit 2025.3.6 wheel by interpreter and platform tags, including bundled .whl.ice files. Fail before inference when no compatible build exists; preserve the main RDKit 2026.03.3 environment.'''

def simulator_rdkit_wheel(pkg):
    from pip._vendor.packaging.tags import sys_tags
    from pip._vendor.packaging.utils import parse_wheel_filename
    compatible = set(sys_tags())
    bundled = sorted(glob.glob(os.path.join(pkg, 'wheels', 'rdkit-2025.3.6-*.whl*')))
    external = sorted(glob.glob('/kaggle/input/**/rdkit-2025.3.6-*.whl', recursive=True))
    for path in bundled + external:
        name = os.path.basename(path).removesuffix('.ice')
        _, version, _, tags = parse_wheel_filename(name)
        if str(version) == '2025.3.6' and compatible.intersection(tags):
            return path
    raise FileNotFoundError(
        f'RDKit 2025.3.6 wheel compatible with Python {sys.version_info.major}.{sys.version_info.minor} '
        f'and this platform is required for ICEBERG/GLACIER; candidates: {bundled + external}')

_simulator_pkg = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))
print('Simulator RDKit wheel:', simulator_rdkit_wheel(_simulator_pkg))


In [ ]:
# v42 determinism policy, adapted from its restored tree to E006's original mounts.
# E006 cell6 has already verified the original v4b manifest and all its hashes.
def _v43_patch_engine(source_root, work_root):
    import hashlib
    from pathlib import Path
    source_root = Path(source_root).resolve(strict=True)
    work_root = Path(work_root)
    original = source_root / 'casmi' / 'engine.py'
    raw = original.read_bytes()
    old = 'ad = max(set(adducts), key=adducts.count)'
    new = 'ad = max(adducts, key=adducts.count)'
    text = raw.decode('utf-8')
    assert text.count(old) == 1, 'Expected exactly one source main-engine adduct tie'
    patched = text.replace(old, new, 1).encode('utf-8')
    assert source_root != work_root.resolve() and source_root not in work_root.resolve().parents
    work_root.mkdir(parents=True, exist_ok=True)
    # Preserve the original relative source tree. Only engine.py is copied/changed.
    for source in sorted(source_root.rglob('*')):
        if '__pycache__' in source.parts or source.suffix == '.pyc':
            continue
        dest = work_root / source.relative_to(source_root)
        if source.is_dir():
            dest.mkdir(parents=True, exist_ok=True)
        elif source == original:
            assert not dest.is_symlink(), 'Patched engine must be a work-tree file'
            if dest.exists():
                assert dest.read_bytes() == patched, 'Existing work-tree patch differs'
            else:
                dest.write_bytes(patched)
        else:
            if dest.is_symlink():
                assert dest.resolve(strict=True) == source.resolve(strict=True)
            else:
                assert not dest.exists(), 'Existing work-tree source differs'
                dest.symlink_to(source)
    assert original.read_bytes() == raw, 'Mounted source changed'
    assert (work_root / 'casmi/engine.py').read_bytes() == patched
    return dict(applied=True, original_source_root=str(source_root), work_source_root=str(work_root),
        original_sha256=hashlib.sha256(raw).hexdigest(), patched_sha256=hashlib.sha256(patched).hexdigest(),
        original_line=old, patched_line=new, subprocess_pythonhashseed='0',
        torch_seed=0, cuda_seed=0, cudnn_deterministic=True, cudnn_benchmark=False)

V43_DETERMINISM = _v43_patch_engine(V4CODE, '/kaggle/working/v43_v4code')
V4CODE = V43_DETERMINISM['work_source_root']
os.environ['PYTHONHASHSEED'] = '0'


In [ ]:
"""Read-only E004 observer, before the source smoke switch."""


def _e004_preflight(scope):
    import json
    import os
    import platform
    import subprocess
    import pandas as pd
    import rdkit

    slug = 'enveda-CASMI26-molecule-id-mass-spectra'
    comp = os.path.realpath(scope['COMP'])
    assert comp in {f'/kaggle/input/{slug}', f'/kaggle/input/competitions/{slug}'}
    assert platform.python_version_tuple()[:2] == ('3', '12')
    assert rdkit.__version__ == '2026.03.3'
    # Do not initialize a parent CUDA context before the source's Engine2 subprocess.
    gpu = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
    assert 'T4' in gpu, gpu
    ids = sorted(set(pd.read_parquet(os.path.join(comp, 'test.parquet'),
                                    columns=['molecule_id']).molecule_id.astype(str)))
    sample = pd.read_csv(os.path.join(comp, 'sample_submission.csv'), dtype=str)
    assert ids and set(sample.molecule_id) == set(ids)
    result = dict(runtime_input_path=comp, input_ids=ids, python=platform.python_version(),
                  rdkit=rdkit.__version__, gpu=gpu)
    print('E004_PREFLIGHT', json.dumps({k: v for k, v in result.items() if k != 'input_ids'}))
    return result


E004_PREFLIGHT = _e004_preflight(globals())
del _e004_preflight


In [ ]:
'''## 2. Commit switch

When the visible placeholder test is detected, the notebook runs only 12 molecules with a short ICEBERG budget, so saving a version is quick. The scoring rerun on the hidden test runs everything. Nothing else depends on this switch.
'''


In [ ]:
'''SMOKE switch: commits (dummy test) run 12 molecules; the scoring rerun (hidden test) runs everything.'''
import pandas as pd, hashlib
_te = pd.read_parquet(os.path.join(COMP, 'test.parquet'), columns=['molecule_id', 'spectrum_id'])
_sig = hashlib.md5(','.join(sorted(_te.spectrum_id.astype(str))).encode()).hexdigest()
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or _sig != '654e030e4173780e945252b4a5adf70b'
SMOKE_N = 12
if not IS_RERUN:
    SM = '/kaggle/working/smoke_comp'; os.makedirs(SM, exist_ok=True)
    _full = pd.read_parquet(os.path.join(COMP, 'test.parquet'))
    _keep = sorted(_full.molecule_id.unique())[:SMOKE_N]
    _full[_full.molecule_id.isin(_keep)].to_parquet(os.path.join(SM, 'test.parquet'))
    for _f in ['train.parquet', 'sample_submission.csv']:
        if not os.path.exists(os.path.join(SM, _f)):
            os.symlink(os.path.join(COMP, _f), os.path.join(SM, _f))
    COMP = SM
    ICE_BUDGET = 300
print('RERUN', IS_RERUN, '| sig', _sig, '| COMP', COMP, '| ICE_BUDGET', ICE_BUDGET)


In [ ]:
'''## 3. Engine 2: two-ranker engine (BIO + AFIX)

This engine runs first in its own process, while GPU memory is still empty. It writes a top-40 list per molecule with tautomer-canonical InChIKey-14 keys.

- **Candidates** come from COCONUT, standardised ChEBI/LIPID MAPS (BIO), PubChemLite and the training structures.
- **Evidence** comes from library matching (including adduct-shifted matches), analog propagation, prvsiyan's fingerprint models and MetFrag-lite fragment matching.
- **Ranking** blends two gradient-boosted rankers with depth-6 trees, averaging eight and four fitted models.
'''


In [ ]:
'''Our engine (two-ranker + BIO + AFIX) in a subprocess, then weighted reciprocal-rank fusion with the v4g lists.'''
ENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)
ENG_FILES = {'pv.py': '"""prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\nAdditions are marked `# ADDED`.\n"""\nimport math\nimport numpy as np\nfrom numba import njit, prange\n\n\nclass CFG:\n    PPM_WIN = 10.0\n    PPM_FALLBACK = 30.0\n    INT_FLOOR = 0.002\n    MAX_PEAKS = 256\n    MZ_TOL = 0.01\n    INT_POWER = 1.0\n    ENT_WEIGHT = True\n    ANALOG_WIN = 200.0\n    N_ANALOG = 80\n    SIM_POWER = 3.0\n    W1_PRIORS = (0.30, 0.60)\n    SEEDS = (0, 1, 2, 3, 4, 5, 6, 7)  # 8-seed bag (ghazarosbarseghyan91 v14 seeds)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n    TOPN = 25\n\n\n# ----------------------------------------------------------------------------------------------\n# similarity kernels\n# ----------------------------------------------------------------------------------------------\n@njit(cache=True, fastmath=True)\ndef _clean(mz, it, floor, topk, power, ent_weight):\n    n = len(mz)\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx: mx = it[i]\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx; c = 0\n    for i in range(n):\n        if it[i] >= thr: c += 1\n    idx = np.empty(c, np.int64); j = 0\n    for i in range(n):\n        if it[i] >= thr: idx[j] = i; j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c): v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk): k2[i] = idx[o[i]]\n        k2.sort(); idx = k2; c = topk\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\n    if s > 0:\n        for i in range(c): oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S; s2 = 0.0\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\n            if s2 > 0:\n                for i in range(c): oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0; tot = 0.0\n    buf = np.empty(n + m, np.float64); b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\n    for x in range(b): tot += buf[x]\n    if tot <= 0: return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0: SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\n    if shift > -0.001 and shift < 0.001: return a\n    sm = np.empty(len(cmz), np.float32)\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\n    b = entropy_sim(qmz, qp, sm, cp, tol)\n    return a if a > b else b\n\n\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\n    out = np.zeros(hi - lo, np.float32)\n    for k in prange(hi - lo):\n        a = roff[lo + k]; b = roff[lo + k + 1]\n        if b <= a: continue\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\n    return out\n\n\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\n    return out\n\n\n# ADDED: clean a list of spectra into a flat store\n@njit(cache=True)\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\n    n = len(rows)\n    lens = np.zeros(n + 1, np.int64)\n    tmp_m = []\n    tmp_i = []\n    for k in range(n):\n        r = rows[k]\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\n        tmp_m.append(cm); tmp_i.append(cp)\n        lens[k + 1] = lens[k] + len(cm)\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\n    for k in range(n):\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\n    return lens, om, oi\n\n\n# ----------------------------------------------------------------------------------------------\n# adducts\n# ----------------------------------------------------------------------------------------------\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nE = 0.00054857990; PROTON = MASS[\'H\'] - E; H2O = 2 * MASS[\'H\'] + MASS[\'O\']\nNH4 = MASS[\'N\'] + 4 * MASS[\'H\']; FORMATE = MASS[\'C\'] + 2 * MASS[\'H\'] + 2 * MASS[\'O\']\nACETATE = 2 * MASS[\'C\'] + 4 * MASS[\'H\'] + 2 * MASS[\'O\']\nADDUCTS = {\n    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS[\'Na\'] - E),\n    "[M+K]+": (1, 1, MASS[\'K\'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),\n    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),\n    "[M+CH3OH+H]+": (1, 1, PROTON + MASS[\'C\'] + 4 * MASS[\'H\'] + MASS[\'O\']),\n    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS[\'C\'] + 3 * MASS[\'H\'] + MASS[\'N\']),\n    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),\n    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS[\'Cl\'] + E), "[M]-": (1, 1, E),\n    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS[\'Na\'] - E), "[2M+NH4]+": (2, 1, NH4 - E),\n    "[2M+K]+": (2, 1, MASS[\'K\'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),\n    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = (ad == a)\n        if m.any(): out[m] = (mz[m] * z - d) / n\n    return out\n\n\n# ----------------------------------------------------------------------------------------------\n# ranking features (31)\n# ----------------------------------------------------------------------------------------------\nN_ANALOG = 80\nP_SIM = 3.0\nN_FEAT = 31\n\n\ndef _rank_norm(x):\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\n\n\ndef _z(x):\n    s = x.std()\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\n\n\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\n    nc = cand_fp.shape[0]\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\n    lv = np.asarray(cand_lib, np.float32)\n    lvmax = float(lv.max()) if nc else 0.0\n    if analog_fp is not None and len(analog_sim):\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\n        inter = cf @ af.T\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\n        a1 = (tan * w[None, :]).max(1)\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\n    else:\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\n    apmax = float(ap.max()) if nc else 0.0\n    if model_logits is not None:\n        raw = cf @ np.asarray(model_logits, np.float32)\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\n                 (raw == raw.max()).astype(np.float32)]\n    else:\n        mfeat = [np.zeros(nc, np.float32)] * 6\n    if model_logits is not None and nc:\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\n    else:\n        xfeat = [np.zeros(nc, np.float32)] * 6\n    if frag is not None:\n        fr = np.asarray(frag, np.float32)\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\n    else:\n        ffeat = [np.zeros(nc, np.float32)] * 4\n    return np.column_stack([\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\n        np.full(nc, np.log(max(nc, 1))),\n        *mfeat, *ffeat, *xfeat,\n    ]).astype(np.float32)\n\n\n# ----------------------------------------------------------------------------------------------\n# MetFrag-lite\n# ----------------------------------------------------------------------------------------------\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\n    if (w == 0).any(): return None\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\n    return w, bonds, n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop: continue\n        adj[a].append(b); adj[b].append(a)\n    seen = np.zeros(n, bool); comps = []\n    for s in range(n):\n        if seen[s]: continue\n        stack = [s]; seen[s] = True; cur = [s]\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\n    g = mol_graph(smi)\n    if g is None: return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef frag_masses_safe(smi):\n    try: return fragment_masses(smi)\n    except Exception: return np.zeros(0)\n\n\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\nCARRIERS = {\n    "[M+H]+": (PROTON,), "[M-H2O+H]+": (PROTON,), "[M-2H2O+H]+": (PROTON,),\n    "[M+NH4]+": (PROTON, NH4 - E), "[M+Na]+": (PROTON, MASS[\'Na\'] - E), "[M+K]+": (PROTON, MASS[\'K\'] - E),\n    "[M-H]-": (-PROTON,), "[M-H2O-H]-": (-PROTON,), "[M+CH2O2-H]-": (-PROTON, FORMATE - PROTON),\n    "[M+Cl]-": (-PROTON, MASS[\'Cl\'] + E),\n}\n\n\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    car = CARRIERS.get(adduct, (PROTON,) if "+" in adduct[-2:] else (-PROTON,))\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    ion = []\n    for dh in h_shifts:\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\n    ion = np.sort(np.concatenate(ion))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)', 'pv_fp.py': '"""Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch."""\nimport math\nimport numpy as np\n# ----------------------------------------------------------------------------------------------\n# spectrum -> fingerprint model\n# ----------------------------------------------------------------------------------------------\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nFP_MAX_PEAKS = 128\nADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",\n               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",\n               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",\n               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\nINSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]\n\n\ndef instr_family(s):\n    if s is None: return 4\n    t = str(s).lower()\n    if \'timstof\' in t: return 0\n    if \'orbitrap\' in t or \'qft\' in t or \'ftms\' in t or \'hybrid ft\' in t or \'itft\' in t or \'exactive\' in t: return 1\n    if \'tof\' in t: return 2\n    if \'trap\' in t or \'qq\' in t: return 3\n    return 4\n\n\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = (mz <= prec_mz + 1.5)\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    mx = it.max()\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = it >= floor * mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        order = np.argsort(-it)\n        bucket = (mz // win).astype(np.int64)\n        cnt = {}; sel = []\n        for i in order:\n            b = bucket[i]; c = cnt.get(b, 0)\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\n            need = max_peaks - len(sel)\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    v = np.sqrt(it / it.max())\n    return mz.astype(np.float32), v.astype(np.float32)\n\n\nclass SinEmb(nn.Module):\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\n        super().__init__()\n        n = dim // 2\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\n        self.register_buffer(\'inv\', (2 * math.pi) / wav)\n\n    def forward(self, x):\n        a = x.unsqueeze(-1) * self.inv\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__(); self.h = h\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\n        self.n2 = nn.LayerNorm(d)\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x, pad):\n        B, N, D = x.shape; y = self.n1(x)\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n        m = (~pad)[:, None, None, :]\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\n        return x + self.drop(self.ff(self.n2(x)))\n\n\nclass FPNet(nn.Module):\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\n        super().__init__()\n        self.d = d\n        self.mz_emb = SinEmb(d)\n        self.nl_emb = SinEmb(d)\n        self.pk = nn.Linear(2 * d + 1, d)\n        self.prec_emb = SinEmb(d)\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\n        self.gl = nn.Linear(d + 3, d)\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n        B, N = mz.shape\n        nl = (prec[:, None] - mz).clamp(min=0)\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n        g = self.gl(torch.cat([self.prec_emb(prec),\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\n        x = torch.cat([g.unsqueeze(1), p], 1)\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n        for b in self.blocks: x = b(x, pad)\n        x = self.norm(x)\n        cls = x[:, 0]\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\n        return self.head(torch.cat([cls, mean], -1))\n\n\ndef load_fp_models(paths, dev):\n    single, merged = [], []\n    nbits = None\n    for pth in paths:\n        ck = torch.load(pth, map_location=\'cpu\', weights_only=False)\n        net = FPNet(ck[\'nbits\'], d=ck[\'d\'], layers=ck[\'layers\']).to(dev).eval()\n        net.load_state_dict(ck[\'model\'])\n        (merged if \'merged\' in str(pth).replace(\'\\\\\', \'/\').split(\'/\')[-1] else single).append(net)\n        nbits = ck[\'nbits\']\n    return single, merged, nbits\n\n\ndef merge_peaks(specs):\n    """specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum."""\n    specs = [(a, b) for a, b in specs if len(a)]\n    if not specs: return np.zeros(0), np.zeros(0)\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    keep = np.ones(len(mz), bool)\n    for j in range(1, len(mz)):\n        if mz[j] - mz[j - 1] < 0.005:\n            if it[j] >= it[j - 1]: keep[j - 1] = False\n            else: keep[j] = False\n    return mz[keep], it[keep]\n\n\n@torch.no_grad()\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\n    """P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode."""\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\n    if not keep: return None\n    P = [P[i] for i in keep]\n    B = len(P); N = max(len(a) for a, _ in P)\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\n    for i, (a, b) in enumerate(P):\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\n    T = lambda x: torch.as_tensor(x, device=dev)\n    args = (T(mz), T(it), T(pad),\n            T(np.asarray(prec, np.float32)[keep]),\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX[\'<unk>\']) for a in np.asarray(adduct, object)[keep]])),\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\n            T(np.asarray(ce, np.float32)[keep]),\n            T(np.asarray(mode, np.float32)[keep]))\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\n\n\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\n    """prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum."""\n    single, merged, dev = models\n    out = []\n    if single:\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\n        if za is not None: out.append(za)\n    if merged:\n        mz, it = merge_peaks(specs)\n        pm = float(np.median(prec))\n        P = [prep_peaks(mz, it, pm)]\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\n        if zb is not None: out.append(zb)\n    return np.mean(out, axis=0) if out else None', 'casmi_engine.py': '"""End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\n\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\n"""\nimport os, sys, glob, time, pickle, math\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\nimport pv\nfrom pv import CFG\nimport re as _re\n_MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100, P=30.97376163,\n             Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473, Si=27.9769265325, Se=79.9165213,\n             Na=22.9897692809, K=38.96370668, B=11.0093054, As=74.9215965, Fe=55.9349375, D=2.0141017778)\n_TOK = _re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    """ADDED: neutral monoisotopic mass by arithmetic on molecular_formula (prvsiyan fix; immune to\n    riken\'s 0.005 Da precursor precision and gnps adduct mislabels). NaN if an element is untabulated."""\n    if not isinstance(f, str) or not f: return np.nan\n    m = 0.0\n    for el, n in _TOK.findall(f):\n        if el not in _MONO: return np.nan\n        m += _MONO[el] * (int(n) if n else 1)\n    return m\n\nPOOL = None\n\n\nclass RANK:\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\n    SEEDS = (0, 1, 2, 3, 4, 5)  # 6-seed bag (ghazarosbarseghyan91 v14 seeds)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n\nT0 = time.time()\nLOCAL = os.environ.get("CASMI_LOCAL") == "1"\nROOTS = [r"C:\\Users\\HW-LEE\\Desktop\\CASMI"] if LOCAL else ["/kaggle/input"]\n\n\ndef find(name):\n    for root in ROOTS:\n        hits = glob.glob(os.path.join(root, "**", name), recursive=True)\n        if hits: return sorted(hits, key=len)[0]\n    raise FileNotFoundError(name)\n\n\ndef log(msg):\n    print(f"[{time.time()-T0:6.0f}s] {msg}", flush=True)\n\n\n# ---------------------------------------------------------------------------------------------\n# library\n# ---------------------------------------------------------------------------------------------\ndef load_library(path):\n    """Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB."""\n    f = pq.ParquetFile(path)\n    n_rows = f.metadata.num_rows\n    npk = 0\n    for i in range(f.num_row_groups):\n        c = f.read_row_group(i, columns=["ms2_mzs"]).column(0).combine_chunks()\n        npk += len(c.values)\n    off = np.zeros(n_rows + 1, np.int64)\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\n    prec, add, pol, ik, smi = [], [], [], [], []\n    fo, tims = [], []\n    r0 = p0 = 0\n    for i in range(f.num_row_groups):\n        t = f.read_row_group(i, columns=["inchikey14", "normalized_smiles", "adduct", "precursor_mz", "ionization_mode",\n                                         "ms2_mzs", "ms2_normalized_intensities", "molecular_formula", "instrument_type"])\n        mzc = t.column("ms2_mzs").combine_chunks(); itc = t.column("ms2_normalized_intensities").combine_chunks()\n        o = mzc.offsets.to_numpy().astype(np.int64)\n        n = len(o) - 1; k = int(o[-1] - o[0])\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\n        prec.append(t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64))\n        add += t.column("adduct").cast(pa.string()).to_pylist()\n        pol.append(np.array(t.column("ionization_mode").cast(pa.string()).to_pylist(), dtype=object) == "positive")\n        ik += t.column("inchikey14").cast(pa.string()).to_pylist()\n        smi += t.column("normalized_smiles").cast(pa.string()).to_pylist()\n        fo += t.column("molecular_formula").cast(pa.string()).to_pylist()\n        tims.append(np.array([isinstance(x, str) and x.lower().replace("-", "").replace(" ", "") == "timstof"\n                              for x in t.column("instrument_type").cast(pa.string()).to_pylist()]))\n        r0 += n; p0 += k\n        del t, mzc, itc\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\n    nmp = pv.neutral_mass(prec, add)\n    _fm = {x: formula_mass(x) for x in set(v for v in fo if isinstance(v, str))}\n    nmf = np.array([_fm.get(x, np.nan) if isinstance(x, str) else np.nan for x in fo])\n    nm = np.where(np.isfinite(nmf), nmf, nmp)\n    tims = np.concatenate(tims)\n    log(f"library mass index: formula {np.isfinite(nmf).mean():.1%}, timsTOF rows {tims.mean():.1%}")\n    del fo\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol, tims=tims,\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\n\n\n# ---------------------------------------------------------------------------------------------\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\n# ---------------------------------------------------------------------------------------------\n_g = {}\n\n\ndef _fp_init():\n    from rdkit import RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog("rdApp.*")\n    _g["bits"] = np.load(find("fp_bits.npy"))\n    _g["m2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n    _g["m3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\n    _g["rk"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\n\n\ndef fp_and_mass(smi):\n    from rdkit import Chem\n    from rdkit.Chem import MACCSkeys\n    from rdkit.Chem.Descriptors import ExactMolWt\n    if not _g: _fp_init()\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    try:\n        fp = np.concatenate([_g["m2"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["m3"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["rk"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g["bits"]]\n        return np.packbits(fp), float(ExactMolWt(m))\n    except Exception:\n        return None\n\n\ndef canon_key(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog("rdApp.*")\n    if "te" not in _g: _g["te"] = rdMolStandardize.TautomerEnumerator()\n    try:\n        m = Chem.MolFromSmiles(smi)\n        return Chem.MolToInchiKey(_g["te"].Canonicalize(m))[:14] if m is not None else None\n    except Exception:\n        return None\n\n\ndef build_pool(L, workers):\n    from multiprocessing import Pool as MPool\n    d = os.path.dirname(find("coco_fp.npy"))\n    cm = pickle.load(open(os.path.join(d, "coco_meta.pkl"), "rb"))\n    co_fp = np.load(os.path.join(d, "coco_fp.npy")); co_mass = np.load(os.path.join(d, "coco_mass.npy"))\n    co_keys = np.asarray(cm["keys"], dtype=object); co_smi = np.asarray(cm["smiles"], dtype=object)\n    # ADDED: ChEBI + LIPID MAPS (prvsiyan/chebi-lipidmaps-casmi26), same 6,930-bit packed fingerprints\n    bd = os.path.dirname(find(os.path.join("casmi-chebi-lmsd-structures", "bio_fp.npy"))); bm = pickle.load(open(os.path.join(bd, "bio_meta.pkl"), "rb"))\n    bkeys = np.asarray(bm["keys"], dtype=object); bnew = ~pd.Series(bkeys).isin(set(co_keys)).values\n    co_fp = np.vstack([co_fp, np.load(os.path.join(bd, "bio_fp.npy"))[bnew]])\n    co_mass = np.concatenate([co_mass, np.load(os.path.join(bd, "bio_mass.npy"))[bnew]])\n    co_keys = np.concatenate([co_keys, bkeys[bnew]])\n    co_smi = np.concatenate([co_smi, np.asarray(bm["smiles"], dtype=object)[bnew]])\n    log(f"+ ChEBI/LIPID MAPS {int(bnew.sum()):,} new structures")\n    # PubChemLite CC0 candidates (idea: guhongbin/casmi26-v447-pubchemlite; block as in ahmedberatozer/casmi26-v14-pubchemlite;\n    # data: franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints, copied into the attributed v4g assets).\n    # New arrays are built first and swapped in only if every step succeeds; any failure keeps the previous pool.\n    try:\n        pcd = os.path.dirname(find(os.path.join("casmi26-pubchemlite-cc0-candidate-fingerprints", "pubchemlite_fp.npy")))\n        pcm = pickle.load(open(os.path.join(pcd, "pubchemlite_meta.pkl"), "rb"))\n        pckeys = np.asarray(pcm["keys"], dtype=object)\n        pcnew = ~pd.Series(pckeys).isin(set(co_keys)).values\n        _fp = np.vstack([co_fp, np.load(os.path.join(pcd, "pubchemlite_fp.npy"))[pcnew]])\n        _ms = np.concatenate([co_mass, np.load(os.path.join(pcd, "pubchemlite_mass.npy"))[pcnew]])\n        _ks = np.concatenate([co_keys, pckeys[pcnew]])\n        _sm = np.concatenate([co_smi, np.asarray(pcm["smiles"], dtype=object)[pcnew]])\n        assert len(_fp) == len(_ms) == len(_ks) == len(_sm), (len(_fp), len(_ms), len(_ks), len(_sm))\n        co_fp, co_mass, co_keys, co_smi = _fp, _ms, _ks, _sm\n        log(f"+ PubChemLite CC0 {int(pcnew.sum()):,} new structures")\n    except Exception as _pe:\n        log(f"PubChemLite CC0 skipped: {_pe!r}")\n    tr = pd.DataFrame({"ik": L["ik"], "smi": L["smi"]}).dropna().drop_duplicates("ik")\n    tr = tr[~tr.ik.isin(set(co_keys))]\n    log(f"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}")\n    with MPool(workers) as mp:\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\n    ok = np.array([r is not None for r in res])\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\n    good = np.isfinite(mass)\n    o = np.argsort(np.where(good, mass, 1e18), kind="mergesort")[: good.sum()]\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\n    P["k2i"] = pd.Series(np.arange(len(o)), index=P["keys"]); P["k2i"] = P["k2i"][~P["k2i"].index.duplicated()]\n    log(f"pool {len(o):,} structures")\n    return P\n\n\ndef pool_fps(idx):\n    return np.unpackbits(np.asarray(POOL["fp"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\n\n\ndef pool_window(t, ppm):\n    a = np.searchsorted(POOL["mass"], t * (1 - ppm / 1e6), "left")\n    b = np.searchsorted(POOL["mass"], t * (1 + ppm / 1e6), "right")\n    return np.arange(a, b)\n\n\n# ---------------------------------------------------------------------------------------------\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\n# ---------------------------------------------------------------------------------------------\ndef _reps(L, rows):\n    npk = np.diff(L["off"])[rows]\n    df = pd.DataFrame({"row": rows, "p": L["row_p"][rows], "npk": npk, "tims": L["tims"][rows]})\n    # ADDED: prefer the test instrument (timsTOF) representative, then the richest spectrum\n    df = df.sort_values(["p", "tims", "npk", "row"], ascending=[True, False, False, True]).drop_duplicates("p")\n    rep = df.row.values; rep_nm = L["nm"][rep]\n    o = np.argsort(rep_nm, kind="mergesort"); rep = rep[o]\n    roff, rmz, rit = pv.clean_store(rep, L["off"], L["mz"], L["it"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n    return dict(rep=rep, rep_p=L["row_p"][rep], rep_nm=L["nm"][rep], roff=roff, rmz=rmz, rit=rit)\n\n\ndef build_index(L):\n    ok = np.isfinite(L["nm"]) & (L["row_p"] >= 0)\n    rows = np.where(ok)[0]\n    o = np.argsort(L["nm"][rows], kind="mergesort")\n    I = dict(lib_rows=rows[o], lib_nm=L["nm"][rows[o]])\n    I["all"] = _reps(L, rows)\n    I["pos"] = _reps(L, rows[L["pol"][rows] == 1])\n    I["neg"] = _reps(L, rows[L["pol"][rows] == -1])\n    log(f"index: {len(rows):,} library rows; representatives {len(I[\'all\'][\'rep\']):,}")\n    return I\n\n\n# ---------------------------------------------------------------------------------------------\n# per-molecule channels (identical to e02_channels.molecule_channels)\n# ---------------------------------------------------------------------------------------------\ndef clean(mz, it):\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n\n\ndef analog_search(R, Q, target):\n    lo = np.searchsorted(R["rep_nm"], target - CFG.ANALOG_WIN, "left")\n    hi = np.searchsorted(R["rep_nm"], target + CFG.ANALOG_WIN, "right")\n    asim = np.zeros(hi - lo, np.float32)\n    if hi > lo and Q:\n        shift = (target - R["rep_nm"][lo:hi]).astype(np.float32)\n        for qm, qp in Q:\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R["roff"], R["rmz"], R["rit"], CFG.MZ_TOL, shift))\n    return R["rep_p"][lo:hi], asim\n\n\ndef top_analogs(p, s):\n    if len(p) == 0: return []\n    df = pd.DataFrame({"p": p, "s": s}).groupby("p", sort=False).s.max().sort_values(ascending=False, kind="mergesort")\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\n\n\ndef molecule_channels(L, I, specs, precs, pols, target):\n    Q = [clean(a, b) for a, b in specs]\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\n    tol = target * CFG.PPM_WIN / 1e6\n    lo = np.searchsorted(I["lib_nm"], target - tol, "left"); hi = np.searchsorted(I["lib_nm"], target + tol, "right")\n    crow = I["lib_rows"][lo:hi]\n    u, inv = np.unique(L["row_p"][crow], return_inverse=True)\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\n    for j, (qm, qp) in enumerate(Q):\n        if len(crow) == 0: break\n        s = pv.search(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\n        np.maximum.at(M[j], inv, s)\n        sh = (precs[j] - L["prec"][crow]).astype(np.float32)\n        s2 = pv.search_shift_rows(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\n        np.maximum.at(MS[j], inv, s2)\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\n    z = np.zeros(0, np.float32)\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\n    p, s = analog_search(I["all"], Q, target)\n    ana = top_analogs(p, s)\n    ps, ss = [], []\n    for sign, name in ((1, "pos"), (-1, "neg")):\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\n        if QQ:\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\n    return lib, ana, anapol\n\n\ndef parse_ce(v):\n    try:\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\n        return float(a.mean()) if len(a) else 25.0\n    except Exception:\n        return 25.0\n\n\n\n\n# ---------------------------------------------------------------------------------------------\n# ranking features: prvsiyan\'s 31 + our blocks (identical to e05_rank.py BLOCKS)\n# ---------------------------------------------------------------------------------------------\ndef _rel(v):\n    v = np.asarray(v, np.float32)\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\n\n\ndef _analog_feats(cfp, ana):\n    nc = len(cfp)\n    if not ana: return np.zeros((nc, 5), np.float32)\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\n    inter = cf @ afp.T\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\n    ap = (tan * (w ** 3)[None, :]).max(1)\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\n\n\ndef blk_base(r, cfp):\n    ana = r["ana"]\n    afp = pool_fps([p for p, _ in ana]) if ana else None\n    sims = np.array([s for _, s in ana], np.float32)\n    return pv.rank_features(cfp, r["lib"], afp, sims, r["zlog"], r["frag"])\n\n\ndef blk_mass(r, cfp):\n    ppm = (POOL["mass"][r["cand"]] - r["target"]) / r["target"] * 1e6\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\n\n\ndef blk_libx(r, cfp):\n    cnt = np.log1p(r["lib_cnt"])\n    return np.column_stack(_rel(r["lib_mean"]) + _rel(r["libsh"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\n\n\ndef blk_anapol(r, cfp):\n    return _analog_feats(cfp, r["anapol"])\n\n\ndef blk_fragx(r, cfp):\n    return np.column_stack(_rel(r["frag_ad"]) + _rel(r["frag_mean"]) + [r["frag_pol"]]).astype(np.float32)\n\n\nBLOCKS = {"base": blk_base, "mass": blk_mass, "libx": blk_libx, "anapol": blk_anapol, "fragx": blk_fragx}\n\n\n# ---------------------------------------------------------------------------------------------\n# run\n# ---------------------------------------------------------------------------------------------\ndef compute_channels(L, I, models, te):\n    mols = list(te.groupby("molecule_id", sort=True))\n    recs = []\n    for gi, (mid, sub) in enumerate(mols):\n        specs = []\n        for r in sub.itertuples():\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\n            k_ = a_ <= float(r.precursor_mz) + 2.0\n            a_, b_ = a_[k_], b_[k_]\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\n        nms = nms[np.isfinite(nms)]\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\n        if len(nms):\n            target = float(np.median(nms))\n            modes = np.where(sub.ionization_mode.astype(str).values == "positive", 1.0, -1.0)\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\n            cand = pool_window(target, CFG.PPM_WIN)\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\n            import pv_fp\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\n                                         ces, modes)\n            u, lmax, lmean, lsh, lcnt = lib\n            pos = {int(x): i for i, x in enumerate(u)}\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\n                       ana=ana, anapol=anapol)\n            for nm_, arr in (("lib", lmax), ("lib_mean", lmean), ("libsh", lsh), ("lib_cnt", lcnt)):\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\n        recs.append(rec)\n        if gi % 50 == 0: log(f"  channels {gi}/{len(mols)}")\n    return mols, recs\n\n\ndef compute_frag(recs, workers):\n    from multiprocessing import Pool as MPool\n    uc = np.unique(np.concatenate([r["cand"] for r in recs]))\n    log(f"MetFrag-lite on {len(uc):,} candidates")\n    with MPool(workers) as mp:\n        fr = mp.map(pv.frag_masses_safe, list(POOL["smiles"][uc]), chunksize=16)\n    fmap = dict(zip(uc.tolist(), fr))\n    for r in recs:\n        if not len(r["cand"]): continue\n        peaks = []\n        for a_, b_ in r.pop("specs"):\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n        F1 = np.zeros((len(r["cand"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\n        for j, c in enumerate(r["cand"]):\n            fm = fmap[int(c)]\n            for k, (x, y) in enumerate(peaks):\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r["mode"], tol=CFG.MZ_TOL)\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r["adducts"][k], tol=CFG.MZ_TOL)\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r["modes"][k], tol=CFG.MZ_TOL)\n        r["frag"] = F1.max(1); r["frag_ad"] = F2.max(1); r["frag_mean"] = F3.mean(1); r["frag_pol"] = F3.max(1)\n\n\ndef _hgb(X, Y, W, seed, gbm):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\n    return m\n\n\ndef fit_rankers(rank_train_path):\n    """Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim)."""\n    z = np.load(rank_train_path, allow_pickle=True)\n    X, Y, S = z["X"], z["Y"], z["S"]\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\n    log(f"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features")\n    return rankers, [str(b) for b in z["blocks"]]\n\n\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\n    """prvsiyan\'s ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim)."""\n    z = np.load(pv_rank_path)\n    rankers = [_hgb(z["X"], z["Y"], np.where(z["M"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\n    log(f"prvsiyan ranker: {len(rankers)} GBMs on {z[\'X\'].shape[0]:,} rows x {z[\'X\'].shape[1]} features")\n    return rankers, z["X"].shape[1]\n\n\ndef score_molecules(recs, rankers, blocks, use_fp):\n    out = {}\n    for r in recs:\n        if not len(r["cand"]): continue\n        rr = r if use_fp else dict(r, zlog=None)\n        cfp = pool_fps(r["cand"])\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n        out[r["mid"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\n    return out\n\n\ndef blend_scores(a, b, wa):\n    """Within-molecule rank blend (both inputs are per-candidate probabilities)."""\n    out = {}\n    for mid in a:\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\n    return out\n\n\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\n    """score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all."""\n    from multiprocessing import Pool as MPool\n    cand = {r["mid"]: r["cand"] for r in recs if len(r["cand"])}\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind="mergesort")][:topn + 15] for mid, sc in S.items()}\n               for n, S in score_sets.items()}\n    allc = [v for o in ordered.values() for v in o.values()]\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\n    with MPool(workers) as mp:\n        ck = mp.map(canon_key, list(POOL["smiles"][short]), chunksize=64)\n    ckey = dict(zip(short.tolist(), ck))\n    samp = pd.read_csv(sample_path)\n    subs = {}\n    for n, o in ordered.items():\n        rows = []\n        for mid, _ in mols:\n            out, seen = [], set()\n            for c in o.get(mid, []):\n                k = ckey.get(int(c)) or POOL["keys"][c]\n                if k in seen: continue\n                seen.add(k); out.append(POOL["smiles"][c])\n                if len(out) == topn: break\n            while len(out) < topn:\n                out.append("CCO")\n            rows.append((mid, ";".join(out[:topn])))\n        sub = samp[["molecule_id"]].merge(pd.DataFrame(rows, columns=["molecule_id", "smiles"]), on="molecule_id", how="left")\n        sub["smiles"] = sub["smiles"].fillna("CCO")\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\n        assert (sub.smiles.str.split(";").map(len) <= topn).all()\n        subs[n] = sub\n    return subs\n\n\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\n    global POOL\n    import torch\n    import pv_fp\n    L = load_library(train_path); log(f"library {len(L[\'prec\']):,} spectra")\n    POOL = build_pool(L, workers)\n    L["row_p"] = POOL["k2i"].reindex(L["ik"]).fillna(-1).values.astype(np.int64)\n    del L["ik"], L["smi"]\n    I = build_index(L)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\n    log(f"fp models: {len(single)} single + {len(merged)} merged on {dev}")\n    te = pq.read_table(test_path).to_pandas()\n    if limit:\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\n    log(f"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules")\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\n    del L, I\n    compute_frag(recs, workers)\n    ours, blocks = fit_rankers(our_rank_path)\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\n    del ours\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\n    s_pv = score_molecules(recs, pvr, ["base"], use_fp=True)\n    del pvr\n    subs = make_submissions(mols, recs, {"blend": blend_scores(s_pv, s_ours, w_pv), "pv": s_pv, "ours": s_ours},\n                            sample_path, workers)\n    log("submissions ready")\n    return subs, recs, {"pv": s_pv, "ours": s_ours}\n'}
for _n, _src in ENG_FILES.items():
    open(os.path.join(ENG_DIR, _n), 'w').write(_src)
open(os.path.join(ENG_DIR, 'eng_runner.py'), 'w').write('"""Our engine (two-ranker, BIO + AFIX) -> {molecule_id: {smiles: [...], keys: [...]}} (blend order, top 40)."""\nimport os, sys, json, time\nimport numpy as np\nif __name__ == \'__main__\':\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\n    test, train, sample, out = sys.argv[1:5]\n    import casmi_engine as E\n    _orig_find = E.find\n    def _find(n):                                  # v4g ships its own fp_bits.npy (10,226 bits): take prvsiyan\'s 6,930\n        if n == \'fp_bits.npy\':\n            return os.path.join(os.path.dirname(_orig_find(\'coco_fp.npy\')), \'fp_bits.npy\')\n        return _orig_find(n)\n    E.find = _find\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\n    import glob\n    fp_models = sorted(p for p in glob.glob(\'/kaggle/input/**/fp_*.pt\', recursive=True) if \'casmi26-fp-models-v2\' in p)\n    print(\'engine fp models\', fp_models, flush=True)\n    T0 = time.time()\n    subs, recs, S = E.main(test, train, sample, E.find(\'sim_rank_rows_nofp.npz\'), E.find(\'rank_train.npz\'), fp_models,\n                           workers=os.cpu_count(), w_pv=0.88)\n    bl = E.blend_scores(S[\'pv\'], S[\'ours\'], 0.88)\n    res = {}\n    for r in recs:\n        mid = r[\'mid\']\n        if mid not in bl:\n            continue\n        order = np.argsort(-bl[mid], kind=\'mergesort\')[:80]\n        smis, keys, seen = [], [], set()\n        for c in r[\'cand\'][order]:\n            s = E.POOL[\'smiles\'][c]; k = E.canon_key(s)\n            if k is None or k in seen:\n                continue\n            seen.add(k); smis.append(s); keys.append(k)\n            if len(smis) >= 40:\n                break\n        res[str(mid)] = dict(smiles=smis, keys=keys)\n    json.dump(res, open(out, \'w\'))\n    print(\'engine lists\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n')
ENG = {}
try:
    _t0 = time.time()
    rr = subprocess.run([sys.executable, os.path.join(ENG_DIR, 'eng_runner.py'), os.path.join(COMP, 'test.parquet'),
                         os.path.join(COMP, 'train.parquet'), os.path.join(COMP, 'sample_submission.csv'),
                         '/kaggle/working/eng_lists.json'], capture_output=True, text=True, timeout=4 * 3600, cwd=ENG_DIR, env={**os.environ, 'PYTHONHASHSEED': '0'})
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    ENG = json.load(open('/kaggle/working/eng_lists.json'))
    print('engine lists', len(ENG), f'{time.time()-_t0:.0f}s')
except Exception as e:
    print('ENGINE FAILED -> v4g lists kept:', repr(e))


In [ ]:
'''## 4. PubChem-only channel with a popularity prior

This channel screens 105,875,489 PubChem rows and returns up to 25 distinct proposals outside the main pool. Its fingerprint logits combine the FPNet A+B mean with half the fold-safe checkpoint's logits before candidate-score standardisation; the checkpoint excludes identity fold 0 and the panel molecules.

The score is `S = z(fingerprint score) + 0.15 · pop_PC`, with `pop_PC = ln(1 + substances) + ln(1 + PubMed articles) + 0.5 · I_spectral + 1.0 · I_NP + 0.5 · ln(1 + LOTUS references)`. The enriched prior also selects up to 200 additional documented candidates and enters the promotion condition `pop_PC ≥ 5`. A later library gate controls insertion into the main list. Section 11 states the limits of the development measurements.'''


In [ ]:
V17_PATCH = "\n\n# ---- v17 patch: popularity prior (log1p SID + log1p PMID from PubChem, mapped onto this store by stereo-stripped SMILES) ----\n_POP_DIR = os.environ.get('CASMI_POP_DIR'); _POP_LAM = float(os.environ.get('CASMI_POP_LAM', '0')); _POP_UNION = int(os.environ.get('CASMI_POP_UNION', '0'))\n_SRC_BONUS = float(os.environ.get('CASMI_SRC_BONUS', '0')); _SRC_BITS = int(os.environ.get('CASMI_SRC_BITS', '0'))   # v5 depositor-source bonus\n_NP_BONUS = float(os.environ.get('CASMI_NP_BONUS', '0'))   # natural-product membership bonus (data notebook v5)\n_LOTUS_W = float(os.environ.get('CASMI_LOTUS_W', '0'))   # pre-aligned literature prior\n_init_worker0 = init_worker\n\n\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    _init_worker0(pc_dir, bits_path, pool_meta_path, code_dir)\n    if _POP_DIR:\n        _W['ls'] = np.load(os.path.join(_POP_DIR, 'pc_lsid.npy'), mmap_mode='r')\n        _W['lp'] = np.load(os.path.join(_POP_DIR, 'pc_lpmid.npy'), mmap_mode='r')\n        if _SRC_BONUS > 0 and _SRC_BITS:\n            _W['sm'] = np.load(os.path.join(_POP_DIR, 'pc_src_mask.npy'), mmap_mode='r')\n        if _NP_BONUS > 0:\n            _W['np'] = np.load(os.path.join(_POP_DIR, 'pc_np.npy'), mmap_mode='r')\n        if _LOTUS_W > 0:\n            import sys as _sys\n            _sys.path.insert(0, _POP_DIR)\n            from aligned_public_priors import load_lotus_refs, lotus_log_prior\n            _W['lotus'] = load_lotus_refs(_POP_DIR, tier_rows=len(_W['mass']))\n            _W['lotus_log'] = lotus_log_prior\n\n\ndef probe_one(task):\n    mid, target, z = task\n    t0 = time.time()\n    chem = _W['chem']; Chem = _W['Chem']; gen = _W['ecfp4']\n    tol = float(target) * PPM * 1e-6\n    a0 = int(np.searchsorted(_W['mass'], float(target) - tol, 'left'))\n    smis = _window(float(target))\n    n = len(smis)\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0, fz_top=None)\n    if n == 0:\n        diag['secs'] = time.time() - t0\n        return mid, [], [], [], diag\n    raw_e = _W['raw_e']; z_e = z[_W['sel_e']].astype(np.float32)\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\n    for i, s in enumerate(smis):\n        m = Chem.MolFromSmiles(s)\n        if m is None:\n            continue\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\n    sc1 = E @ z_e\n    sc1[~ok] = -np.inf\n    n1 = min(PC_N1, int(ok.sum()))\n    diag['n_pass1'] = n1\n    if n1 == 0:\n        diag['secs'] = time.time() - t0\n        return mid, [], [], [], diag\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\n    use_pop = ('ls' in _W) and _POP_LAM > 0\n    pop = (np.asarray(_W['ls'][a0:a0 + n], np.float32) + np.asarray(_W['lp'][a0:a0 + n], np.float32)) if use_pop else np.zeros(n, np.float32)\n    if use_pop and 'sm' in _W:\n        pop = pop + _SRC_BONUS * ((np.asarray(_W['sm'][a0:a0 + n]) & _SRC_BITS) != 0).astype(np.float32)\n    if use_pop and 'np' in _W:\n        pop = pop + _NP_BONUS * (np.asarray(_W['np'][a0:a0 + n]) != 0).astype(np.float32)\n    if use_pop and 'lotus' in _W:\n        _lr = _W['lotus_log'](_W['lotus'][a0:a0 + n])\n        pop = pop + _LOTUS_W * _lr\n    if use_pop and _POP_UNION > 0:\n        pp = np.where(ok, pop, -np.inf); ku = min(_POP_UNION, int(ok.sum()))\n        top1 = np.union1d(top1, np.argpartition(-pp, ku - 1)[:ku])\n    bits = _W['bits']\n    fz, idx = [], []\n    for i in top1:\n        fp = chem.raw_fingerprint(smis[i])\n        if fp is None:\n            continue\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\n    diag['n_pass2'] = len(idx)\n    if not idx:\n        diag['secs'] = time.time() - t0\n        return mid, [], [], [], diag\n    fz = np.asarray(fz, np.float64); idx = np.asarray(idx)\n    keyc = {}\n    def key(j):\n        if j not in keyc:\n            keyc[j] = chem.score_key(smis[idx[j]]); diag['n_keyed'] += 1\n        return keyc[j]\n    for j in np.argsort(-fz, kind='stable'):        # gate quantity = v16's pc_fz[0] (best non-pool structure by raw f.z)\n        k = key(j)\n        if k is not None and k not in _W['pool_keys']:\n            diag['fz_top'] = float(fz[j]); break\n    zf = (fz - fz.mean()) / (fz.std() + 1e-9)\n    sc = zf + _POP_LAM * pop[idx]\n    order = np.argsort(-sc, kind='stable')\n    out, out_s, out_k, out_p, out_sc, seen = [], [], [], [], [], set()\n    for j in order:\n        k = key(j)\n        if k is None or k in seen:\n            continue\n        seen.add(k)\n        if k in _W['pool_keys']:\n            diag['n_in_pool'] += 1\n            continue\n        out.append(smis[idx[j]]); out_s.append(float(sc[j])); out_k.append(k); out_p.append(float(pop[idx[j]])); out_sc.append(float(sc[j]))\n        if len(out) >= K:\n            break\n    diag['n_listed'] = len(out)\n    diag['S'] = float(out_sc[0]) if out_sc else 0.0\n    diag['top_pop'] = float(out_p[0]) if out_p else 0.0\n    diag['secs'] = time.time() - t0\n    return mid, out, out_s, out_k, diag\n"
'''PubChem-only channel in its own process (S1 code snapshot + FPNet A+B of casmi26-v3-models) -- identical to v3.6'''
CORE = '"""P_pubchem probe core (lever 3). Embedded verbatim into the Kaggle notebook (written to /kaggle/working and imported,\nso worker processes can import it) and imported directly by the local smoke test.\n\nPer molecule:\n  1. f.z logits of the A+B FPNet ensemble, exactly like Engine.model_logits: z = 0.5 * (mean per-spectrum logits +\n     mean per-polarity merged-spectrum logits); models averaged inside ModelBank.\n  2. PubChem window: +-10 ppm around the target neutral mass (median over the molecule\'s spectra), no subsampling.\n  3. pass 1 (src/casmi/pubchem.py tier): partial f.z on the ECFP4 block of the selected bits, keep top PC_N1 = 1000.\n  4. pass 2: full selected-bit fingerprints, full f.z, sort descending.\n  5. walk down the f.z order, compute the metric score key (tautomer-canonical InChIKey14); keep a structure only if\n     its key is NOT in the candidate pool (train U COCONUT) and not already listed; stop at 25.\n  An empty list becomes the single \'CCO\' fallback (never padded).\nReading: sum of reciprocal ranks of PubChem-only truths / ~130 public molecules.\n"""\nfrom __future__ import annotations\n\nimport os\nimport time\n\nimport numpy as np\n\nPC_N1 = 5000\nPPM = 10.0\nK = 25\n\n_W = {}\n\n\n# ------------------------------------------------------------------------------------------ model part (main process)\ndef molecule_logits(bank, spectra):\n    """spectra: list of dict(mz, it, mode, adduct, prec, ce, ce_n). Returns z_avg (nbits,) float32 or None."""\n    from casmi import chem, fpnet\n    from casmi.spectra import merge_spectra\n    items_s = []\n    for s in spectra:\n        pm, pi = fpnet.prep_peaks(s[\'mz\'], s[\'it\'], s[\'prec\'])\n        ce_ok = s[\'ce\'] is not None and not np.isnan(s[\'ce\'])\n        items_s.append(dict(mz=pm, it=pi, prec=float(s[\'prec\']), adduct_ix=chem.adduct_index(s[\'adduct\']),\n                            ce=float(s[\'ce\']) if ce_ok else 0.0, ce_known=1.0 if ce_ok else 0.0,\n                            n_merged=min(int(s.get(\'ce_n\', 1)), 8), mode=float(s[\'mode\'])))\n    items_m = []\n    for md in (1, -1):\n        grp = [s for s in spectra if s[\'mode\'] == md]\n        if not grp:\n            continue\n        mm, ii = merge_spectra([(s[\'mz\'], s[\'it\']) for s in grp])\n        prec = float(np.median([s[\'prec\'] for s in grp]))\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\n        ces = [s[\'ce\'] for s in grp if s[\'ce\'] is not None and not np.isnan(s[\'ce\'])]\n        adducts = [s[\'adduct\'] for s in grp]\n        ad = max(adducts, key=adducts.count)          # deterministic tie-break (first occurrence)\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad),\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                            n_merged=min(sum(max(1, int(s.get(\'ce_n\', 1))) for s in grp), 8), mode=float(md)))\n    if not items_s:\n        return None\n    zs, _ = bank.logits_el(items_s)\n    zm, _ = bank.logits_el(items_m)\n    # 1.5 * mean(A, B, fold0) = mean(A, B) + 0.5 * fold0.\n    # Positive scaling preserves shortlist/order/z-scores while retaining the A+B coefficient in raw-margin gates.\n    return (1.5 * 0.5 * (zs.mean(0) + zm.mean(0))).astype(np.float32)\n\n\n# ------------------------------------------------------------------------------------------ PubChem part (workers)\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    import sys\n    if code_dir and code_dir not in sys.path:\n        sys.path.insert(0, code_dir)\n    import pandas as pd\n    from casmi import chem\n    _W[\'chem\'] = chem\n    _W[\'mass\'] = np.load(os.path.join(pc_dir, \'pc_mass.npy\'), mmap_mode=\'r\')\n    _W[\'off\'] = np.load(os.path.join(pc_dir, \'pc_off.npy\'), mmap_mode=\'r\')\n    _W[\'buf\'] = np.load(os.path.join(pc_dir, \'pc_smiles.npy\'), mmap_mode=\'r\')\n    bits = np.load(bits_path)\n    _W[\'bits\'] = bits\n    _W[\'sel_e\'] = np.where(bits < 4096)[0]\n    _W[\'raw_e\'] = bits[_W[\'sel_e\']]\n    _W[\'pool_keys\'] = set(pd.read_parquet(pool_meta_path, columns=[\'key\']).key.values.tolist())\n    r = chem._rdkit()\n    _W[\'ecfp4\'] = r[\'gen\'].GetMorganGenerator(radius=2, fpSize=4096)\n    _W[\'Chem\'] = r[\'Chem\']\n\n\ndef _window(target):\n    tol = target * PPM * 1e-6\n    a = int(np.searchsorted(_W[\'mass\'], target - tol, \'left\')); b = int(np.searchsorted(_W[\'mass\'], target + tol, \'right\'))\n    off = np.asarray(_W[\'off\'][a:b + 1]).astype(np.int64)\n    if b <= a:\n        return []\n    raw = bytes(_W[\'buf\'][off[0]:off[-1]])\n    base = off[0]\n    return [raw[off[i] - base:off[i + 1] - base].decode(\'ascii\') for i in range(b - a)]\n\n\ndef probe_one(task):\n    """task = (molecule_id, target_mass, z (nbits,) float32). Returns (molecule_id, [smiles], [f.z], [score keys], diag dict)."""\n    mid, target, z = task\n    t0 = time.time()\n    chem = _W[\'chem\']; Chem = _W[\'Chem\']; gen = _W[\'ecfp4\']\n    smis = _window(float(target))\n    n = len(smis)\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0)\n    if n == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    raw_e = _W[\'raw_e\']; z_e = z[_W[\'sel_e\']].astype(np.float32)\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\n    for i, s in enumerate(smis):\n        m = Chem.MolFromSmiles(s)\n        if m is None:\n            continue\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\n    sc1 = E @ z_e\n    sc1[~ok] = -np.inf\n    n1 = min(PC_N1, int(ok.sum()))\n    diag[\'n_pass1\'] = n1\n    if n1 == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\n    bits = _W[\'bits\']\n    fz, idx = [], []\n    for i in top1:\n        fp = chem.raw_fingerprint(smis[i])\n        if fp is None:\n            continue\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\n    diag[\'n_pass2\'] = len(idx)\n    order = np.argsort(-np.asarray(fz), kind=\'stable\')\n    out, out_fz, out_k, seen = [], [], [], set()\n    for o in order:\n        s = smis[idx[o]]\n        k = chem.score_key(s)\n        diag[\'n_keyed\'] += 1\n        if k is None or k in seen:\n            continue\n        seen.add(k)\n        if k in _W[\'pool_keys\']:\n            diag[\'n_in_pool\'] += 1\n            continue\n        out.append(s); out_fz.append(float(fz[o])); out_k.append(k)\n        if len(out) >= K:\n            break\n    diag[\'n_listed\'] = len(out)\n    diag[\'secs\'] = time.time() - t0\n    return mid, out, out_fz, out_k, diag\n'
RUNNER = '"""PubChem-only channel for the final submission (runs in its own process so its casmi package\n(S1 snapshot, FPNet A+B) never collides with the v1 engine\'s casmi package).\n\nPer molecule: z = A+B logits (probe_core2.molecule_logits, identical to the probe that scored 0.037);\nbest_pool_fz = max f.z over the pool window (+-10 ppm, fallback 30 ppm) with the same z (as P1 logits.py);\nPubChem-only list = probe_core2.probe_one (top 25 by f.z, keys not in the pool) with f.z values and score keys.\nWrites a json {molecule_id: {pc, pc_fz, pc_keys, best_pool_fz, target}}.\nusage: python pc_runner.py <v3_code_dir> <v3_models_dir> <pool_dir> <pubchem_dir> <test_parquet> <out_json> <n_workers> [max_mols]\n"""\nimport os, sys, json, time\nimport numpy as np, pandas as pd\n\nif __name__ == \'__main__\':\n    V3CODE, V3MOD, POOL_DIR, PC_DIR, TEST, OUT, NW = sys.argv[1:8]\n    MAXM = int(sys.argv[8]) if len(sys.argv) > 8 else 0\n    NW = int(NW)\n    sys.path.insert(0, V3CODE); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n    import torch, glob\n    import probe_core2 as pc\n    from casmi import fpnet, chem\n    T0 = time.time()\n    models = sorted(glob.glob(os.path.join(V3MOD, \'fpnet_*.pt\')))\n    assert len(models) == 2, models\n    fold0 = os.environ[\'CASMI_FPNET_FOLD0\']\n    assert os.path.isfile(fold0), fold0\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    bank = fpnet.ModelBank(models + [fold0], device=dev)\n    bits = np.load(os.path.join(POOL_DIR, \'fp_bits.npy\')); nbits = len(bits)\n    assert bank.nbits == nbits, (bank.nbits, nbits)\n    assert np.array_equal(bits, np.load(os.path.join(os.path.dirname(fold0), \'fp_bits.npy\'))), \'Fingerprint bit order differs\'\n    pmass = pd.read_parquet(os.path.join(POOL_DIR, \'pool_meta.parquet\'), columns=[\'mass\']).mass.values.astype(np.float64)\n    assert np.all(np.diff(pmass) >= 0), \'pool not mass-sorted\'\n    pfp = np.load(os.path.join(POOL_DIR, \'pool_fp.npy\'), mmap_mode=\'r\')\n    te = pd.read_parquet(TEST)\n    te[\'nm\'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\n    tasks, extra = [], {}\n    groups = list(te.groupby(\'molecule_id\', sort=False))\n    if MAXM:\n        groups = groups[:MAXM]\n    for mid, sub in groups:\n        nms = sub.nm.values[np.isfinite(sub.nm.values)]\n        if not len(nms):\n            continue\n        target = float(np.median(nms))\n        spectra = []\n        for r in sub.itertuples():\n            ce = r.collision_energy_ev\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                                mode=1 if r.ionization_mode == \'positive\' else -1, adduct=r.adduct,\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\n        spectra = spectra[:16]\n        try:\n            z = pc.molecule_logits(bank, spectra)\n        except Exception as e:\n            print(\'ERROR logits\', mid, repr(e), flush=True); z = None\n        if z is None:\n            continue\n        tol = target * 10e-6\n        a = np.searchsorted(pmass, target - tol, \'left\'); b = np.searchsorted(pmass, target + tol, \'right\')\n        if b <= a:\n            tol = target * 30e-6\n            a = np.searchsorted(pmass, target - tol, \'left\'); b = np.searchsorted(pmass, target + tol, \'right\')\n        if b > a:\n            F = np.unpackbits(np.asarray(pfp[a:b]), axis=1)[:, :nbits].astype(np.float64)\n            best = float((F @ z.astype(np.float64)).max())\n        else:\n            best = float(\'nan\')\n        extra[mid] = dict(best_pool_fz=best, target=target)\n        tasks.append((mid, target, z))\n    del bank; torch.cuda.empty_cache()\n    print(f\'logits for {len(tasks)} molecules {time.time()-T0:.0f}s\', flush=True)\n    res = {}\n    import multiprocessing as mp\n    ctx = mp.get_context(\'fork\' if sys.platform != \'win32\' else \'spawn\')\n    with ctx.Pool(NW, initializer=pc.init_worker, initargs=(PC_DIR, os.path.join(POOL_DIR, \'fp_bits.npy\'),\n                                                            os.path.join(POOL_DIR, \'pool_meta.parquet\'), V3CODE)) as P:\n        for k, (mid, smis, fzs, keys, d) in enumerate(P.imap_unordered(pc.probe_one, tasks, chunksize=1)):\n            res[mid] = dict(pc=smis, pc_fz=fzs, pc_keys=keys, fz_top=d.get(\'fz_top\'), S=d.get(\'S\'), top_pop=d.get(\'top_pop\', 0.0), **extra[mid])\n            if (k + 1) % 25 == 0 or k + 1 == len(tasks):\n                print(f\'  pubchem {k+1}/{len(tasks)} {time.time()-T0:.0f}s\', flush=True)\n    json.dump(res, open(OUT, \'w\'))\n    print(\'pc_runner done\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n'
open('/kaggle/working/probe_core2.py', 'w').write(CORE + V17_PATCH)
open('/kaggle/working/pc_runner.py', 'w').write(RUNNER)
print('v17 patch appended to probe_core2:', 'def probe_one' in V17_PATCH)
PC = {}
try:
    rr = subprocess.run([sys.executable, '/kaggle/working/pc_runner.py', V3CODE, V3MOD, POOL_DIR, PC_DIR,
                         os.path.join(COMP, 'test.parquet'), '/kaggle/working/pc_lists.json', '4'],
                        capture_output=True, text=True, timeout=4 * 3600, env={**os.environ, 'PYTHONHASHSEED': '0'})
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    PC = json.load(open('/kaggle/working/pc_lists.json'))
except Exception as e:
    print('PUBCHEM CHANNEL FAILED -> base lists only:', repr(e))
print('pubchem lists', len(PC), f'{time.time()-T0:.0f}s')


In [ ]:
'''## 5. Engine 1: Ahmed Berat Özer's v4n

This step builds the spectral library from `train.parquet` and links the precomputed candidate pool: training structures plus COCONUT, about 711k in total. The pool comes with FPNet fingerprints and fragment tables.
'''


In [ ]:
'''v1 engine code (the 0.358 model's engine) + fe_v4; pool symlinks; spectrum cache from train.parquet'''
sys.path.insert(0, V4CODE)
os.makedirs('work', exist_ok=True)
for f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:
    src = os.path.join(POOL_DIR, f)
    dst = f'work/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
from casmi.build import build_spec_cache
build_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')
import casmi; print('casmi from', casmi.__file__)
sys.path.insert(1, os.path.join(V4CODE, 'fe_v4'))
print(f'cache built {time.time()-T0:.0f}s')


In [ ]:
'''v1 engine exactly as in v3.6 (FPNet A, EngineCfg(generate=True)) wrapped by V1FE (recording hooks + families)'''
import numpy as np, pandas as pd, torch, lightgbm as lgb
from casmi.library import Library
from casmi.pool import Pool
from casmi.engine import Engine, EngineCfg, FEATURES
from casmi import fpnet, chem
import v1engine
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
L = Library('work'); P = Pool('work')
tfp = np.load('work/train_fp_sel.npy')
_FULL = glob.glob('/kaggle/input/**/casmi26-fpnet-full1/**/fpnet_full1.pt', recursive=True)   # v4m
bank = fpnet.ModelBank([_FULL[0] if _FULL else os.path.join(V4MOD, 'fpnet_0.pt')], device=dev)
print('engine FP bank:', _FULL[0] if _FULL else 'fpnet_0 (A) FALLBACK', flush=True)
E = Engine(L, P, tfp, EngineCfg(generate=True), bank)
V = v1engine.V1FE(E, [os.path.join(V4MOD, m['file']) for m in MAN['fe_models']], MAN['families'], device=dev)
if 'derivation' in MAN['families']:
    n_bad = V.mods['derivation'].self_test()
    print('derivation prior self-test mismatches:', n_bad)
    assert n_bad == 0, 'RDKit build differs from the one the derivation priors were built with'
RK = pickle.load(open(os.path.join(V4MOD, MAN['ranker']['file']), 'rb'))
RK_FEATS = list(RK['features']); RK_B = [lgb.Booster(model_str=s) for s in RK['boosters']]
def rank_score(X, names):
    pos = {n: i for i, n in enumerate(names)}
    miss = [f for f in RK_FEATS if f not in pos]
    assert not miss, f'ranker features not produced (would be zero-filled): {miss}'
    Xs = np.ascontiguousarray(X[:, [pos[f] for f in RK_FEATS]], dtype=np.float32)
    return np.mean([b.predict(Xs, num_threads=4) for b in RK_B], axis=0)
print('engine ready; ranker features', len(RK_FEATS), 'boosters', len(RK_B), f'{time.time()-T0:.0f}s')


In [ ]:
'''## 6. Main candidate lists + pool popularity re-rank

For each molecule, v4n scores candidates with FPNet fingerprints, its feature families and a 4-booster LightGBM ranker, and keeps the top 60. The list is then re-ranked by

$$z(\\text{ranker score}) + 0.15\\,\\big(\\log(1+\\#\\text{substances}) + \\log(1+\\#\\text{papers})\\big)$$

Only pool structures are re-ranked. Generated candidates keep their positions and form their own simulator groups. Historical complete-version scores are listed in Section 14.
'''


In [ ]:
'''all molecules -> base lists (v3.6 spectra construction and ordering)'''
te = pd.read_parquet(os.path.join(COMP, 'test.parquet'))
te['nm'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id', sort=False))
BASE, n_err = {}, 0
POOLPOP_MU = 0.15
POOL_POP = np.load(find('pool_lsid.npy')).astype(np.float64) + np.load(find('pool_lpmid.npy')).astype(np.float64)
assert len(POOL_POP) == len(P.key), (len(POOL_POP), len(P.key))
n_pp = 0
print('v19 pool popularity re-rank, mu', POOLPOP_MU, 'pool', len(POOL_POP))
for gi, (mid, sub) in enumerate(mols):
    nms = sub.nm.values[np.isfinite(sub.nm.values)]
    smis, keys, lib_max, scs, forms = [], [], 0.0, [], []
    pids_ = []
    if len(nms):
        target = float(np.median(nms))
        spectra = []
        for r in sub.itertuples():
            ce = r.collision_energy_ev
            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)
            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),
                                mode=1 if r.ionization_mode == 'positive' else -1, adduct=r.adduct,
                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,
                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))
        try:
            C, X, F, names, info = V.run(spectra, target)
            lib_max = float(info.get('lib_max', 0.0))
            if len(C.pid):
                score = rank_score(np.concatenate([X, F], 1), list(FEATURES) + list(names))
                order = np.argsort(-score, kind='stable')
                seen = set()
                for i in order:
                    k = C.key[i]
                    if k in seen: continue
                    seen.add(k); smis.append(C.smiles[i]); keys.append(k); scs.append(float(score[i])); forms.append(C.formula[i]); pids_.append(int(C.pid[i]))
                    if len(smis) >= TOPN: break
        except Exception as e:
            print('ERROR', mid, repr(e)); n_err += 1
    if POOLPOP_MU > 0 and len(scs) > 1:            # v19: popularity re-rank of the top-60 base list
        try:
            s_ = np.asarray(scs, np.float64); z_ = (s_ - s_.mean()) / (s_.std() + 1e-9)
            pid_ = np.asarray(pids_); isg_ = pid_ < 0
            f_ = z_ + POOLPOP_MU * np.where(isg_, 0.0, POOL_POP[np.maximum(pid_, 0)])
            pool_i = np.where(~isg_)[0]; o_ = np.empty(len(pid_), np.int64)
            o_[isg_] = np.where(isg_)[0]; o_[~isg_] = pool_i[np.argsort(-f_[pool_i], kind='stable')]   # generated keep their slots
            smis = [smis[i] for i in o_]; keys = [keys[i] for i in o_]; scs = [float(f_[i]) for i in o_]
            forms = [forms[i] + ('|gen' if isg_[i] else '') for i in o_]      # generated form their own ICEBERG groups
            n_pp += 1
        except Exception as e:
            print('poolpop failed', mid, repr(e))
    BASE[mid] = (smis, keys, lib_max, scs, forms)
    if (gi + 1) % 25 == 0 or gi == len(mols) - 1:
        print(f'  {gi+1}/{len(mols)} molecules {time.time()-T0:.0f}s', flush=True)
print('errors', n_err, '| popularity re-ranked lists', n_pp)
'''Keep ranked overflow for append-only final coverage.'''
BASE_TAIL_CANDIDATES = {str(k): list(v[0]) for k, v in BASE.items()}


In [ ]:
'''## 7. ICEBERG + GLACIER spectrum simulators

Both models predict candidate spectra for comparison with measured views. Second-engine candidates join the simulator input. GLACIER uses [M+H]+ spectra only. At this stage the main list keeps its ranker order; within-formula PubChem reordering is applied only where ICEBERG returned scores, using GLACIER first and ICEBERG where GLACIER has none. The final fused-list pass is described below and in Section 10.'''


In [ ]:
'''--- CLAW FIX (b418/b419): stage health is now RECORDED, never silently swallowed ---'''
'''`glob('/kaggle/input/**/x')` WITHOUT recursive=True matches one directory level only, so it'''
'''misses /kaggle/input/datasets/<owner>/<slug>/. That silently disabled ICEBERG+GLACIER:'''
'''ICE FAILED -> ranker order kept: FileNotFoundError('cp313 rdkit wheel not found')  [b417 log]'''
'''rdkit-2025.3.6-cp312-...whl is not a supported wheel on this platform               [b417 log]'''
'''recursive=True finds the wheel under BOTH mount layouts (same 2025.3.6 cp313 file either way).'''
CLAW_ICE_HEALTH = 'ICE NOT RUN'
CLAW_GL_HEALTH = 'GL NOT RUN'
BASE_ICE = False         # b421: base-list ICEBERG fusion OFF (b420 showed it costs 0.009); see header
'''ICEBERG post-ranker isomer re-scoring (casmi26-iceberg). Optional: any failure keeps the ranker order.'''
import gc
ICE_SCORES, ice_stats = {}, dict(molecules=0, changed_top25=0, changed_top1=0)


'''--- CLAW FIX (b415c): make ICEBERG/GLACIER runnable on a cp313 Kaggle image ---'''
'''The ICEBERG package bundles rdkit-2025.3.6 built for **cp312**; this image is cp313,'''
'''so pip rejects that wheel and install_site() raises before anything is scored'''
'''(n_mols_scored == 0 for ICEBERG *and* GLACIER). The MANIFEST pins 2025.3.6 because'''
'''"FragmentEngine Kekulize differs in 2026.03", so swapping in the runtime's 2026.03'''
'''rdkit would change the scoring. Instead we install the SAME 2025.3.6 release built'''
'''for cp313, so the pinned version is honoured exactly and RDKIT_REQ passes natively.'''
''''''
'''install_site() is idempotent via <site>/.ice_site_ok, which it compares against the'''
'''name list of `pkg/wheels`. We pre-build the site and write that exact list, so both'''
'''runners short-circuit and never invoke the failing pip.'''
def _claw_preseed_site(pkg, site):
    import glob as _g, os as _o, shutil as _s, subprocess as _sp, sys as _sys
    srcs = sorted(_g.glob(_o.path.join(pkg, 'wheels', '*.whl.ice')) +
                  _g.glob(_o.path.join(pkg, 'wheels', '*.whl')))
    # install everything EXCEPT the cp-tag-mismatched rdkit
    keep = [w for w in srcs if not _o.path.basename(w).lower().startswith('rdkit')]
    dropped = [_o.path.basename(w) for w in srcs if w not in keep]
    new = [simulator_rdkit_wheel(pkg)]
    keep = keep + new[:1]
    _o.makedirs(site, exist_ok=True)
    wdir = _o.path.join(site, '.wheels')
    _o.makedirs(wdir, exist_ok=True)
    whl = []
    for w in keep:
        dst = _o.path.join(wdir, _o.path.basename(w)[:-4] if w.endswith('.ice') else _o.path.basename(w))
        _s.copyfile(w, dst)
        whl.append(dst)
    r = _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',
                 '--no-compile', '--disable-pip-version-check', '--upgrade', '--target', site] + whl,
                stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)
    print('[claw] site preseed rc', r.returncode, (r.stdout or '')[-400:], flush=True)
    if r.returncode != 0:
        raise RuntimeError('claw preseed pip failed: ' + (r.stdout or '')[-600:])
    _s.rmtree(wdir, ignore_errors=True)
    # the marker MUST equal exactly what install_site() derives from the PACKAGE list
    want = '\n'.join(_o.path.basename(w) for w in srcs)
    with open(_o.path.join(site, '.ice_site_ok'), 'w') as f:
        f.write(want)
    print('[claw] site ready; substituted', dropped, '->', [_o.path.basename(w) for w in new[:1]], flush=True)
    return want


try:
    ICE_PKG = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))
    sys.path.append(ICE_PKG)
    import fuse as ice_fuse
    del V, E, bank; gc.collect(); torch.cuda.empty_cache()
    # priority: molecules without a library-level match first (class 1 rarely changes), then the rest
    mids = sorted(BASE, key=lambda m: BASE[m][2])
    cands = {m: (ice_fuse.ice_candidates(BASE[m][0], BASE[m][4], TOPN) if BASE_ICE else []) for m in mids}
    if ENG:                              # ADDED (ICE_UNION): also score our engine's candidates (same-formula groups
        from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF   # formed over the v4g top-N + engine top-40)
        from rdkit import Chem as _Ch
        def _f2(s):
            try:
                return _CMF(_Ch.MolFromSmiles(s))
            except Exception:
                return s
        n_add = 0
        for m in mids:
            e = ENG.get(str(m))
            if not e or not e.get('smiles'):
                continue
            bs, bk, bf = list(BASE[m][0][:TOPN]), set(BASE[m][1][:TOPN]), list(BASE[m][4][:TOPN])
            es = [s for s, k in zip(e['smiles'][:40], e['keys'][:40]) if k not in bk]
            ef = [_f2(s) for s in es]
            extra = ice_fuse.ice_candidates(bs + es, bf + ef, len(bs) + len(es))
            before = len(cands.get(m, []))
            cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))
            n_add += len(cands[m]) - before
        print('ICE_UNION: engine candidates added to ICE input:', n_add, flush=True)
    if ICE_PC:                          # also the PubChem-only list of molecules the v3.6 gate may use (lib_max < LIB_TAU)
        from rdkit.Chem.rdMolDescriptors import CalcMolFormula
        from rdkit import Chem as _Chem
        def _form(s):
            try:
                return CalcMolFormula(_Chem.MolFromSmiles(s))
            except Exception:
                return s
        for m in mids:
            p = PC.get(m) if isinstance(PC, dict) else None
            if p and p.get('pc') and BASE[m][2] < LIB_TAU:
                p['pc_form'] = [_form(x) for x in p['pc']]
                extra = ice_fuse.ice_candidates(p['pc'], p['pc_form'], 25)
                cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))
    cands = {m: c for m, c in cands.items() if c}
    items = ice_fuse.build_ice_input(te, cands)
    print('ICE input molecules', len(items), 'candidates', sum(len(it['cands']) for it in items), f'{time.time()-T0:.0f}s', flush=True)
    _claw_preseed_site(ICE_PKG, '/kaggle/working/ice_site')
    ICE_SCORES = ice_fuse.run_ice(ICE_PKG, items, workdir='/kaggle/working/ice_work', device='cuda', budget_s=ICE_BUDGET,
                                  site='/kaggle/working/ice_site', extra_args=['--any-rdkit'])
    meta_f = '/kaggle/working/ice_work/ice_out.json.meta.json'
    if os.path.exists(meta_f):
        print('ICE meta', open(meta_f).read()[:1500])
except Exception as e:
    CLAW_ICE_HEALTH = f'ICE FAILED: {e!r}'
    print('ICE FAILED -> ranker order kept:', repr(e), flush=True)
'''v4h: GLACIER third re-scoring term (casmi26-glacier) on the SAME runner input; any GLACIER failure -> v4g (ICE only)'''
GL_SCORES, gl_stats, GL_LAM, GL_BUDGET = {}, dict(molecules=0, changed_vs_ice_top25=0, changed_vs_ice_top1=0), 1.0, 4000
try:
    GL_PKG = os.path.dirname(find('casmi26-glacier/**/gl_runner.py'))
    sys.path.append(GL_PKG)
    import gl_fuse
    gc.collect(); torch.cuda.empty_cache()   # ICE ran in its own (finished) process; free the notebook's cached GPU memory
    gl_items = [dict(it, spectra=[s for s in it['spectra'] if s.get('adduct') == '[M+H]+']) for it in items]   # v29: GLACIER on [M+H]+ only (v27: +0.006)
    gl_items = [it for it in gl_items if it['spectra']]
    print('v29 GLACIER [M+H]+ only: molecules', len(gl_items), 'of', len(items))
    GL_SCORES = gl_fuse.run_gl(GL_PKG, gl_items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=GL_BUDGET,
                               site='/kaggle/working/ice_site', wheels=os.path.join(ICE_PKG, 'wheels'), extra_args=['--any-rdkit'])
    meta_f = '/kaggle/working/gl_work/gl_out.json.meta.json'
    print('GL meta', open(meta_f).read()[:1500] if os.path.exists(meta_f) else 'missing')
except Exception as e:
    GL_SCORES = {}
    CLAW_GL_HEALTH = f'GL FAILED: {e!r}'
    print('GL FAILED -> v4g behaviour (ICE only):', repr(e), flush=True)
def gl_of(m):
    g = GL_SCORES.get(str(m), {}) if isinstance(GL_SCORES, dict) else {}
    return g if isinstance(g, dict) and any(v is not None for v in g.values()) else {}
def gl_rerank(order0, smis, keys, scs, forms, ice, gl, top_n, tag=''):
    # z(scs) + ICE_LAM z(ice) + GL_LAM z(gl) in the same slots; no GL scores / any failure -> order0 (= v4g order)
    if not gl:
        return order0
    try:
        o = gl_fuse.rerank_multi(smis, keys, scs, forms, [ice, gl], [ICE_LAM, GL_LAM], top_n=top_n)
    except Exception as e:
        print('gl rerank failed', tag, repr(e)); return order0
    for k, v in (('molecules', 1), ('changed_vs_ice_top25', int(o[:25] != order0[:25])), ('changed_vs_ice_top1', int(o[:1] != order0[:1]))):
        gl_stats[tag + k] = gl_stats.get(tag + k, 0) + v
    return o
for m in list(BASE):
    smis, keys, lib_max, scs, forms = BASE[m]
    ice = ICE_SCORES.get(str(m), {}) if ICE_SCORES else {}
    gl = gl_of(m)
    if BASE_ICE and ice and any(v is not None for v in ice.values()):
        try:
            order = ice_fuse.rerank(smis, keys, scs, forms, ice, lam=ICE_LAM, top_n=TOPN)
            order = gl_rerank(order, smis, keys, scs, forms, ice, gl, TOPN)
            s2 = [smis[i] for i in order]; k2 = [keys[i] for i in order]
            ice_stats['molecules'] += 1
            ice_stats['changed_top25'] += int(s2[:25] != smis[:25]); ice_stats['changed_top1'] += int(bool(s2) and s2[0] != smis[0])
            smis, keys = s2, k2
        except Exception as e:
            print('rerank failed', m, repr(e))
    BASE[m] = (smis[:25], keys[:25], lib_max)
    p = PC.get(m) if (ICE_PC and isinstance(PC, dict)) else None
    if p and p.get('pc_form') and ice:
        try:                             # re-order the PubChem list by z(f.z) + lam z(ice) inside same-formula groups;
            o = ice_fuse.rerank(p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, lam=ICE_LAM, top_n=25)
            o = gl_rerank(o, p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, gl, 25, 'pc_')
            fz0 = p['pc_fz'][0]          # the gate keeps using the ORIGINAL top f.z (rel), so gate decisions are unchanged
            p['pc'] = [p['pc'][i] for i in o]; p['pc_keys'] = [p['pc_keys'][i] for i in o]
            p['pc_fz'] = [fz0] + [p['pc_fz'][i] for i in o][1:]
            ice_stats['pc_changed'] = ice_stats.get('pc_changed', 0) + int(o[:1] != [0])
        except Exception as e:
            print('pc rerank failed', m, repr(e))
print('ICE rerank stats', ice_stats, f'{time.time()-T0:.0f}s')
print('GL rerank stats', gl_stats, f'{time.time()-T0:.0f}s')


In [ ]:
'''## 8. Gated PubChem merge

PubChem-only candidates enter the main list only when there is no strong library match (`lib_max < 0.9`). When the raw PubChem fingerprint margin over the best pool candidate exceeds 600, they take slots 2, 4, 6, 8 and 10. Otherwise they take slots 4, 8, 12, 16 and 20. The leading proposal moves to rank 1 when the margin exceeds 150 or when `S > 6` and `pop_PC ≥ 5`.
'''


In [ ]:
'''gated PubChem merge (v3.6, unchanged) -> submission.csv'''
def merge(base, base_keys, pc, pc_keys, slots, n=25):
    bk = set(base_keys)
    pcs = [s for s, k in zip(pc, pc_keys) if k not in bk]
    out, bi, pj = [], 0, 0
    for pos in range(1, n + 1):
        if pos in slots and pj < len(pcs):
            out.append(pcs[pj]); pj += 1
        elif bi < len(base):
            out.append(base[bi]); bi += 1
        elif pj < len(pcs):
            out.append(pcs[pj]); pj += 1
    return out
'''---- CLAW PROMOTION RULE (b417) --------------------------------------------------'''
'''The shipped gate only INTERLEAVES PubChem-only structures into fixed slots, so the'''
'''main list always keeps rank 1. When the best PubChem proposal is a strong outlier --'''
'''S = z(f.z over the pass-2 set) + 0.25*pop, carried out of the worker -- AND is well'''
'''documented (pop >= 5), let it lead; everything else keeps its usual slot.'''
'''NOTE: S is the Z-SCORED quantity (median ~6). The raw p['pc_fz'][0] (median -1437)'''
'''must never be compared to S_TAU; that would fire on ~71% of molecules.'''
'''EVIDENCE: b416 (promotion, ICEBERG broken) = 0.417 vs base415 (ICE broken, no'''
'''promotion) 0.374 and b415c (ICE working, no promotion) 0.393. The promotion overcame'''
'''losing ICE, so its marginal value is >= +0.024 (est. ~+0.037 using the notebook's own'''
'''ICE +0.013 / GL +0.006 measurements). b417 isolates it on the WORKING-ICE base.'''
S_TAU, POP_TAU = 6.0, 5.0


def promote(base, base_keys, pc, pc_keys, slots, n=25):
    """Confident PubChem proposal leads the usual list; every other entry keeps its slot."""
    usual = merge(base, base_keys, pc, pc_keys, slots, n=n)
    if not pc:
        return usual
    top = pc[0]
    return ([top] + [x for x in usual if x != top])[:n]


'''self-test of both pure functions (this also runs in the cheap smoke pass)'''
_b, _bk = ['A', 'B', 'C'], ['ka', 'kb', 'kc']
_p, _pk = ['X', 'Y'], ['kx', 'ky']
assert merge(_b, _bk, _p, _pk, [2, 4]) == ['A', 'X', 'B', 'Y', 'C'], merge(_b, _bk, _p, _pk, [2, 4])
assert promote(_b, _bk, _p, _pk, [2, 4]) == ['X', 'A', 'B', 'Y', 'C'], promote(_b, _bk, _p, _pk, [2, 4])
assert merge(_b, _bk, _p, _pk, [2, 4])[0] == 'A', 'shipped gate must keep rank 1'
assert merge(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']
assert promote(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']
assert promote(_b, _bk, _p, _pk, [2, 4])[:1] == ['X'] and len(promote(_b, _bk, _p, _pk, [2, 4])) == 5
print('merge/promote self-test OK')

rows, stats = [], dict(untouched=0, gentle=0, aggressive=0, no_pc=0, promoted=0)
for mid in te.molecule_id.unique():
    smis, keys, lib_max = BASE.get(mid, ([], [], 0.0))
    p = PC.get(mid)
    if p is None or not p['pc']:
        stats['no_pc'] += 1; final = smis
    elif lib_max >= LIB_TAU:
        stats['untouched'] += 1; final = smis
    else:
        rel = (p['fz_top'] if p.get('fz_top') is not None else p['pc_fz'][0]) - p['best_pool_fz'] if np.isfinite(p['best_pool_fz']) else 1e9
        slots = SLOTS_AGG if rel > REL_TH else SLOTS_GENTLE
        stats['aggressive' if rel > REL_TH else 'gentle'] += 1
        S = p.get('S'); pop_top = float(p.get('top_pop') or 0.0)
        # Skomuro LB 0.403 Rule: Promote on Popularity OR Structure-Evidence Margin!
        # rel > 250 captures confident PubChem outliers that popularity alone misses
        # v49: rel promotion threshold 150.0 (seyitkaangunes/casmi26-e006-pubchem-promo-rel-150-lb-0-446)
        if (S is not None and float(S) > S_TAU and pop_top >= POP_TAU) or (rel > 150.0):
            stats['promoted'] += 1
            final = promote(smis, keys, p['pc'], p['pc_keys'], slots)
        else:
            final = merge(smis, keys, p['pc'], p['pc_keys'], slots)
    if not final: final = ['CCO']
    rows.append((mid, ';'.join(final[:25])))
print('merge stats', stats)
sub = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
samp = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))
sub = samp[['molecule_id']].merge(sub, on='molecule_id', how='left')
sub['smiles'] = sub['smiles'].fillna('CCO')
assert len(sub) == len(samp) and sub.molecule_id.is_unique and sub.smiles.notna().all()
assert sub.smiles.map(lambda s: len(s.split(';'))).max() <= 25
sub.to_csv('submission.csv', index=False)
print('CLAW STAGE HEALTH |', CLAW_ICE_HEALTH, '|', CLAW_GL_HEALTH, flush=True)
try:
    _ice_n = json.load(open('/kaggle/working/ice_work/ice_out.json.meta.json')).get('n_mols_scored', 0)
    _gl_n = json.load(open('/kaggle/working/gl_work/gl_out.json.meta.json')).get('n_mols_scored', 0)
except Exception:
    _ice_n = _gl_n = -1
print(f'CLAW STAGE SCORED | ice={_ice_n} gl={_gl_n}', flush=True)
json.dump(dict(secs=round(time.time() - T0), merge=stats, n_errors=n_err,
               ice_health=CLAW_ICE_HEALTH, gl_health=CLAW_GL_HEALTH,
               ice_n_scored=_ice_n, gl_n_scored=_gl_n,
               families=MAN['families'],
               gpu=torch.cuda.get_device_name(0) if dev == 'cuda' else None), open('run_manifest.json', 'w'), indent=1)
print('done', sub.shape, f'{time.time()-T0:.0f}s')
sub.head()


In [ ]:
'''## 9. Two-engine fusion + final ordering

The lists are combined after equivalent metric identities have been collapsed to one vote. A candidate's fusion score is `1 / (3 + r_main) + 0.6 / (3 + r_engine2)`, and the best 40 identities enter within-formula reordering by GLACIER, ICEBERG where GLACIER has no scores, or the fragment score where neither has scores. Distinct identities fill up to 25 places. The final step places the leading second-engine identity at rank 2 unless it already leads.'''


In [ ]:
ALPHA, KRR = 0.6, 3.0
CELL19_ICE = False       # b421: cell-19 ICE contribution EXPLICITLY off.
'''Reproduces BY DESIGN what b416 got by accident: when ICE errored it still wrote a'''
'''shell dict of all-None values, cell-19's `if ice:` gate passed, and ICE's column then'''
'''contributed nothing (covered count 0) -- leaving GLACIER alone. b420 (ICE real, both'''
'''applied) scored 0.402; b416 (GLACIER only) scored 0.417. This isolates that one'''
'''arithmetic difference on the healthy base.'''
from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF2
from rdkit import Chem as _Ch2
def _form2(s):
    try:
        return _CMF2(_Ch2.MolFromSmiles(s))
    except Exception:
        return s
def fuse2(v_smis, e_smis, e_keys, n=40):
    sc, smi_of = {}, {}
    for r, s in enumerate(v_smis, 1):
        k = chem.score_key(s) or s
        sc[k] = sc.get(k, 0.0) + 1.0 / (KRR + r); smi_of.setdefault(k, s)
    for r, (s, k) in enumerate(zip(e_smis, e_keys), 1):
        if not k:
            continue
        sc[k] = sc.get(k, 0.0) + ALPHA / (KRR + r); smi_of.setdefault(k, s)
    order = sorted(sc, key=lambda k: -sc[k])[:n]
    return [smi_of[k] for k in order], order, [sc[k] for k in order]

PREFUSION_LISTS = {str(r.molecule_id): str(r.smiles).split(';') for r in pd.read_csv('submission.csv').itertuples(index=False)}
FUSION_OVERFLOW = {}
if ENG:
    v4 = pd.read_csv('submission.csv')
    changed = n_ice = 0; out = []
    for mid, s in zip(v4.molecule_id, v4.smiles):
        vs = [x for x in s.split(';') if x and x != 'CCO']
        e = ENG.get(str(mid))
        if e and e['smiles']:
            fsm, fk, fsc = fuse2(vs, e['smiles'], e['keys'])
            ice = ICE_SCORES.get(str(mid), {}) if isinstance(ICE_SCORES, dict) else {}
            if ice:
                try:
                    o = ice_fuse.rerank(fsm, fk, fsc, [_form2(x) for x in fsm], (ice if CELL19_ICE else {}), lam=ICE_LAM, top_n=len(fsm))
                    _g = gl_of(mid)
                    if _g:
                        o = gl_fuse.rerank_multi(fsm, fk, fsc, [_form2(x) for x in fsm], [(ice if CELL19_ICE else {}), _g], [ICE_LAM, GL_LAM], top_n=len(fsm))
                    n_ice += int(o[:25] != list(range(min(25, len(o)))))
                    fsm = [fsm[i] for i in o]
                except Exception as ex:
                    print('fused ICE rerank failed', mid, repr(ex))
            changed += int(bool(vs) and fsm[:1] != vs[:1])
            FUSION_OVERFLOW[str(mid)] = list(fsm[25:])
            vs = fsm[:25]
        out.append((mid, ';'.join(vs[:25]) if vs else 'CCO'))
    fs = pd.DataFrame(out, columns=['molecule_id', 'smiles'])
    assert len(fs) == len(v4) and fs.molecule_id.is_unique and fs.smiles.notna().all()
    assert fs.smiles.map(lambda s: len(s.split(';'))).max() <= 25
    fs.to_csv('submission.csv', index=False)


print('V1 inference complete; applying separately audited final pass next.')


In [ ]:
'''## Final metric audit

This audit pass cannot demote any existing metric-equivalent answer: it retains the first
occurrence of every valid identity in order, then appends new identities up to 25 candidates.
Runtime assertions check this for every active molecule. This guarantee applies to the
same predictions and current metric; it does not predict the score of a fresh inference run.

Outputs: `submission.csv`, `baseline_submission.csv`, `candidate_audit.csv`,
`candidate_audit.json`, and stage health in `run_manifest.json`.
'''


In [ ]:
from functools import lru_cache
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog('rdApp.*')
_tautomer_canonicalizer = rdMolStandardize.TautomerEnumerator()
def canonicalize_smiles_to_ik14(smiles: str) -> str:
    """Canonicalize SMILES to InChIKey14, or None if parsing fails.

    Stereochemistry is removed *before* the tautomer canonicalization, not left to fall out of
    the InChIKey14 truncation afterwards. Truncating to InChIKey14 discards the stereo block, so
    it looks like it should already make stereochemistry free -- but the canonicalization runs
    first, and `TautomerEnumerator` will not move a double bond whose position a neighbouring
    stereocentre constrains. The same skeleton drawn with and without that stereocentre can
    therefore land on two different canonical tautomers, and different tautomers are different
    connectivity, which is the part of the InChIKey14 that survives truncation. Stripping first
    makes the canonical form depend only on the skeleton, which is what the competition promises.

    Both the answer key and the submission go through this function, so the two sides can never
    be treated asymmetrically.
    """
    if not isinstance(smiles, str):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        Chem.RemoveStereochemistry(mol)
        canonical_mol = _tautomer_canonicalizer.Canonicalize(mol)
        inchikey = Chem.MolToInchiKey(canonical_mol)
        return inchikey[:14]
    # RDKit's MolSanitizeException family subclasses ValueError; RuntimeError covers the C++
    # layer and TypeError covers Boost.Python argument errors. One participant's malformed
    # SMILES must never take the whole scoring run down, so a failure here is just a non-match.
    except (ValueError, RuntimeError, TypeError):
        return None

def heavy_atom_composition(smiles: str):
    """Heavy-atom element counts (excluding H, ignoring charge) as a hashable key, or None
    if parsing fails.

    Cheap (no tautomer enumeration) -- used to gate the expensive canonicalization. Anything
    that shares an InChIKey14 with an accepted structure has the same heavy-atom skeleton, so
    a heavy-atom-composition mismatch lets us skip canonicalization with no change to any
    score. We deliberately do NOT gate on full molecular formula: protomers / charge variants
    (e.g. a zwitterion C(=O)[O-]...[N+] vs its neutral-acid protomer C(=O)O...[N+]) share an
    InChIKey14 but differ in H count and charge, so a formula gate would wrongly reject them.
    """
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    if mol is None:
        return None
    counts = {}
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        if sym != 'H':
            counts[sym] = counts.get(sym, 0) + 1
    return tuple(sorted(counts.items()))

@lru_cache(maxsize=150000)
def metric_identity(smiles):
    key = canonicalize_smiles_to_ik14(smiles)
    return (key, heavy_atom_composition(smiles)) if key else None

def safe_extend(original, extra, limit=25):
    """Preserve first valid metric identities; fill only unoccupied ranks."""
    out, seen = [], set()
    for smi in list(original) + list(extra):
        smi = str(smi).strip()
        identity = metric_identity(smi)
        if identity is not None and identity not in seen:
            seen.add(identity)
            out.append(smi)
            if len(out) == limit:
                break
    return out

def first_ranks(smiles):
    ranks = {}
    for rank, smi in enumerate(smiles, 1):
        identity = metric_identity(smi)
        if identity is not None:
            ranks.setdefault(identity, rank)
    return ranks

def assert_monotone(original, revised):
    old, new = first_ranks(original), first_ranks(revised)
    assert all(identity in new and new[identity] <= rank for identity, rank in old.items())


In [ ]:
METRIC_SOURCE_SHA256 = 'c61d163f1bb768313fd0863822952f035f343924f37e3f5e4b5883ccef7e6201'
'''Final pass: no change to any V1 model, fusion weight, promotion gate or simulator.'''
import collections
import hashlib
from pathlib import Path
assert rdkit.__version__ == '2026.03.3'
assert n_err == 0, f'Base inference errors: {n_err}'
_before = pd.read_csv('submission.csv')
_before.to_csv('baseline_submission.csv', index=False)
_active = set(te.molecule_id.astype(str))
assert _active <= set(map(str, ENG)), 'Engine 2 did not cover all active molecules'
_stage_meta = {}
for _stage, _path in [('ice', 'ice_work/ice_out.json.meta.json'), ('glacier', 'gl_work/gl_out.json.meta.json')]:
    if Path(_path).is_file():
        _stage_meta[_stage] = json.loads(Path(_path).read_text())
    else:
        _stage_meta[_stage] = {'status': 'missing'}
assert _stage_meta['ice'].get('status') == 'ok', _stage_meta['ice']
if (te.adduct == '[M+H]+').any():
    assert _stage_meta['glacier'].get('status') == 'ok', _stage_meta['glacier']

_dm_smiles = _dm_mass = _dm_order = None
_dm_paths = sorted(glob.glob('/kaggle/input/**/dreams_np_mass.npy', recursive=True))
_dreams_used = 0
_audit_rows, _out = [], []
_targets = te.groupby('molecule_id').nm.median().to_dict()
for _row in _before.itertuples(index=False):
    _mid = str(_row.molecule_id)
    _original = [s.strip() for s in str(_row.smiles).split(';') if s.strip()]
    if _mid not in _active:  # Visible smoke placeholders; hidden rerun processes every molecule.
        _out.append((_row.molecule_id, _row.smiles))
        continue
    _ranks = first_ranks(_original)
    _clean = safe_extend(_original, [])
    _extras = list(FUSION_OVERFLOW.get(_mid, [])) + list(BASE_TAIL_CANDIDATES.get(_mid, []))
    _engine = ENG.get(_mid, {})
    _extras.extend(_engine.get('smiles', []))
    _pc = PC.get(_row.molecule_id, PC.get(_mid, {}))
    _extras.extend(_pc.get('pc', []))
    _revised = safe_extend(_clean, _extras)
    _after_ranked = len(_revised)
    _target = float(_targets.get(_row.molecule_id, _targets.get(_mid, np.nan)))
    if len(_revised) < 25 and np.isfinite(_target) and _target > 0 and _dm_paths:
        if _dm_mass is None:
            _dd = Path(_dm_paths[0]).parent
            _dm_mass = np.load(_dd / 'dreams_np_mass.npy').astype(np.float64)
            _dm_smiles = np.load(_dd / 'dreams_np_smiles.npy', allow_pickle=True)
            assert len(_dm_mass) == len(_dm_smiles)
            _dm_order = np.argsort(_dm_mass, kind='stable')
            _dm_sorted = _dm_mass[_dm_order]
        _lo, _hi = np.searchsorted(_dm_sorted, [_target * (1 - 10e-6), _target * (1 + 10e-6)])
        _ix = _dm_order[_lo:_hi]
        _ix = _ix[np.argsort(np.abs(_dm_mass[_ix] - _target), kind='stable')]
        _mass_candidates = [s.decode() if isinstance(s, bytes) else str(s) for s in _dm_smiles[_ix]]
        _revised = safe_extend(_revised, _mass_candidates)
        _dreams_used += len(_revised) - _after_ranked
    assert _revised, f'No valid candidates for {_mid}'
    assert len(_revised) <= 25 and len(first_ranks(_revised)) == len(_revised)
    assert_monotone(_original, _revised)
    _new_ranks = first_ranks(_revised)
    _audit_rows.append(dict(molecule_id=_mid, original_count=len(_original),
        original_unique=len(_ranks), invalid_count=sum(metric_identity(s) is None for s in _original),
        duplicate_count=sum(metric_identity(s) is not None for s in _original) - len(_ranks),
        revised_count=len(_revised), ranked_added=_after_ranked - len(_clean),
        dreams_added=len(_revised) - _after_ranked,
        promoted_original=sum(_new_ranks[k] < v for k, v in _ranks.items()),
        changed=_original != _revised))
    _out.append((_row.molecule_id, ';'.join(_revised)))
_final = pd.DataFrame(_out, columns=['molecule_id', 'smiles'])
_expected = pd.read_csv(os.path.join(COMP, 'sample_submission.csv')).molecule_id
assert _final.molecule_id.is_unique and set(_final.molecule_id) == set(_expected)
assert _final.smiles.notna().all() and _final.smiles.str.split(';').map(len).between(1, 25).all()
if IS_RERUN:
    assert len(_active) == len(_final), 'A hidden molecule was left unprocessed'
_final.to_csv('submission.csv', index=False)
_audit_df = pd.DataFrame(_audit_rows)
_audit_df.to_csv('candidate_audit.csv', index=False)
_summary = dict(experiment='V1.1 metric-safe tail', rdkit=rdkit.__version__,
    hidden_rerun=IS_RERUN, active_molecules=len(_active), output_rows=len(_final),
    changed_molecules=int(_audit_df.changed.sum()),
    invalid_removed=int(_audit_df.invalid_count.sum()),
    equivalent_duplicates_removed=int(_audit_df.duplicate_count.sum()),
    existing_candidates_promoted=int(_audit_df.promoted_original.sum()),
    ranked_candidates_added=int(_audit_df.ranked_added.sum()),
    dreams_candidates_added=int(_audit_df.dreams_added.sum()),
    monotone_rank_assertions_passed=True,
    metric_source_sha256=METRIC_SOURCE_SHA256,
    baseline_sha256=hashlib.sha256(Path('baseline_submission.csv').read_bytes()).hexdigest(),
    submission_sha256=hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest(),
    stages=_stage_meta, runtime_seconds=round(time.time()-T0, 1),
    validation_note='Visible test is a smoke check only and overlaps training; no generalization score claimed.')
Path('candidate_audit.json').write_text(json.dumps(_summary, indent=2, default=str))
print(json.dumps({k:v for k,v in _summary.items() if k != 'stages'}, indent=2))
display(_audit_df)


In [ ]:
'''Independent experimental overlay
One vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.'''

def metric_fusion(first, second, identity, alpha=0.6, offset=3.0, limit=40):
    scores, representatives = {}, {}
    collapsed = 0
    for candidates, weight in ((first, 1.0), (second, alpha)):
        seen = set()
        for candidate in candidates:
            key = identity(candidate)
            if key is None:
                continue
            if key in seen:
                collapsed += 1
                continue
            seen.add(key)
            scores[key] = scores.get(key, 0.0) + weight / (offset + len(seen))
            representatives.setdefault(key, candidate)
    ordered = sorted(scores, key=lambda key: -scores[key])[:limit]
    return [representatives[key] for key in ordered], [scores[key] for key in ordered], collapsed

EXPERIMENT = 'metric_identity_fusion'
_reference = pd.read_csv('submission.csv')
_reference.to_csv('submission_reference.csv', index=False)
_rows, _audit = [], []
for _row in _reference.itertuples(index=False):
    _mid = str(_row.molecule_id)
    _original = str(_row.smiles).split(';')
    _revised, _collapsed = list(_original), 0
    _engine = ENG.get(_mid)
    if _mid in PREFUSION_LISTS and _engine and _engine.get('smiles'):
        _first = [s for s in PREFUSION_LISTS[_mid] if s and s != 'CCO']
        _candidates, _scores, _collapsed = metric_fusion(_first, _engine['smiles'], metric_identity,
                                                       alpha=ALPHA, offset=KRR)
        _keys = [chem.score_key(s) or s for s in _candidates]
        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}
        _gl = gl_of(_mid)
        if _ice and _gl:
            _order = gl_fuse.rerank_multi(_candidates, _keys, _scores,
                [_form2(s) for s in _candidates], [{}, _gl], [ICE_LAM, GL_LAM], top_n=len(_candidates))
            _candidates = [_candidates[i] for i in _order]
        if _candidates:
            _revised = safe_extend(_candidates[:25], _candidates[25:] + _original)
    _rows.append((_row.molecule_id, ';'.join(_revised)))
    _audit.append(dict(molecule_id=_mid, collapsed_source_votes=_collapsed,
        changed=_revised != _original, changed_top1=_revised[:1] != _original[:1]))

_result = pd.DataFrame(_rows, columns=['molecule_id', 'smiles'])
assert list(_result.molecule_id) == list(_reference.molecule_id)
assert _result.molecule_id.is_unique and _result.smiles.notna().all()
assert _result.smiles.map(lambda value: 1 <= len(value.split(';')) <= 25).all()
_result.to_csv('submission.csv', index=False)
pd.DataFrame(_audit).to_csv('experiment_audit.csv', index=False)
_changed = sum(row['changed'] for row in _audit)
_report = dict(experiment=EXPERIMENT, changed_molecules=_changed,
               changed_top1=sum(row['changed_top1'] for row in _audit),
               reference_sha256=sha256('submission_reference.csv'),
               submission_sha256=sha256('submission.csv'), measured_score=None)
json.dump(_report, open('experiment_audit.json', 'w'), indent=2)
print(_report)
if not _changed:
    print('NO-OP: predictions match the reference; do not spend a second submission on this file.')


In [ ]:
"""Observe the frozen source output and preserve same-run fusion inputs for later work."""


def _e004_postflight(scope):
    import hashlib
    import json
    import time
    from pathlib import Path
    import pandas as pd
    import rdkit

    pre = scope['E004_PREFLIGHT']
    ids = set(pre['input_ids'])
    processed = set(scope['te'].molecule_id.astype(str))
    full = bool(scope['IS_RERUN'])
    assert processed == (ids if full else set(sorted(ids)[:scope['SMOKE_N']]))
    frame = pd.read_csv('submission.csv', dtype=str, keep_default_na=False)
    assert list(frame.columns) == ['molecule_id', 'smiles']
    assert frame.molecule_id.is_unique and set(frame.molecule_id) == ids
    assert frame.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()
    base, eng, pc = scope['BASE'], scope['ENG'], scope['PC']
    assert set(map(str, base)) == processed
    assert processed <= set(map(str, eng))
    assert scope['n_err'] == 0
    if not full:
        assert all(v[0] for v in base.values())
        assert all(eng[mid]['smiles'] and pc.get(mid, {}).get('pc') for mid in processed)
    stages = {}
    for stage in ('ice', 'gl'):
        path = Path(f'{stage}_work/{stage}_out.json.meta.json')
        meta = json.loads(path.read_text()) if path.is_file() else {'status': 'missing'}
        # The frozen source owns full-run failure policy. Extra acceptance checks
        # only gate the visible smoke, where all expected stages are exercised.
        if not full:
            assert meta['status'] == 'ok' and meta['device'].startswith('cuda'), (stage, meta)
            assert meta['rdkit'] == '2025.03.6'
            assert not meta.get('errors') and meta.get('n_pred_fail', 0) == 0
            assert meta.get('n_feat_fail', 0) == 0
            if meta.get('n_mols_covered', 0):
                assert meta['n_mols_scored'] > 0 and meta['n_scored'] > 0
        stages[stage] = meta
    payload = {
        'prefusion': {mid: scope['PREFUSION_LISTS'][mid] for mid in sorted(processed)},
        'engine2': {mid: eng[mid] for mid in sorted(processed)},
        'pubchem': {mid: pc.get(mid, {}) for mid in sorted(processed)},
        'base_tail': {mid: scope['BASE_TAIL_CANDIDATES'][mid] for mid in sorted(processed)},
        'fusion_overflow': {mid: scope['FUSION_OVERFLOW'].get(mid, []) for mid in sorted(processed)},
    }
    Path('e004_candidate_inputs.json').write_text(json.dumps(payload, indent=2, default=str) + '\n')
    hashes = {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in (
        'submission.csv', 'baseline_submission.csv', 'submission_reference.csv',
        'candidate_audit.json', 'experiment_audit.json', 'e004_candidate_inputs.json')}
    report = dict(experiment='E004', source_version_id=356740494, source_version=80,
                  source_code_sha256='eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2',
                  **pre, is_full_run=full, processed_ids=sorted(processed),
                  processed_molecules=len(processed), input_molecules=len(ids),
                  output_rows=len(frame), placeholder_fill_rows=len(ids - processed),
                  base_errors=int(scope['n_err']), base_nonempty=sum(bool(v[0]) for v in base.values()),
                  engine2_nonempty=sum(bool(eng[mid]['smiles']) for mid in processed),
                  pubchem_nonempty=sum(bool(pc.get(mid, {}).get('pc')) for mid in processed),
                  main_rdkit=rdkit.__version__, stages=stages, hashes=hashes,
                  elapsed_seconds=round(time.time() - scope['T0'], 2),
                  limitations=['Visible smoke is not independent validation.',
                               'Author 0.436 is not this reproduction score.',
                               'Whole v80 recipe differs from E003 in multiple factors.'])
    Path('e004_verification.json').write_text(json.dumps(report, indent=2) + '\n')
    print('E004_VERIFICATION', json.dumps(report))
    return report


E004_REPORT = _e004_postflight(globals())
del _e004_postflight


In [ ]:
# Fragment score for molecules without forward-model scores (since v46), copied verbatim from
# ahmedberatozer/casmi26-v7c-e006-fraggap version 1; its frag_rescore.py is attributed there to
# Huseyin Emre Aksoy (Apache-2.0).
# v7c: MetFrag-lite fragment evidence for molecules with NO forward-model coverage (no GLACIER, no finite ICEBERG)
FRAG_GAP = {}
try:
    FRAG_SRC = '"""Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\'s <=2-bond-cleavage fragments explain.\n\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\nremember the fewest cleavages that give each mass, then match every peak against\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\nA peak\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\nand a spectrum\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\n\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\n\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\'s periodic table\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\nE = 0.00054857990\nPROTON = H - E\nH2O = 2 * H + AMU[\'O\']\nNH4 = AMU[\'N\'] + 4 * H\nFORMATE = AMU[\'C\'] + 2 * H + 2 * AMU[\'O\']\n\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\nCARRIERS = {\n    \'[M+H]+\': (PROTON,), \'[M-H2O+H]+\': (PROTON,), \'[M-2H2O+H]+\': (PROTON,),\n    \'[M+NH4]+\': (PROTON, NH4 - E), \'[M+Na]+\': (PROTON, AMU[\'Na\'] - E), \'[M+K]+\': (PROTON, AMU[\'K\'] - E),\n    \'[M-H]-\': (-PROTON,), \'[M-H2O-H]-\': (-PROTON,), \'[M+CH2O2-H]-\': (-PROTON, FORMATE - PROTON),\n    \'[M+Cl]-\': (-PROTON, AMU[\'Cl\'] + E),\n}\n\nDEFAULT_CFG = dict(\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\n    intensity=\'linear\',          # \'linear\' | \'sqrt\' | a float power applied to `it`\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\n    tol=0.005,                   # absolute m/z tolerance, Da\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\n    max_breaks=2,                # bond cleavages per fragment (1..3)\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\n)\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\'sqrt\', tol=0.01, h_shifts=tuple(range(-2, 3)))\n\n\ndef resolve_cfg(cfg=None):\n    """DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently)."""\n    out = dict(DEFAULT_CFG)\n    for k, v in (cfg or {}).items():\n        if k not in out:\n            raise KeyError(f\'frag_rescore: unknown cfg key {k!r}\')\n        out[k] = v\n    out[\'h_shifts\'] = tuple(int(d) for d in out[\'h_shifts\'])\n    if not 1 <= int(out[\'max_breaks\']) <= 3:\n        raise ValueError(\'frag_rescore: max_breaks must be 1, 2 or 3\')\n    return out\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\n# ---------------------------------------------------------------------------------------------------------------\ndef _find(parent, a):\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\n    """Union the kept bonds; if the graph splits, append each piece\'s mass with cleavage count k."""\n    for a in range(n):\n        parent[a] = a\n        acc[a] = 0.0\n    ncomp = n\n    for e in range(ea.shape[0]):\n        if removed[e]:\n            continue\n        ra = _find(parent, ea[e])\n        rb = _find(parent, eb[e])\n        if ra != rb:\n            parent[ra] = rb\n            ncomp -= 1\n    if ncomp < 2:\n        return cnt\n    for a in range(n):\n        r = _find(parent, a)\n        acc[r] += w[a]\n    for a in range(n):\n        if parent[a] == a:\n            out_m[cnt] = acc[a]\n            out_k[cnt] = k\n            cnt += 1\n    return cnt\n\n\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\n    """All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces."""\n    nb = ea.shape[0]\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\n    out_m = np.empty(cap, np.float64)\n    out_k = np.empty(cap, np.int8)\n    parent = np.empty(n, np.int64)\n    acc = np.empty(n, np.float64)\n    removed = np.zeros(nb, np.bool_)\n    cnt = 0\n    for i in range(nb):\n        removed[i] = True\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\n        if max_breaks >= 2:\n            for j in range(i + 1, nb):\n                removed[j] = True\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\n                if max_breaks >= 3:\n                    for l in range(j + 1, nb):\n                        removed[l] = True\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\n                        removed[l] = False\n                removed[j] = False\n        removed[i] = False\n    return out_m[:cnt], out_k[:cnt]\n\n\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\n    from numba import njit\n    _find = njit(cache=True, nogil=True)(_find)\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\n    HAVE_NUMBA = True\nexcept Exception:  # pragma: no cover - fallback path\n    HAVE_NUMBA = False\n\n\ndef mol_graph(smi):\n    """(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    try:\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n    except Exception:\n        m = None\n    if m is None or m.GetNumAtoms() == 0:\n        return None\n    pt = Chem.GetPeriodicTable()\n    w = np.empty(m.GetNumAtoms(), np.float64)\n    for a in m.GetAtoms():\n        s = a.GetSymbol()\n        am = AMU.get(s)\n        if am is None:\n            am = pt.GetMostCommonIsotopeMass(s)\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\n    return w, ea, eb, len(Chem.GetMolFrags(m))\n\n\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\n    """Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\n\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\n    SMILES does not parse.\n    """\n    g = mol_graph(smi)\n    if g is None:\n        return None\n    w, ea, eb, npieces = g\n    nb = len(ea)\n    mb = int(max_breaks)\n    if mb >= 3 and nb > max_bonds_3:\n        mb = 2\n    if mb >= 2 and nb > max_bonds_2:\n        mb = 1\n    levels = [np.array([w.sum()])]\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smi)\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\n    if nb == 0:\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\n    fm = np.concatenate([levels[0], fm])\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\n    key = np.round(fm, 5)\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\n    key, fm, fk = key[order], fm[order], fk[order]\n    first = np.ones(len(key), bool)\n    first[1:] = key[1:] != key[:-1]\n    fm, fk = fm[first], fk[first]\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# scoring\n# ---------------------------------------------------------------------------------------------------------------\ndef _carriers(adduct, mode):\n    c = CARRIERS.get(adduct)\n    if c is not None:\n        return c\n    if isinstance(adduct, str) and adduct.endswith(\'-\'):\n        return (-PROTON,)\n    if isinstance(adduct, str) and adduct.endswith(\'+\'):\n        return (PROTON,)\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\n\n\ndef _prep_spectrum(s, c):\n    """Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    mz, it = mz[ok], it[ok]\n    if len(mz) == 0:\n        return None\n    p = c[\'intensity\']\n    iw = it if p == \'linear\' else np.sqrt(it) if p == \'sqrt\' else it ** float(p)\n    if c[\'mz_power\']:\n        prec = float(s.get(\'prec\') or mz.max())\n        iw = iw * (mz / prec) ** float(c[\'mz_power\'])\n    tot = iw.sum()\n    if not tot > 0:\n        return None\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\'adduct\'), s.get(\'mode\')) for dh in c[\'h_shifts\']])\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\n    return iw / tot, q, len(shifts), len(mz)\n\n\ndef _score_spectrum(levels, weights, prep, tol):\n    iw, q, ns, npk = prep\n    best = np.zeros(npk)\n    for arr, wt in zip(levels, weights):\n        if len(arr) == 0 or wt <= 0:\n            continue\n        a = np.searchsorted(arr, q - tol, \'left\')\n        b = np.searchsorted(arr, q + tol, \'right\')\n        hit = (b > a).reshape(ns, npk).any(axis=0)\n        best = np.where(hit & (wt > best), wt, best)\n    return float(iw @ best)\n\n\ndef candidate_levels(smi, cfg=None, cache=None):\n    """Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter)."""\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\n\n\ndef _levels_cached(smi, c, cache):\n    key = (smi, int(c[\'max_breaks\']), int(c[\'max_bonds_2\']), int(c[\'max_bonds_3\']))\n    if cache is not None and key in cache:\n        return cache[key]\n    try:\n        lv = fragment_levels(smi, c[\'max_breaks\'], c[\'max_bonds_2\'], c[\'max_bonds_3\'])\n    except Exception:\n        lv = None\n    if cache is not None:\n        cache[key] = lv\n    return lv\n\n\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\n    """One score per candidate: mean explained-intensity share over the molecule\'s spectra (None: unparsable)."""\n    c = resolve_cfg(cfg)\n    cache = {} if cache is None else cache\n    weights = [1.0, float(c[\'w1\']), float(c[\'w2\']), float(c[\'w3\'])][: int(c[\'max_breaks\']) + 1]\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\n    tol = float(c[\'tol\'])\n    out = []\n    for smi in cand_smiles:\n        lv = _levels_cached(smi, c, cache)\n        if lv is None:\n            out.append(None)\n            continue\n        if not preps:\n            out.append(0.0)\n            continue\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\n    return out\n\n\ndef warmup():\n    """Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it."""\n    fragment_levels(\'CCOC(=O)c1ccccc1O\')\n    return HAVE_NUMBA\n'
    open('/kaggle/working/frag_rescore.py', 'w').write(FRAG_SRC)
    if '/kaggle/working' not in sys.path:
        sys.path.insert(0, '/kaggle/working')
    import frag_rescore
    frag_rescore.warmup()
    _fcache = {}
    _by = {str(m): g for m, g in te.groupby('molecule_id', sort=False)}
    _t0 = time.time(); _nf = 0
    for _mid, _lst in PREFUSION_LISTS.items():
        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}
        if gl_of(_mid) or any(v is not None for v in (_ice or {}).values()):
            continue
        _e = ENG.get(_mid) or {}
        _c = [s for s in dict.fromkeys(list(_lst) + list(_e.get('smiles', []))) if s and s != 'CCO']
        _g = _by.get(_mid)
        if _g is None or len(_c) < 2:
            continue
        _spec = [dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),
                      adduct=r.adduct, prec=float(r.precursor_mz), mode=1 if r.ionization_mode == 'positive' else -1)
                 for r in _g.itertuples()][:16]
        try:
            _sc = frag_rescore.score_molecule(_spec, _c, cache=_fcache)
            FRAG_GAP[_mid] = {s: float(v) for s, v in zip(_c, _sc) if v is not None}
            _nf += 1
        except Exception as _ex:
            print('frag failed', _mid, repr(_ex))
    print('v7c FRAG gap molecules', _nf, 'of', len(PREFUSION_LISTS), f'{time.time()-_t0:.0f}s', flush=True)
except Exception as _e:
    FRAG_GAP = {}
    print('v7c FRAG FAILED -> v7a behaviour:', repr(_e), flush=True)


In [ ]:
def _factorial_entry(scope):
    """Append-only E005–E007 overlay; original v80 functions remain authoritative.

    Embedded function-locally after the complete frozen E004 notebook. This module
    never installs dependencies, changes model scores, or recanonicalizes SMILES.
    """

    import hashlib
    import json
    import math
    from pathlib import Path


    def _finite_coverage(scores, candidates, keys):
        count = 0
        for smi, key in zip(candidates, keys):
            value = scores.get(smi)
            if value is None:
                value = scores.get(key)
            try:
                count += value is not None and math.isfinite(float(value))
            except (TypeError, ValueError):
                pass
        return count


    def rerank_row(scope, mid, original, *, window, ice_fallback, processed):
        """Rebuild final metric fusion from same-run prefusion and engine lists.

        ``original`` is the metric-safe reference BEFORE the frozen final overlay,
        not the E004 final output. The original identity, representative selection,
        same-formula slot rerank and tail fill are reused without modification.
        """
        if window not in (40, 80):
            raise ValueError("Only the prespecified 40/80 candidate windows are supported")
        original = list(original)
        audit = {
            "molecule_id": mid,
            "processed": bool(processed),
            "branch": "placeholder",
            "candidate_count": 0,
            "collapsed_source_votes": 0,
            "ice_finite_count": 0,
            "gl_finite_count": 0,
            "fallback_changed": False,
            "fallback_changed_top1": False,
        }
        if not processed:
            return original, audit
        audit["branch"] = "no_engine"
        engine = scope["ENG"].get(mid)
        if mid not in scope["PREFUSION_LISTS"] or not engine or not engine.get("smiles"):
            return original, audit
        first = [s for s in scope["PREFUSION_LISTS"][mid] if s and s != "CCO"]
        candidates, scores, collapsed = scope["metric_fusion"](
            first,
            engine["smiles"],
            scope["metric_identity"],
            alpha=scope["ALPHA"],
            offset=scope["KRR"],
            limit=window,
        )
        keys = [scope["chem"].score_key(s) or s for s in candidates]
        ice_scores = scope["ICE_SCORES"]
        ice = ice_scores.get(mid, {}) if isinstance(ice_scores, dict) else {}
        gl = scope["gl_of"](mid)
        audit.update(
            branch="unrescored",
            candidate_count=len(candidates),
            collapsed_source_votes=collapsed,
            ice_finite_count=_finite_coverage(ice, candidates, keys),
            gl_finite_count=_finite_coverage(gl, candidates, keys),
        )
        before = list(candidates)
        if ice and gl:
            # Preserve v80's exact GL-present gate and GL-only score arguments.
            order = scope["gl_fuse"].rerank_multi(
                candidates,
                keys,
                scores,
                [scope["_form2"](s) for s in candidates],
                [{}, gl],
                [scope["ICE_LAM"], scope["GL_LAM"]],
                top_n=len(candidates),
            )
            candidates = [candidates[i] for i in order]
            audit["branch"] = "glacier"
        elif (ice_fallback and not gl and scope.get("FRAG_GAP", {}).get(mid)
              and not any(v is not None for v in (ice or {}).values())):
            # v7c: no forward-model evidence at all -> MetFrag-lite gap re-rank (lambda 0.5), once
            order = scope["gl_fuse"].rerank_multi(
                candidates, keys, scores, [scope["_form2"](s) for s in candidates],
                [scope["FRAG_GAP"][mid]], [0.5], top_n=len(candidates))
            candidates = [candidates[i] for i in order]
            audit["branch"] = "frag_gap"
        elif ice_fallback and ice and not gl:
            # Public v1 / script 356785434: ICE is used only where gl_of(mid) is empty.
            # Keep its truthy-dict gate: all-None ICE is an observable harmless no-op.
            order = scope["ice_fuse"].rerank(
                candidates,
                keys,
                scores,
                [scope["_form2"](s) for s in candidates],
                ice,
                lam=scope["ICE_LAM"],
                top_n=len(candidates),
            )
            candidates = [candidates[i] for i in order]
            audit["branch"] = "ice_fallback"
        result = (
            scope["safe_extend"](candidates[:25], candidates[25:] + original)
            if candidates
            else original
        )
        if audit["branch"] == "ice_fallback":
            without_ice = (
                scope["safe_extend"](before[:25], before[25:] + original) if before else original
            )
            audit["fallback_changed"] = result != without_ice
            audit["fallback_changed_top1"] = result[:1] != without_ice[:1]
        return result, audit


    def run_overlay(scope, *, experiment, window, ice_fallback, provenance, workdir="."):
        """Save the E004 bytes, validate same-run control, then emit one treatment."""
        import pandas as pd

        root = Path(workdir)
        path = root / "submission.csv"
        control_bytes = path.read_bytes()
        control_sha = hashlib.sha256(control_bytes).hexdigest()
        base_report = json.loads((root / "e004_verification.json").read_text())
        assert base_report["experiment"] == "E004"
        assert base_report["hashes"]["submission.csv"] == control_sha
        # Copy bytes directly; never reserialize or overwrite original E004 audit files.
        (root / "submission_e004_control.csv").write_bytes(control_bytes)
        control = pd.read_csv(path, dtype=str, keep_default_na=False)
        reference = pd.read_csv(root / "submission_reference.csv", dtype=str, keep_default_na=False)
        assert list(reference.columns) == list(control.columns) == ["molecule_id", "smiles"]
        assert list(reference.molecule_id) == list(control.molecule_id)
        assert control.molecule_id.is_unique
        processed = set(base_report["processed_ids"])
        assert processed == set(scope["te"].molecule_id.astype(str))
        controls = dict(zip(control.molecule_id, control.smiles))
        output, audits = [], []
        for row in reference.itertuples(index=False):
            mid, original = str(row.molecule_id), str(row.smiles).split(";")
            active = mid in processed
            # The exact original pre-overlay tail is essential for paired equivalence.
            rebuilt, _ = rerank_row(
                scope, mid, original, window=40, ice_fallback=False, processed=active
            )
            assert ";".join(rebuilt) == controls[mid], f"E004 control reconstruction mismatch: {mid}"
            result, audit = rerank_row(
                scope, mid, original, window=window, ice_fallback=ice_fallback, processed=active
            )
            assert 1 <= len(result) <= 25
            if active:
                identities = [scope["metric_identity"](s) for s in result]
                assert None not in identities and len(identities) == len(set(identities)), mid
            else:
                assert ";".join(result) == controls[mid], f"Placeholder changed: {mid}"
            old = controls[mid].split(";")
            audit.update(changed=result != old, changed_top1=result[:1] != old[:1])
            audits.append(audit)
            output.append((mid, ";".join(result)))
        frame = pd.DataFrame(output, columns=["molecule_id", "smiles"])
        assert list(frame.molecule_id) == list(control.molecule_id)
        frame.to_csv(path, index=False)
        report = {
            "experiment": experiment,
            "window": window,
            "ice_fallback": bool(ice_fallback),
            "source_experiment": "E004",
            "base_report_file": "e004_verification.json",
            "control_file": "submission_e004_control.csv",
            "control_sha256": control_sha,
            "submission_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "processed_ids": sorted(processed),
            "output_rows": len(frame),
            "changed_rows": sum(a["changed"] for a in audits),
            "changed_top1": sum(a["changed_top1"] for a in audits),
            "fallback_rows": sum(a["branch"] == "ice_fallback" for a in audits),
            "fallback_finite_rows": sum(
                a["branch"] == "ice_fallback" and a["ice_finite_count"] > 0 for a in audits
            ),
            "fallback_changed_rows": sum(a["fallback_changed"] for a in audits),
            "fallback_changed_top1": sum(a["fallback_changed_top1"] for a in audits),
            "same_run_control_reconstructed": True,
            "provenance": provenance,
            "limitations": [
                "Visible smoke is not independent validation.",
                "An inactive fallback in visible smoke says nothing about hidden benefit.",
            ],
        }
        pd.DataFrame(audits).to_csv(root / "experiment_audit_factorial.csv", index=False)
        (root / "experiment_audit_factorial.json").write_text(
            json.dumps(dict(report, rows=audits), indent=2, allow_nan=False) + "\n"
        )
        (root / "experiment_verification.json").write_text(
            json.dumps(report, indent=2, allow_nan=False) + "\n"
        )
        print("FACTORIAL_VERIFICATION", json.dumps(report))
        return report

    return run_overlay(scope, **{'experiment': 'E006', 'window': 40, 'ice_fallback': True, 'provenance': {'base_notebook_sha256': '1f0f9b3026a93ddb5ad6fa6993c8ac0a4bfaddc54594eb1dd6dda7a20d8ef0d6', 'base_source_code_sha256': 'eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2', 'helper_source_sha256': 'a027a69ddda2c2e14639e3ec5196d5f1eb2ab57cba10ef929723148f31fc8f34', 'source_script_version_id': 356740494, 'ice_policy_source_script_version_id': 356785434, 'ice_policy_source': 'samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438/1'}})

FACTORIAL_REPORT = _factorial_entry(globals())
del _factorial_entry


In [ ]:
# Rank-2 placement of the second engine's leading structure (since v48), after
# antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert (MODE 'rank2'): the first structure of the second-engine
# list moves to rank 2 unless it already holds rank 1; keys = E006 metric identity; errors keep E006's file unchanged.
def _rank2_overlay(scope):
    import hashlib, json, os
    from pathlib import Path
    import pandas as pd
    path = Path('submission.csv')
    control = path.read_bytes()
    report = dict(control_sha256=hashlib.sha256(control).hexdigest(), rows=0, inserted=0, status='fallback')
    try:
        identity = scope['metric_identity']
        engine = scope['ENG'] if isinstance(scope.get('ENG'), dict) else {}
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
        rows = []
        for mid, answer in frame.itertuples(index=False):
            report['rows'] += 1
            toks = [t for t in str(answer).split(';') if t]
            listed = [s for s in ((engine.get(str(mid)) or {}).get('smiles') or []) if s and s != 'CCO']
            top = listed[0] if listed else None
            key = identity(top) if top else None
            if toks and toks[0] != 'CCO' and key is not None:
                keys = [identity(t) for t in toks]
                if key != keys[0]:
                    rest = [t for t, k in zip(toks, keys) if k != key]
                    toks = (rest[:1] + [top] + rest[1:])[:25]
                    report['inserted'] += 1
            rows.append((mid, ';'.join(toks) if toks else answer))
        out = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
        assert out.molecule_id.tolist() == frame.molecule_id.tolist() and out.smiles.notna().all()
        assert out.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()
        for s in out.smiles:
            ids = [identity(x) for x in s.split(';')]
            assert s == 'CCO' or (None not in ids and len(ids) == len(set(ids)))
        Path('submission_e006_control.csv').write_bytes(control)
        out.to_csv('submission_stage.csv', index=False)
        os.replace('submission_stage.csv', path)
        report['status'] = 'ok'
    except Exception as exc:
        path.write_bytes(control)
        report.update(status='fallback', error=f'{type(exc).__name__}: {exc}'[:400], inserted=0)
    report['submission_sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
    Path('rank2_overlay.json').write_text(json.dumps(report, indent=2))
    print('RANK2_OVERLAY', json.dumps(report), flush=True)
    return report


RANK2_REPORT = _rank2_overlay(globals())
del _rank2_overlay


In [ ]:
V43_DATASET_SOURCES = ['dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'dmitriigluzdov/casmi26-natural-product-panel-benchmark', 'dmitriigluzdov/casmi26-natural-product-knowledge-table', 'dmitriigluzdov/casmi26-fold-safe-fpnet']
V43_E006_SHA256 = 'ee55b55c7f5bdb08f1a8154396b18fdb0203fb7178b3a52429e0999e69ad7e53'
# Read the final E006 file; validate it without changing its bytes or candidate order.
def _v43_receipt(scope):
    import hashlib
    import json
    import os
    import time
    from pathlib import Path
    import pandas as pd
    import rdkit

    def digest(path):
        path = Path(path)
        h = hashlib.sha256()
        with path.open('rb') as source:
            for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
                h.update(block)
        return dict(path=str(path), bytes=path.stat().st_size, sha256=h.hexdigest())

    final_path = Path('submission.csv')
    final_before = digest(final_path)
    frame = pd.read_csv(final_path, dtype=str, keep_default_na=False)
    sample = pd.read_csv(Path(scope['COMP']) / 'sample_submission.csv', dtype=str)
    assert list(frame.columns) == ['molecule_id', 'smiles']
    assert frame.molecule_id.tolist() == sample.molecule_id.tolist()
    assert frame.molecule_id.is_unique and frame.smiles.notna().all()
    processed = set(scope['te'].molecule_id.astype(str))
    inputs = set(scope['E004_PREFLIGHT']['input_ids'])
    assert processed == (inputs if scope['IS_RERUN'] else set(sorted(inputs)[:scope['SMOKE_N']]))
    for mid, answer in frame.itertuples(index=False):
        guesses = answer.split(';')
        assert 1 <= len(guesses) <= 25, mid
        identities = [scope['metric_identity'](s) for s in guesses]
        assert None not in identities and len(set(identities)) == len(identities), mid
    assert scope['n_err'] == 0
    assert scope['FACTORIAL_REPORT']['same_run_control_reconstructed']
    assert scope['FACTORIAL_REPORT']['submission_sha256'] == scope['RANK2_REPORT']['control_sha256']
    assert scope['RANK2_REPORT']['submission_sha256'] == final_before['sha256']

    # Hash all files of the mounted public datasets plus the competition files read.
    # The inventory records content hashes, including model arrays and source modules.
    roots = {str(Path(scope['E004_PREFLIGHT']['runtime_input_path']).resolve())}
    input_root = Path('/kaggle/input')
    dataset_roots = set()
    for slug in V43_DATASET_SOURCES:
        owner, name = slug.split('/', 1)
        direct = input_root / name
        if direct.is_dir():
            dataset_roots.add(direct.resolve())
        else:
            dataset_roots.update(p.resolve() for p in (input_root / 'datasets' / owner).glob(name) if p.is_dir())
    mounted = sorted({p.resolve() for root in dataset_roots for p in root.rglob('*') if p.is_file()})
    comp_root = Path(scope['E004_PREFLIGHT']['runtime_input_path'])
    mounted = sorted(set(mounted) | {comp_root.joinpath(name).resolve() for name in
                                    ('train.parquet', 'test.parquet', 'sample_submission.csv')})
    def quick_digest(path, limit=32 << 20, edge=4 << 20):
        path = Path(path)
        size = path.stat().st_size
        if size <= limit:
            return dict(digest(path), full=True)
        h = hashlib.sha256()
        with path.open('rb') as source:
            h.update(source.read(edge))
            source.seek(size - edge)
            h.update(source.read(edge))
        return dict(path=str(path), bytes=size, sha256_head_tail=h.hexdigest(), full=False)

    input_hashes = [quick_digest(path) for path in mounted]   # large files: size + head/tail hash (full reads took > 2 h)
    assert len(dataset_roots) == len(V43_DATASET_SOURCES), (len(dataset_roots), len(V43_DATASET_SOURCES))
    Path('v43_mounted_input_hashes.json').write_text(json.dumps(input_hashes, indent=2) + '\n')
    stages = scope['E004_REPORT']['stages']
    result = dict(
        version=49, method='canqiang_E006_window40_ice_fallback_plus_v7c_fragment_gap_plus_e2_rank2_rel150_e2_bags_chebi_lmsd_pubchem_spectral_bonus_np_bonus_pubchemlite_lotus_refs_depth6_poplam015_fold0',
        source_notebook_sha256=V43_E006_SHA256,
        determinism=scope.get('V43_DETERMINISM', {'applied': False}),
        rdkit=rdkit.__version__, output_rows=len(frame), input_molecules=len(inputs),
        processed_molecules=len(processed), processed_ids=sorted(processed),
        scoring_rerun=bool(scope['IS_RERUN']), smoke_n=int(scope['SMOKE_N']),
        placeholder_rows=len(inputs - processed), main_errors=int(scope['n_err']),
        main_lists=len(scope['BASE']), engine2_lists=len(scope['ENG']), pubchem_lists=len(scope['PC']),
        engine2_nonempty=sum(bool(scope['ENG'].get(mid, {}).get('smiles')) for mid in processed),
        pubchem_nonempty=sum(bool(scope['PC'].get(mid, {}).get('pc')) for mid in processed),
        merge=scope['stats'], stages=stages,
        final_overlay=scope['FACTORIAL_REPORT'], rank2_overlay=scope['RANK2_REPORT'],
        pubchem_fpnet_ensemble=dict(models='A+B+fold0', fold0_weight=0.3333333333333333, raw_logit_scale=1.5, fold0_path=scope['FOLD0_PATH']),
        mounted_datasets=V43_DATASET_SOURCES, mounted_files=len(input_hashes),
        mounted_input_inventory=digest('v43_mounted_input_hashes.json'),
        train=digest(comp_root / 'train.parquet'), test=digest(comp_root / 'test.parquet'),
        processed_test=digest(Path(scope['COMP']) / 'test.parquet'),
        submission=final_before, source_manifest=digest(Path(scope['V4MOD']) / 'MANIFEST.json'),
        elapsed_seconds=round(time.time() - scope['T0'], 2),
        notebook_elapsed_seconds=round(time.monotonic() - scope['NOTEBOOK_STARTED'], 2),
    )
    assert digest(final_path)['sha256'] == final_before['sha256']
    Path('v43_execution_receipt.json').write_text(json.dumps(result, indent=2, default=str) + '\n')
    print('V43_EXECUTION_RECEIPT', json.dumps({k: result[k] for k in
          ('output_rows', 'processed_molecules', 'placeholder_rows', 'main_errors', 'elapsed_seconds')}))
    return result

V43_RECEIPT = _v43_receipt(globals())
del _v43_receipt


### One molecule through the final gate

One processed molecule of the unlabelled visible example illustrates the final output. The summary line gives its number of measured views, estimated neutral mass and strongest library match, then the merge rule that applied (no PubChem proposals, a strong library match, or insertion at positions 2/4/6/8/10 or 4/8/12/16/20, with or without promotion of the leading proposal) and the final reordering branch (GLACIER, ICEBERG fallback, fragment score or none). Δ is the best PubChem fingerprint score minus the best pool fingerprint score, before the documentation bonus; Section 10 gives the thresholds. The table lists the first five submitted structures with the channels that proposed them, and Figure 21 draws the first three. Table and figure are read from the final submission file and its audit; the displayed example has no attached answer label, so it illustrates the output order.


In [ ]:
# Figure 21 reads the final CSV after the E006 rebuild and never reruns the ranking.
example_ids = list(te.molecule_id.drop_duplicates())
eligible_ids = [m for m in example_ids if BASE[m][2] < LIB_TAU and PC.get(m, {}).get('pc')]
example_mid = eligible_ids[0] if eligible_ids else min(example_ids, key=lambda m: BASE[m][2])
example_query = te[te.molecule_id.eq(example_mid)]
main_smiles, main_keys, example_lib = BASE[example_mid]
proposal = PC.get(example_mid, {})
_ft = proposal.get('fz_top')
if _ft is None and proposal.get('pc_fz'):
    _ft = proposal['pc_fz'][0]
_bp = proposal.get('best_pool_fz')
_delta = (_ft - _bp if _ft is not None and _bp is not None and np.isfinite(_bp) else 1e9)
if not proposal.get('pc'):
    branch = 'main: no PubChem proposals'
elif example_lib >= LIB_TAU:
    branch = 'main: strong library match at the PubChem gate'
else:
    branch = 'PubChem slots 2/4/6/8/10' if _delta > REL_TH else 'PubChem slots 4/8/12/16/20'
    if (proposal.get('S') is not None and float(proposal['S']) > S_TAU and
        float(proposal.get('top_pop') or 0.0) >= POP_TAU) or _delta > 150.0:
        branch += ', confident proposal promoted'
_v43_audits = json.loads(Path('experiment_audit_factorial.json').read_text())['rows']
_v43_row = next(row for row in _v43_audits if row['molecule_id'] == str(example_mid))
_example_branch_labels = {
    'glacier': 'GLACIER scores', 'ice_fallback': 'ICEBERG scores',
    'frag_gap': 'fragment scores', 'unrescored': 'fused order kept',
    'no_engine': 'main order kept', 'placeholder': 'format-example placeholder',
}
branch += '; final reordering: ' + _example_branch_labels.get(_v43_row['branch'], _v43_row['branch'])
delta_text = f' · Δ {_delta:.0f}' if _delta != 1e9 else ''
display(Markdown(f'**{example_mid}** · {len(example_query)} measured view(s) · '
    f'neutral mass {np.nanmedian(example_query.nm):.4f} Da · library match {example_lib:.3f} · '
    f'**{branch}**' + delta_text))
_v43_final = pd.read_csv('submission.csv', dtype=str, keep_default_na=False).set_index('molecule_id')
final_smiles = _v43_final.at[str(example_mid), 'smiles'].split(';')
known = {name: {metric_identity(s) for s in candidates} for name, candidates in (
    ('main', BASE_TAIL_CANDIDATES[str(example_mid)]), ('PubChem', proposal.get('pc', [])),
    ('second engine', ENG.get(str(example_mid), {}).get('smiles', [])))}
example_rows = []
for rank, smiles in enumerate(final_smiles[:5], 1):
    identity = metric_identity(smiles)
    channels = [name for name, identities in known.items() if identity in identities]
    example_rows.append({'Rank': rank, 'Channel': ' + '.join(channels) or 'additional candidate',
                         'Connectivity key': identity[0], 'SMILES': smiles})
display(Markdown('**Worked-example candidates.**'))
display(pd.DataFrame(example_rows))
drawn = [Chem.MolFromSmiles(s) for s in final_smiles[:3]]
if all(m is not None for m in drawn):
    from IPython.display import Image
    _example_drawer = Draw.MolDraw2DCairo(1440, 360, 480, 360)
    _example_drawer.DrawMolecules(drawn, legends=[f'Rank {i}' for i in range(1, len(drawn) + 1)])
    _example_drawer.FinishDrawing()
    display(Image(data=_example_drawer.GetDrawingText()))


*Figure 21. The first three submitted structures for the molecule summarised above, in final output order. The table above the figure lists the first five with the channel that proposed each one; the summary line names the merge rule and the final reordering branch that applied.*


## 13. Discussion

The public spectral libraries and structure catalogues listed in Section 15 offer different kinds of evidence: experimental references such as GNPS, MassBank and MoNA; broad catalogues such as PubChem and ChEBI; and natural-product or biosynthetic resources including COCONUT, LOTUS, NPAtlas, NP67M, MINE and LIPID MAPS. Their roles differ: a measured spectrum can support direct or analogue matching, whereas a structure entry supplies only a candidate. The sources overlap extensively, and every added catalogue also adds competing structures that can displace correct answers, which limits the value of catalogue expansion alone.

Other approaches include rule-based bond cleavage, explanation of peaks and neutral losses by fragmentation trees, and quantum-chemical CID simulation with xTB/GFN methods and QCxMS. The pipeline uses the learned models for candidate ranking. Enumerating plausible fragments does not predict their intensities reliably; rearrangements and alternative protonation sites complicate the mapping from a molecular graph to a spectrum; and a quantum-chemical simulation has to be run for every candidate, which does not fit the candidate volumes of Section 10 within the competition's inference budget. Fragmentation trees remain useful descriptions of mass relationships, and hybrid methods such as CFM-ID learn fragmentation probabilities from data.

The pipeline combines fingerprint networks, learned rankers and the ICEBERG and GLACIER forward models. No paired comparison with the physical methods is reported here. Selective physical calculations could still separate a small set of ambiguous candidates or supply fragment and protonation features to a learned ranker. The literature behind these approaches is listed in Section 15.

A favourable development replay need not transfer to the public score: combining the promotion rule of Section 11 with heavier forward-model weights scored below the previous version. Further work could target representation learning from synthetic spectra, better selection of spectral analogues and more selective candidate generation. A useful evaluation would fix candidate coverage, fit each learned component on separate molecular identities, and measure both newly recovered answers and correct answers displaced by additional proposals.

Five further public refinements were tested separately on a pipeline that scored 0.440 public MRR@25 and did not raise it: GLACIER prediction at every step of a stepped collision energy, merged as in the [FIORA CASMI evaluation](https://doi.org/10.1038/s41467-025-57422-4) (0.436); [MSEntropy](https://github.com/YuanyueLi/MSEntropy) centroiding before the GLACIER spectrum comparison (0.440); separate merged spectra per polarity and adduct in the fingerprint branch of the second engine (0.424); a reciprocal-rank fusion of the final 40 identities with the spectrum-structure similarity of [JESTR](https://doi.org/10.1093/bioinformatics/btaf354) (0.418); and a same-formula reordering by the CFM-ID 4 spectra of the LOTUS [in-silico database](https://zenodo.org/records/14887271), for molecules whose library similarity is below 0.7 (0.440 at weight 0.25, 0.422 at weight 1.0). The two larger losses came from changes that alter the input or the order for many molecules: the fingerprint model of the second engine was probably fitted to the original single merged spectrum, and the released JESTR encoders were trained on the NPLIB1 library. Joining the fragment score to ICEBERG where GLACIER has no scores lowered the score from 0.441 to 0.440.


## 14. Key versions

The table lists the versions with the main gains in public score. Each row names the change made in that version; version 43 replaced the accumulated method with the complete E006 pipeline of Section 10.

| Version | Main change | Public MRR@25 |
|---|---|---:|
| 13 | Two-ranker blend with PubChemLite candidates | 0.341 |
| 17 | New main engine (v4g) with a PubChem search and ICEBERG | 0.366 |
| 18 | GLACIER added as a second forward model | 0.373 |
| 19 | Deeper PubChem shortlist and the documentation prior | 0.386 |
| 21 | Second-engine fusion added to the version 19 method | 0.401 |
| 24 | GLACIER restricted to protonated positive ions | 0.406 |
| 31 | Fragment scoring where ICEBERG scores were absent | 0.410 |
| 39 | Main-list ordering and second-engine fusion weights changed | 0.415 |
| 40 | GLACIER alone for the final reordering | 0.422 |
| 43 | E006: metric-identity fusion and ICEBERG fallback | 0.440 |
| 46 | Fragment score where neither forward model has scores | 0.441 |
| 48 | Rank-2 placement of the second engine's leading structure | 0.443 |
| 49 | PubChem promotion threshold lowered from Δ > 250 to Δ > 150 | 0.446 |
| 51 | Deeper trees (depth 7) in the second-engine rankers | 0.450 |
| 52 | Standardised ChEBI and LIPID MAPS candidates in the second engine | 0.451 |
| 55 | Enriched PubChem prior and fold-safe FPNet; PubChemLite and depth-6 rankers in the second engine | 0.455 |

Version 55 uses the pipeline configuration of [version 12 of the data notebook](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures), which scored the same 0.455 there.


## 15. Sources

The datasets marked *copied* are mounted as attributed copies inside the author's datasets ([CASMI26 Public Models and Candidate Sets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets), [PubChem Popularity Prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior), [Natural-Product Panel Benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark)); their original licences and notices apply. The [Natural Product Knowledge Table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) supplies chemical annotations and LOTUS reference counts, and [CASMI26 Fold-Safe FPNet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) supplies the added checkpoint. The models collection follows CC BY-NC-SA 4.0 with component-specific terms retained; the fold-safe checkpoint follows the competition's non-commercial attribution terms, while its training and evaluation scripts are Apache-2.0. PubChem information is courtesy of the U.S. National Library of Medicine and retains source-specific rights.

| Resource | Type | Contribution |
|---|---|---|
| [CASMI 2026](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra), [data](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data), [rules](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/rules), [metric](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank) | Competition | Spectra, submission contract and molecular-identity scoring. |
| [canqiang E006](https://www.kaggle.com/code/canqiang/casmi26-e006-window-40-ice-fallback-true) | Notebook | Complete inference base. |
| [Ahmed Berat Özer v7c](https://www.kaggle.com/code/ahmedberatozer/casmi26-v7c-e006-fraggap) | Notebook | Fragment score where neither forward model has scores; its `frag_rescore.py` is attributed to [Huseyin Emre Aksoy](https://www.kaggle.com/huseyinemreaksoy) (Apache-2.0). |
| [antriksharya, engine-2 rank-2 insert](https://www.kaggle.com/code/antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert) | Notebook | Rank-2 placement of the second engine's leading structure. |
| [seyitkaangunes, PubChem promotion Δ > 150](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-pubchem-promo-rel-150-lb-0-446) | Notebook | Lower promotion threshold for the leading PubChem proposal. |
| [ghazarosbarseghyan91](https://www.kaggle.com/code/ghazarosbarseghyan91/enveda-v7c-e006-fraggap-evgen-v14seeds), building on [evgendvorkin, version 14](https://www.kaggle.com/code/evgendvorkin/enveda-casmi-2026) | Notebooks | Deeper trees (depth 7) for the earlier second-engine rankers; the current rankers use depth 6. |
| [guhongbin v447](https://www.kaggle.com/code/guhongbin/casmi26-v447-pubchemlite), [Ahmed Berat Özer v14](https://www.kaggle.com/code/ahmedberatozer/casmi26-v14-pubchemlite) | Notebooks | PubChemLite candidates in the second engine of the E006 line (guhongbin); the block used here (Ahmed Berat Özer). |
| [guhongbin v465](https://www.kaggle.com/code/guhongbin/casmi26-v465-v15-pop015) | Notebook | PubChem-channel documentation-prior weight 0.15. |
| [franciscoangulo](https://www.kaggle.com/datasets/franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints), [cosmicbe](https://www.kaggle.com/datasets/cosmicbe/casmi26-pubchemlite), [Schymanski et al., PubChemLite](https://doi.org/10.1186/s13321-021-00489-0) | Dataset (copied), CC0; source dataset and paper | PubChemLite candidates and fingerprints for the second engine. |
| [CASMI26 Fold-Safe FPNet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) | Model and code | Architecture and fingerprint bits by Ahmed Berat Özer; training kit, checkpoint and split audit by Dmitrii Gluzdov. |
| [sergiomalv, ChEBI + LIPID MAPS structures](https://www.kaggle.com/datasets/sergiomalv/casmi-chebi-lmsd-structures) | Dataset (copied), CC BY 4.0 | Standardised ChEBI and LIPID MAPS candidates for the second engine. |
| [Obstacle V1.1](https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail?scriptVersionId=356393001), [Bob V18](https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith) | Notebooks | Inherited ensemble, proposal promotion, identity audit and candidate filling. |
| [flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-public-baseline-adaptation-experiments), [samanyu1808](https://www.kaggle.com/code/samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438) | Notebooks | Metric-identity fusion and ICEBERG fallback adopted by E006. |
| [Ahmed Berat Özer v4n](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4n-inference), [v4m](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4m-inference) | Notebooks | Main-engine family and the full1 fingerprint checkpoint. |
| [Prvsiyan](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline), [megayak](https://www.kaggle.com/code/megayak/casmi26-two-rankers-one-engine), [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-v4n-engine-fusion-union-lb-0-399) | Notebooks | Analogue propagation, two-ranker scoring and engine fusion. |
| [wangpenghua v21](https://www.kaggle.com/code/wangpenghua/casmi26-v21-v4i-pop), [v29](https://www.kaggle.com/code/wangpenghua/casmi26-v29-fusion-pop-glmh) | Notebooks | Documentation prior and protonated-ion restriction for GLACIER. |
| [Ranker features](https://www.kaggle.com/datasets/prvsiyan/casmi26-ranker-features), [FP models](https://www.kaggle.com/datasets/prvsiyan/casmi26-fp-models-v2), [simulated ranker rows](https://www.kaggle.com/datasets/megayak/casmi26-simulated-ranker-rows) | Datasets (copied) | Features, fingerprint weights and fitting rows for the second engine. |
| [v3 models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v3-models), [v4b models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v4b-models), [FPNet full1](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-fpnet-full1) | Models (copied) | v3: earlier code and the FPNet A+B weights of the PubChem channel; v4b and full1: main-engine code, rankers and the full1 fingerprint weights. |
| [v2 pool](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v2-pool), [PubChem tier](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-pubchem-tier) | Datasets (copied) | The training/COCONUT candidate pool with fragments and fingerprints, and the structure-only PubChem index for the PubChem channel. |
| [COCONUT candidates](https://www.kaggle.com/datasets/prvsiyan/coconut-casmi26-candidates), [ChEBI/LIPID MAPS](https://www.kaggle.com/datasets/prvsiyan/chebi-lipidmaps-casmi26) | Datasets (copied) | COCONUT candidates for the second engine and the earlier ChEBI/LIPID MAPS bank retained in the models package. |
| [PubChem popularity prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) | Dataset | Public reconstruction of substance and literature counts, depositor flags, natural-product membership and aligned LOTUS reference counts used in ranking. |
| [ICEBERG](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-iceberg), [GLACIER](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier), [ms-pred](https://github.com/coleygroup/ms-pred), [ICEBERG paper](https://doi.org/10.1021/acs.analchem.3c04654) | Models and code (copied), paper | Candidate-specific forward spectrum prediction. |
| [Hengck23 DreaMS resource](https://www.kaggle.com/datasets/hengck23/hengck23-dreams-enveda-casmi26), [DreaMS paper](https://www.nature.com/articles/s41587-025-02663-3) | Dataset (copied) and paper | Inherited mass-based candidate filling; background on learned spectral representations. |
| [Metric RDKit wheel](https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel), [simulator wheels](https://www.kaggle.com/datasets/eveliaveldrine/rdkit-wheels) | Packages (copied) | Molecular normalisation and compatible simulator environments. |
| [Identity-disjoint folds](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-identity-disjoint-validation-folds), [natural-product knowledge table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table), [panel benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | Datasets | EDA annotations, LOTUS literature counts and the public development analyses in Section 11; the folds are copied unchanged inside the panel package. |
| [GNPS](https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/), [MassBank](https://github.com/MassBank/MassBank-data), [MoNA](https://massbank.us/), [Enveda-180](https://github.com/enveda/enveda-180) | Spectral libraries reviewed | Experimental references and acquisition context. |
| [PubChem](https://pubchem.ncbi.nlm.nih.gov/docs/downloads), [ChEBI](https://www.ebi.ac.uk/chebi/downloads), [HMDB](https://www.hmdb.ca/downloads), [LIPID MAPS](https://www.lipidmaps.org/) | Structure sources reviewed | Broad chemical, metabolite and lipid catalogues; HMDB is not among the mounted inputs. |
| [COCONUT](https://coconut.naturalproducts.net/download), [LOTUS frozen export of 2026-04-13](https://doi.org/10.5281/zenodo.19360665), [NPAtlas](https://www.npatlas.org/) | Structure sources | Natural-product structures and biological annotations. The LOTUS frozen export is CC BY 4.0 ([Rutz et al.](https://doi.org/10.7554/eLife.70780)); NPAtlas is CC BY-NC 4.0 ([van Santen et al.](https://doi.org/10.1093/nar/gkab941)), with attribution and non-commercial terms retained. Pathway labels are from [NPClassifier, Kim et al.](https://doi.org/10.1021/acs.jnatprod.1c00399); PubChem documentation is credited to [Kim et al., PubChem 2025 update](https://doi.org/10.1093/nar/gkae1059). |
| [NP67M](https://doi.org/10.6084/m9.figshare.c.6482266.v1), [MINE](https://pmc.ncbi.nlm.nih.gov/articles/PMC9237697/) | Structure sources reviewed | Generated or biochemically expanded candidate catalogues; neither is among the mounted inputs. |
| [CSI:FingerID](https://www.pnas.org/doi/10.1073/pnas.1509788112), [MIST](https://www.nature.com/articles/s42256-023-00708-3) | Papers | Fingerprint prediction from fragmentation evidence. |
| [MS2Query](https://www.nature.com/articles/s41467-023-37446-4), [spectral entropy](https://doi.org/10.1038/s41592-021-01331-z) | Papers | Analogue retrieval and comparison of measured spectra. |
| [MSNovelist](https://www.nature.com/articles/s41592-022-01486-3), [competition de novo tutorial](https://www.kaggle.com/code/inversion/casmi-denovo-tutorial-notebook) | Paper and notebook | Structure generation beyond catalogue retrieval. |
| [MassSpecGym](https://github.com/pluskal-lab/MassSpecGym) | Benchmark | Retrieval and structure-generation tasks with generalisation protocols. |
| [MetFrag](https://doi.org/10.1186/s13321-016-0115-9), [PubChemLite](https://doi.org/10.1186/s13321-021-00489-0) | Papers | Fragmentation and chemical-documentation evidence in candidate ranking. |
| [CFM-ID 4.0](https://doi.org/10.1021/acs.analchem.1c01465), [CASMI 2016 comparison](https://pmc.ncbi.nlm.nih.gov/articles/PMC5368104/) | Papers | Learned fragmentation probabilities and comparison with rule-based identification. |
| [GFN2-xTB](https://doi.org/10.1021/acs.jctc.8b01176), [QCxMS](https://doi.org/10.1021/jasms.1c00098), [QCxMS2 CID](https://doi.org/10.1021/jasms.5c00234), [CIDMD](https://pmc.ncbi.nlm.nih.gov/articles/PMC11492810/) | Papers | Quantum-chemical methods considered for candidate fragmentation. |
| [Reciprocal-rank fusion](https://doi.org/10.1145/1571941.1572114) | Paper | Combining rankings without equating their raw scores. |
| [CASMI overview](https://pmc.ncbi.nlm.nih.gov/articles/PMC3901296/), [2013 analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC4337369/), [2022 results](https://www.fiehnlab.ucdavis.edu/casmi/casmi-2022-results) | Challenge reports | Historical context for candidate coverage and structure identification. |

Scores quoted in the titles of linked notebooks are those notebooks' own public scores, not values from Section 14.
